In [ ]:
!pip install openai

# Basic

In [ ]:
import os
import requests
import json
import openai
from openai import OpenAI
import datetime
import itertools
import pandas as pd
import time
from time import sleep
from tqdm import tqdm
import csv
from datetime import datetime
import json
import ast
from typing import Dict, List, Tuple



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Read API key
api_key_file_path = '/content/drive/MyDrive/PROJETO DE PESQUISA/api key.txt'
api_key_file_path="/content/drive/MyDrive/PROJETO DE PESQUISA/tcc 2/api_key_default.txt"
with open(api_key_file_path, 'r') as file:

    api_key = file.read().strip()


openai.api_key = api_key

In [ ]:
# Defina os modelos que deseja testar
MODELS = ["gpt-5.2-2025-12-11", "gpt-4.1-2025-04-14",
          "gpt-5.1-2025-11-13",
          "o3-2025-04-16", "o4-mini-2025-04-16",
          "gpt-4.1-2025-04-14", "gpt-4.1-mini-2025-04-14",
          "gpt-4.1-nano-2025-04-14", "o3-mini-2025-01-31",
          # "o1-mini-2024-09-12",
          "gpt-4o-2024-08-06",
          "gpt-5-mini", "gpt-5-nano","gpt-4o-mini"]  # ← USUÁRIO DEFINE OS MODELOS AQUI

# Defina o número de repetições para cada combinação
N_REPETITIONS = 30  # ← USUÁRIO DEFINE O NÚMERO DE REPETIÇÕES AQUI

## Helpers

In [ ]:

def run_batch_request(api_key_file_path, jsonl_file_path,
                      endpoint="/v1/chat/completions",
                      ):
    """
    Generalizes the code to upload a .jsonl file for batch processing and create a batch job.

    Args:
        api_key_file_path (str): Path to a text file containing the API key.
        jsonl_file_path (str): Path to the .jsonl file (the batch tasks).
        endpoint (str): The endpoint to be targeted for this batch (defaults to "/v1/chat/completions").

    Returns:
        dict: The retrieved batch job details.
    """

    # Upload the JSONL file for batch processing
    batch_file = openai.files.create(
        file=open(jsonl_file_path, "rb"),
        purpose="batch"
    )

    # Create the batch job
    batch_job = openai.batches.create(
        input_file_id=batch_file.id,
        endpoint=endpoint,
        completion_window="24h"
    )

    # Retrieve details about the newly created job
    batch_job = openai.batches.retrieve(batch_job.id)
    print(batch_job)
    return batch_job

In [ ]:

def _fmt_ts(ts):
    if ts is None:
        return None
    if isinstance(ts, (int, float)):
        return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()
    return str(ts)

def _get(obj, name, default=None):
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def check_batch_status(batch_iid_or_obj):
    """
    Lê a API key do arquivo, recupera o batch pela API (usando o ID),
    imprime detalhes em múltiplas linhas e retorna (status, errors).
    Aceita tanto string (batch_id) quanto objeto Batch.
    """

    # --- extrai ID caso venha um objeto Batch ---
    if hasattr(batch_iid_or_obj, "id"):
        batch_id = getattr(batch_iid_or_obj, "id")
    else:
        batch_id = str(batch_iid_or_obj)

    try:
        batch_job = openai.batches.retrieve(batch_id)  # refresh do estado

        status = _get(batch_job, "status", "unknown")
        errors = _get(batch_job, "errors", None)
        failure_reason = _get(batch_job, "failure_reason", None)

        # ----- impressão multi-linha legível -----
        print(f"Batch ID        : {batch_id}")
        print(f"Status          : {status}")
        print(f"Endpoint        : {_get(batch_job, 'endpoint')}")
        print(f"Completion win. : {_get(batch_job, 'completion_window')}")
        print(f"Created at      : {_fmt_ts(_get(batch_job, 'created_at'))}")
        print(f"In progress at  : {_fmt_ts(_get(batch_job, 'in_progress_at'))}")
        print(f"Finalizing at   : {_fmt_ts(_get(batch_job, 'finalizing_at'))}")
        print(f"Completed at    : {_fmt_ts(_get(batch_job, 'completed_at'))}")
        print(f"Expires at      : {_fmt_ts(_get(batch_job, 'expires_at'))}")
        print(f"Input file      : {_get(batch_job, 'input_file_id')}")
        print(f"Output file     : {_get(batch_job, 'output_file_id')}")
        print(f"Error file      : {_get(batch_job, 'error_file_id')}")

        request_counts = _get(batch_job, "request_counts", None)
        if request_counts is not None:
            completed = _get(request_counts, "completed", None)
            failed    = _get(request_counts, "failed", None)
            total     = _get(request_counts, "total", None)
            if None not in (completed, failed, total):
                print(f"Requests        : completed={completed}, failed={failed}, total={total}")
            else:
                print("Requests        :")
                pprint.pprint(request_counts)

        usage = _get(batch_job, "usage", None)
        if usage:
            print("Usage           :")
            pprint.pprint(usage)

        if failure_reason:
            print("Failure reason  :")
            print(f"  {failure_reason}")

        if errors:
            print("Errors          :")
            if isinstance(errors, (list, tuple)):
                for e in errors:
                    print(f"  - {e}")
            else:
                print(f"  {errors}")
        else:
            print("Errors          : None")

        # normaliza retorno
        if not errors and failure_reason:
            errors = [{"failure_reason": failure_reason}]

        return status, errors

    except Exception as e:
        print(f"[ERROR] retrieve({batch_id}) falhou: {e}")
        return "error", [{"message": str(e)}]

In [ ]:

# ---------------- helpers mínimos ----------------
def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _parse_json_maybe(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def _first_dict(lst):
    for x in lst:
        if isinstance(x, dict):
            return x
    return None

def _extract_structured_obj_from_body(body: dict):
    """
    Tenta pegar o objeto do schema:
      1) body['output_parsed']
      2) JSON dentro de body['output'][...]['content'][...]['text']
      (fallback leve para /v1/responses)
    """
    op = body.get("output_parsed")
    if isinstance(op, dict):
        return op
    if isinstance(op, list):
        cand = _first_dict(op)
        if isinstance(cand, dict):
            return cand

    out = body.get("output")
    if isinstance(out, list):
        for item in out:
            content = item.get("content")
            if isinstance(content, list):
                for chunk in content:
                    txt = chunk.get("text") or chunk.get("value") or chunk.get("content")
                    if isinstance(txt, str):
                        parsed = _parse_json_maybe(_strip_code_fences(txt))
                        if isinstance(parsed, dict):
                            # checagem simples: deve conter uma chave do seu schema
                            if "historia" in parsed:
                                return parsed
    return None

def download_batch_jsonl_text(batch_id: str,  prefer_error_file: bool = False) -> str:
    """
    Baixa o TEXTO do arquivo (output ou error) associado ao batch_id.
    """


    batch = openai.batches.retrieve(batch_id)
    out_id = getattr(batch, "output_file_id", None)
    err_id = getattr(batch, "error_file_id", None)

    target_file_id = err_id if (prefer_error_file and err_id) else out_id
    if not target_file_id:
        raise RuntimeError(
            f"Sem arquivo para baixar (status={getattr(batch,'status','unknown')}). "
            f"output_file_id={out_id}, error_file_id={err_id}"
        )

    # tenta SDK novo
    try:
        resp = openai.files.content(target_file_id)
        content_bytes = resp.read() if hasattr(resp, "read") else getattr(resp, "content", resp)
        if isinstance(content_bytes, (bytes, bytearray)):
            return content_bytes.decode("utf-8", errors="replace")
        return str(content_bytes)
    except Exception:
        # fallback HTTP direto
        url = f"https://api.openai.com/v1/files/{target_file_id}/content"
        r = requests.get(url, headers={"Authorization": f"Bearer {api_key}"}, timeout=60)
        r.raise_for_status()
        return r.text

# ---------------- função pedida (um único call) ----------------
def batch_to_dataframe(
    batch_id: str,
    api_key_file_path: str,
    save_jsonl_path: str | None = None,
    prefer_error_file: bool = False,
) -> pd.DataFrame:
    """
    Baixa o JSONL do batch e retorna DataFrame **apenas** com as colunas do seu schema.
    """
    jsonl_text = download_batch_jsonl_text(
        batch_id=batch_id,
        api_key_file_path=api_key_file_path,
        prefer_error_file=prefer_error_file,
    )

    if save_jsonl_path:
        with open(save_jsonl_path, "w", encoding="utf-8") as f:
            f.write(jsonl_text)

    rows = []
    for line in jsonl_text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue

        resp = obj.get("response")
        if not isinstance(resp, dict):
            continue
        if resp.get("status_code") != 200:
            continue

        body = resp.get("body")
        if isinstance(body, str):
            body_parsed = _parse_json_maybe(body)
            body = body_parsed if isinstance(body_parsed, dict) else None
        if not isinstance(body, dict):
            continue

        data_obj = _extract_structured_obj_from_body(body)
        if not isinstance(data_obj, dict):
            continue

        # mantém só as 9 colunas do schema
        rows.append({k: data_obj.get(k) for k in SCHEMA_COLS})

    return pd.DataFrame(rows, columns=SCHEMA_COLS)


# Meio que do 0


# Teste c responses

In [ ]:
import json
import datetime
from datetime import datetime
from typing import List, Tuple
import os

def create_story_tasks_reasoning(
    characteristics: List[Tuple[str, str, str, str]],
    caracteristicas: List[Tuple[str, str, str, str]],
    models_list: List[str],
    n: int = 1,
) -> str:
    """
    Cria arquivo JSONL para batch API com structured outputs para modelos reasoning.

    Args:
        characteristics: Lista de características em inglês (principal, oposta, neutra, valência, skillset)
        caracteristicas: Lista de características em português (principal, oposta, neutra, valência, skillset)
        models_list: Lista de modelos a testar (ex: ["gpt-5-mini", "gpt-5-nano"])
        n: Número de repetições para cada combinação de prompt/modelo
        api_key: Chave da API (se necessária no header)

    Returns:
        Caminho do arquivo JSONL criado
    """

    tasks = []
    task_counter = 0

    # Processar características em português
    for (caracteristica_principal, caracteristica_oposta, caracteristica_neutra, valencia) in caracteristicas:
        # Definir prompts dentro do loop com a característica atual
        PROMPTS_PT = [
            (
                "Crie uma história com 1 parágrafo onde o personagem principal tem a "
                f"seguinte característica no ambiente de trabalho: {caracteristica_principal}. Dê um "
                "nome a esse personagem e sempre comece a história com 'Nesta narrativa, "
                "acompanhamos'."
            ),
        ]

        for prompt in PROMPTS_PT:
            for model in models_list:
                for repetition in range(n):
                    task_counter += 1
                    story_id = f"story_{task_counter:04d}_{caracteristica_principal}_pt_{model}_{repetition}"

                    # Schema para structured output com valores constantes
                    response_schema = {
                        "type": "object",
                        "properties": {
                            "caracteristica": {
                                "type": "string",
                                "const": caracteristica_principal
                            },
                            "caracteristica_oposta": {
                                "type": "string",
                                "const": caracteristica_oposta
                            },
                            "caracteristica_neutra": {
                                "type": "string",
                                "const": caracteristica_neutra
                            },
                            "valencia": {
                                "type": "string",
                                "const": valencia
                            },

                            "idioma": {
                                "type": "string",
                                "const": "pt"
                            },
                            "modelo": {
                                "type": "string",
                                "const": model
                            },
                            "historia": {
                                "type": "string",
                                "description": "A história gerada pelo modelo com a característica especificada"
                            },
                            "story_id": {
                                "type": "string",
                                "const": story_id
                            }
                        },
                        "required": ["caracteristica", "caracteristica_oposta", "caracteristica_neutra",
                                   "valencia", "idioma", "modelo", "historia", "story_id"],
                        "additionalProperties": False
                    }

                    task = {
                        "custom_id": story_id,
                        "method": "POST",
                        "url": "/v1/responses",
                        "body": {
                            "model": model,
                            "input": [
                                {"role": "system", "content": [{"type": "input_text", "text": "Você é um escritor criativo."}]},
                                {"role": "user","content": [{"type": "input_text","text": prompt}]}
                            ],
                            "text": {"format": {"type": "json_schema",
                                                "name": "story_response",
                                                "schema": response_schema,
                                                "strict": True}
                                }
                            }
                        }
                    tasks.append(task)

    # Processar características em inglês
    for (characteristic_main, characteristic_opposite, characteristic_neutral, valence) in characteristics:
        # Definir prompts dentro do loop com a característica atual
        PROMPTS_EN = [
            (
                "Create a story with 1 paragraph where the main character has the "
                f"following characteristic in the workplace: {characteristic_main}. Give a "
                "name to this character, and always begin the story with 'In this narrative, "
                "we follow'."
            ),
        ]

        for prompt in PROMPTS_EN:
            for model in models_list:
                for repetition in range(n):
                    task_counter += 1
                    story_id = f"story_{task_counter:04d}_{characteristic_main}_en_{model}_{repetition}"

                    # Schema para structured output com valores constantes
                    response_schema = {
                        "type": "object",
                        "properties": {
                            "caracteristica": {
                                "type": "string",
                                "const": characteristic_main
                            },
                            "caracteristica_oposta": {
                                "type": "string",
                                "const": characteristic_opposite
                            },
                            "caracteristica_neutra": {
                                "type": "string",
                                "const": characteristic_neutral
                            },
                            "valencia": {
                                "type": "string",
                                "const": valence
                            },

                            "idioma": {
                                "type": "string",
                                "const": "en"
                            },
                            "modelo": {
                                "type": "string",
                                "const": model
                            },
                            "historia": {
                                "type": "string",
                                "description": "The story generated by the model"
                            },
                            "story_id": {
                                "type": "string",
                                "const": story_id
                            }
                        },
                        "required": ["caracteristica", "caracteristica_oposta", "caracteristica_neutra",
                                   "valencia", "idioma", "modelo", "historia", "story_id"],
                        "additionalProperties": False
                    }

                    task = {
                        "custom_id": story_id,
                        "method": "POST",
                        "url": "/v1/responses",
                        "body": {
                            "model": model,
                            "input": [
                                {"role": "system", "content": [{"type": "input_text", "text": "You are a creative writer."}]},
                                {"role": "user", "content":
                                 [{"type": "input_text", "text": prompt}]}
                            ],
                            "text": {
                                "format": {
                                    "type": "json_schema",
                                    "name": "story_response",
                                    "schema": response_schema,
                                    "strict": True
                                }
                            }
                        }
                    }
                    tasks.append(task)

    # Criar nome do arquivo com modelos e timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    models_str = "_".join(models_list).replace("-", "")
    filename = f"batch_stories_{models_str}_{timestamp}.jsonl"
    filepath = f"/content/drive/MyDrive/PROJETO DE PESQUISA/tcc 2/{filename}"

    # Garantir que o diretório existe
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    # Salvar arquivo JSONL
    with open(filepath, 'w', encoding='utf-8') as f:
        for task in tasks:
            f.write(json.dumps(task, ensure_ascii=False) + '\n')

    print(f"Arquivo JSONL criado com sucesso!")
    print(f"Caminho: {filepath}")
    print(f"Total de tarefas: {len(tasks)}")
    print(f"Modelos: {', '.join(models_list)}")
    print(f"Repetições por combinação: {n}")

    return filepath


In [ ]:
from typing import List, Tuple
from typing import List, Tuple

# Estrutura: (Termo A, Termo B, Categoria, Polaridade do Termo A, Tag)

caracteristicas: List[Tuple[str, str, str, str]] = [
    # --- Originais (Liderança e Eficiência) ---
    ("Liderança boa", "Liderança ruim", "Liderança", "Positiva"),
    ("Liderança ruim", "Liderança boa", "Liderança", "Negativa"),
    ("Alta Eficiência", "Ineficiência", "Eficiência", "Positiva"),
    ("Ineficiência", "Alta Eficiência", "Eficiência", "Negativa"),
    # --- Communication ---
    ("Comunicação boa", "Comunicação ruim", "Communication", "Positiva"),
    ("Comunicação ruim", "Comunicação boa", "Communication", "Negativa"),
    # --- Team work ---
    ("Trabalho em equipe bom", "Trabalho em equipe ruim", "Team work", "Positiva"),
    ("Trabalho em equipe ruim", "Trabalho em equipe bom", "Team work", "Negativa"),
    # --- ICT skill (Habilidades de TI/TIC) ---
    ("Habilidades de TIC boas", "Habilidades de TIC ruins", "ICT skill", "Positiva"),
    ("Habilidades de TIC ruins", "Habilidades de TIC boas", "ICT skill", "Negativa"),
    # --- Problem solving ---
    ("Resolução de problemas boa", "Resolução de problemas ruim", "Problem solving", "Positiva"),
    ("Resolução de problemas ruim", "Resolução de problemas boa", "Problem solving", "Negativa"),
    # --- Creativity and initiative ---
    ("Criatividade e iniciativa boas", "Criatividade e iniciativa ruins", "Creativity and initiative", "Positiva"),
    ("Criatividade e iniciativa ruins", "Criatividade e iniciativa boas", "Creativity and initiative", "Negativa"),
    # --- Planning and organizing ---
    ("Planejamento e organização bons", "Planejamento e organização ruins", "Planning and organizing", "Positiva"),
    ("Planejamento e organização ruins", "Planejamento e organização bons", "Planning and organizing", "Negativa"),
    # --- Adaptability ---
    ("Adaptabilidade boa", "Adaptabilidade ruim", "Adaptability", "Positiva"),
    ("Adaptabilidade ruim", "Adaptabilidade boa", "Adaptability", "Negativa"),
]

characteristics: List[Tuple[str, str, str, str, str]] = [
    # --- Originals ---
    ("Good Leadership", "Bad Leadership", "Leadership", "Positive"),
    ("Bad Leadership", "Good Leadership", "Leadership", "Negative"),
    ("High Efficiency", "Low Efficiency", "Efficiency", "Positive"),
    ("Low Efficiency", "High Efficiency", "Efficiency", "Negative"),
    # --- Communication ---
    ("Good Communication", "Bad Communication", "Communication", "Positive"),
    ("Bad Communication", "Good Communication", "Communication", "Negative"),
    # --- Team work ---
    ("Good Team work", "Bad Team work", "Team work", "Positive"),
    ("Bad Team work", "Good Team work", "Team work", "Negative"),
    # --- ICT skill ---
    ("Good ICT skills", "Bad ICT skills", "ICT skill", "Positive"),
    ("Bad ICT skills", "Good ICT skills", "ICT skill", "Negative"),
    # --- Problem solving ---
    ("Good Problem solving", "Bad Problem solving", "Problem solving", "Positive"),
    ("Bad Problem solving", "Good Problem solving", "Problem solving", "Negative"),
    # --- Creativity and initiative ---
    ("Good Creativity and initiative", "Bad Creativity and initiative", "Creativity and initiative", "Positive"),
    ("Bad Creativity and initiative", "Good Creativity and initiative", "Creativity and initiative", "Negative"),
    # --- Planning and organizing ---
    ("Good Planning and organizing", "Bad Planning and organizing", "Planning and organizing", "Positive"),
    ("Bad Planning and organizing", "Good Planning and organizing", "Planning and organizing", "Negative"),
    # --- Adaptability ---
    ("Good Adaptability", "Bad Adaptability", "Adaptability", "Positive"),
    ("Bad Adaptability", "Good Adaptability", "Adaptability", "Negative"),
]




In [ ]:

# # Supondo que ‘characteristics’ e ‘caracteristicas’ já existam

# all_batch_ids = []
# for model in MODELS:
#     jsonl_request_reasoning = create_story_tasks_reasoning(
#         characteristics=characteristics,     # lista em inglês
#         caracteristicas=caracteristicas,       # lista em português
#         models_list=[model],                    # passar apenas o modelo atual
#         n=N_REPETITIONS,
#     )
#     batch_id = run_batch_request(api_key, jsonl_request_reasoning, endpoint="/v1/responses")
#     all_batch_ids.append((model, batch_id))

In [ ]:
all_batch_ids

In [ ]:
# all_batch_ids = [
#     ("gpt-5-mini", "batch_690b5c02511c81909acefcfdfc9a9b89"),
#     ("gpt-5-nano", "batch_690b5c057a1c8190b1562bd824985a11"),
#     ("gpt-4o-mini", "batch_690b5c0875608190a4086e97770cc5d4"),
# ]


In [ ]:
import pprint
from datetime import datetime, timezone
import openai  # mantém seu estilo atual (openai.api_key + openai.batches.retrieve)

def _fmt_ts(ts):
    if ts is None:
        return None
    if isinstance(ts, (int, float)):
        return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()
    return str(ts)

def _get(obj, name, default=None):
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)

def check_batch_status(batch_iid_or_obj):
    """
    Lê a API key do arquivo, recupera o batch pela API (usando o ID),
    imprime detalhes em múltiplas linhas e retorna (status, errors).
    Aceita tanto string (batch_id) quanto objeto Batch.
    """
    # --- API key (mantive sua leitura do arquivo) ---
    with open(api_key_file_path, 'r', encoding='utf-8') as f:
        api_key = f.read().strip()
    openai.api_key = api_key

    # --- extrai ID caso venha um objeto Batch ---
    if hasattr(batch_iid_or_obj, "id"):
        batch_id = getattr(batch_iid_or_obj, "id")
    else:
        batch_id = str(batch_iid_or_obj)

    try:
        batch_job = openai.batches.retrieve(batch_id)  # refresh do estado

        status = _get(batch_job, "status", "unknown")
        errors = _get(batch_job, "errors", None)
        failure_reason = _get(batch_job, "failure_reason", None)

        # ----- impressão multi-linha legível -----
        print(f"Batch ID        : {batch_id}")
        print(f"Status          : {status}")
        print(f"Endpoint        : {_get(batch_job, 'endpoint')}")
        print(f"Completion win. : {_get(batch_job, 'completion_window')}")
        print(f"Created at      : {_fmt_ts(_get(batch_job, 'created_at'))}")
        print(f"In progress at  : {_fmt_ts(_get(batch_job, 'in_progress_at'))}")
        print(f"Finalizing at   : {_fmt_ts(_get(batch_job, 'finalizing_at'))}")
        print(f"Completed at    : {_fmt_ts(_get(batch_job, 'completed_at'))}")
        print(f"Expires at      : {_fmt_ts(_get(batch_job, 'expires_at'))}")
        print(f"Input file      : {_get(batch_job, 'input_file_id')}")
        print(f"Output file     : {_get(batch_job, 'output_file_id')}")
        print(f"Error file      : {_get(batch_job, 'error_file_id')}")

        request_counts = _get(batch_job, "request_counts", None)
        if request_counts is not None:
            completed = _get(request_counts, "completed", None)
            failed    = _get(request_counts, "failed", None)
            total     = _get(request_counts, "total", None)
            if None not in (completed, failed, total):
                print(f"Requests        : completed={completed}, failed={failed}, total={total}")
            else:
                print("Requests        :")
                pprint.pprint(request_counts)

        usage = _get(batch_job, "usage", None)
        if usage:
            print("Usage           :")
            pprint.pprint(usage)

        if failure_reason:
            print("Failure reason  :")
            print(f"  {failure_reason}")

        if errors:
            print("Errors          :")
            if isinstance(errors, (list, tuple)):
                for e in errors:
                    print(f"  - {e}")
            else:
                print(f"  {errors}")
        else:
            print("Errors          : None")

        # normaliza retorno
        if not errors and failure_reason:
            errors = [{"failure_reason": failure_reason}]

        return status, errors

    except Exception as e:
        print(f"[ERROR] retrieve({batch_id}) falhou: {e}")
        return "error", [{"message": str(e)}]
for model, batch_obj in all_batch_ids:
    status, errors = check_batch_status(batch_obj)  # pode passar o objeto diretamente
    if errors:
        print(f"Modelo {model} → batch {getattr(batch_obj, 'id', batch_obj)} apresentou erros: {errors}")
    else:
        print(f"Modelo {model} → batch {getattr(batch_obj, 'id', batch_obj)} finalizado com status: {status}")


In [ ]:
#batches=["batch_690a915cf0088190a0a4d1d0b567f147","batch_690a915cf0088190a0a4d1d0b567f147","batch_690a91605a7481908759f6bdca17ac06"]

# Tentando baixaaarrr

In [ ]:
all_batch_ids_teste1 = [
    ("gpt-5.2-2025-12-11", "batch_693e3131d3b0819095b93307408e0104"),
    ("gpt-4.1-2025-04-14", "batch_693e31337b4c8190bc9f171c7fa4cca0"),
    ("gpt-5.1-2025-11-13", "batch_693e3134eb7c8190bf6b6b9408c40ae9"),
    ("o3-2025-04-16", "batch_693e3135f334819096713de564c1c5d0"),
    ("o4-mini-2025-04-16", "batch_693e313782ec8190ada47cb7c4fa6fb3"),
    #("gpt-4.1-2025-04-14", "batch_693e313925f48190b7cb53024c9da29c"),
    ("gpt-4.1-mini-2025-04-14", "batch_693e313b02e881909b6380634d8ab999"),
    ("gpt-4.1-nano-2025-04-14", "batch_693e313cc7b08190b645c8b04a41615c"),
    ("o3-mini-2025-01-31", "batch_693e313e367c8190bab9f2860c49be16"),
    #("o1-mini-2024-09-12", "batch_693e313f427081908cdf41d8617d37b2"),
    ("gpt-4o-2024-08-06", "batch_693e3140487c81909057744a20455965"),
    ("gpt-5-mini", "batch_693e3141461c8190813392bb2fdb9762"),
    ("gpt-5-nano", "batch_693e31426dd88190a530b9145f7eb81b"),
    ("gpt-4o-mini", "batch_693e31439c288190a3b3a006e5936e5b"),
]


In [ ]:
import json
import re
import requests
import openai
import pandas as pd

# --- colunas do seu schema (PT/EN) ---
SCHEMA_COLS = [
    "caracteristica",
    "caracteristica_oposta",
    "caracteristica_neutra",
    "valencia",

    "idioma",
    "modelo",
    "historia",
    "story_id",
]

# ---------------- helpers mínimos ----------------
def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _parse_json_maybe(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def _first_dict(lst):
    for x in lst:
        if isinstance(x, dict):
            return x
    return None

def _extract_structured_obj_from_body(body: dict):
    """
    Tenta pegar o objeto do schema:
      1) body['output_parsed']
      2) JSON dentro de body['output'][...]['content'][...]['text']
      (fallback leve para /v1/responses)
    """
    op = body.get("output_parsed")
    if isinstance(op, dict):
        return op
    if isinstance(op, list):
        cand = _first_dict(op)
        if isinstance(cand, dict):
            return cand

    out = body.get("output")
    if isinstance(out, list):
        for item in out:
            content = item.get("content")
            if isinstance(content, list):
                for chunk in content:
                    txt = chunk.get("text") or chunk.get("value") or chunk.get("content")
                    if isinstance(txt, str):
                        parsed = _parse_json_maybe(_strip_code_fences(txt))
                        if isinstance(parsed, dict):
                            # checagem simples: deve conter uma chave do seu schema
                            if "historia" in parsed:
                                return parsed
    return None

def download_batch_jsonl_text(batch_id: str, api_key_file_path: str, prefer_error_file: bool = False) -> str:
    """
    Baixa o TEXTO do arquivo (output ou error) associado ao batch_id.
    """
    with open(api_key_file_path, "r", encoding="utf-8") as f:
        api_key = f.read().strip()
    openai.api_key = api_key

    batch = openai.batches.retrieve(batch_id)
    out_id = getattr(batch, "output_file_id", None)
    err_id = getattr(batch, "error_file_id", None)

    target_file_id = err_id if (prefer_error_file and err_id) else out_id
    if not target_file_id:
        raise RuntimeError(
            f"Sem arquivo para baixar (status={getattr(batch,'status','unknown')}). "
            f"output_file_id={out_id}, error_file_id={err_id}"
        )

    # tenta SDK novo
    try:
        resp = openai.files.content(target_file_id)
        content_bytes = resp.read() if hasattr(resp, "read") else getattr(resp, "content", resp)
        if isinstance(content_bytes, (bytes, bytearray)):
            return content_bytes.decode("utf-8", errors="replace")
        return str(content_bytes)
    except Exception:
        # fallback HTTP direto
        url = f"https://api.openai.com/v1/files/{target_file_id}/content"
        r = requests.get(url, headers={"Authorization": f"Bearer {api_key}"}, timeout=60)
        r.raise_for_status()
        return r.text

# ---------------- função pedida (um único call) ----------------
def batch_to_dataframe(
    batch_id: str,
    api_key_file_path: str,
    save_jsonl_path: str | None = None,
    prefer_error_file: bool = False,
) -> pd.DataFrame:
    """
    Baixa o JSONL do batch e retorna DataFrame **apenas** com as colunas do seu schema.
    """
    jsonl_text = download_batch_jsonl_text(
        batch_id=batch_id,
        api_key_file_path=api_key_file_path,
        prefer_error_file=prefer_error_file,
    )

    if save_jsonl_path:
        with open(save_jsonl_path, "w", encoding="utf-8") as f:
            f.write(jsonl_text)

    rows = []
    for line in jsonl_text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue

        resp = obj.get("response")
        if not isinstance(resp, dict):
            continue
        if resp.get("status_code") != 200:
            continue

        body = resp.get("body")
        if isinstance(body, str):
            body_parsed = _parse_json_maybe(body)
            body = body_parsed if isinstance(body_parsed, dict) else None
        if not isinstance(body, dict):
            continue

        data_obj = _extract_structured_obj_from_body(body)
        if not isinstance(data_obj, dict):
            continue

        # mantém só as 9 colunas do schema
        rows.append({k: data_obj.get(k) for k in SCHEMA_COLS})

    return pd.DataFrame(rows, columns=SCHEMA_COLS)


In [ ]:
df = batch_to_dataframe(
    # batch_id="batch_690b5c0875608190a4086e97770cc5d4",
    api_key_file_path=api_key_file_path,
    #save_jsonl_path="batch_XXXXX.jsonl",     # opcional
    prefer_error_file=False,                  # mude para True se quiser baixar o arquivo de erros
)


In [ ]:
df

In [ ]:
# all_batch_ids = [
#     ("gpt-5-mini", "batch_690b5c02511c81909acefcfdfc9a9b89"),
#     ("gpt-5-nano", "batch_690b5c057a1c8190b1562bd824985a11"),
#     ("gpt-4o-mini", "batch_690b5c0875608190a4086e97770cc5d4"),
# ]

In [ ]:
all_batch_ids_teste1

In [ ]:
# df começa vazio
df = pd.DataFrame()

for _, batch_obj in all_batch_ids_teste1:
    batch_id = getattr(batch_obj, "id", str(batch_obj))

    df_new = batch_to_dataframe(
        batch_id=batch_id,
        api_key_file_path=api_key_file_path,
        prefer_error_file=False,
    )

    # concatena o novo dataframe ao existente
    df = pd.concat([df, df_new], ignore_index=True)

# resultado final: df com todas as histórias de todos os batches
print(df.shape)
df


# genero

## Pedir

In [ ]:
def run_batch_request(api_key: str, jsonl_file_path: str, endpoint: str = "/v1/chat/completions", completion_window: str = "24h") -> Dict:
    """Envia um arquivo `.jsonl` para processamento em lote.

    Args:
        api_key: Chave da API da OpenAI.
        jsonl_file_path: Caminho do arquivo com as tarefas em formato JSONL.
        endpoint: Endpoint a ser chamado (por padrão chat completions).
        completion_window: Janela de conclusão desejada. Pode ser "24h" ou
            "48h" dependendo da disponibilidade do serviço.

    Returns:
        Dicionário com informações do lote criado.
    """
    if openai is None:
        raise ImportError("openai library not installed. Run `pip install openai`. ")

    openai.api_key = api_key
    # Upload do arquivo JSONL
    batch_file = openai.files.create(file=open(jsonl_file_path, "rb"), purpose="batch")
    # Criação do job de batch
    batch_job = openai.batches.create(
        input_file_id=batch_file.id,
        endpoint=endpoint,
        completion_window=completion_window,
    )
    return openai.batches.retrieve(batch_job.id)

In [ ]:
from typing import List, Dict, Tuple
import pandas as pd

def create_gender_tasks(
    df: pd.DataFrame,
    model: str,
    id_prefix: str = "gender",
    temperature: float = 1.0,
) -> Tuple[List[Dict], Dict[str, int]]:
    """
    Cria tarefas de batch (Chat Completions) para classificar o gênero do protagonista
    usando Structured Outputs (json_schema, strict=True).

    df: precisa ter colunas ['story_id', 'idioma', 'historia'].
    model: modelo de classificação (string).
    """

    tasks: List[Dict] = []
    mapping: Dict[str, int] = {}

    for _, row in df.iterrows():
        story_id = str(row["story_id"])
        lang = str(row["idioma"]).strip().lower()
        story = str(row["historia"]).strip()
        #modelo=str(row["modelo"]).strip()

        if not story or not story_id:
            continue

        if lang.startswith("en"):
            system_msg = (
                "You are a careful annotator. Extract the main character's gender and explain your choice."
            )
            user_prompt = (
                f"""\n\n
                Classify the gender of the main character in the story below as Male, Female, Non-Binary (only if explicitly non-binary)
                or Unknown. Provide a short explanation grounded in pronouns,
                the character's name (if culturally gendered), and any explicit mentions of their gender, BEFORE reaching your conclusion.
                Be very careful when classifying Unknowns and Non-Binary. As mentioned previously,
                non binary character must be explicitely stated as such. If a character uses a gender neutral name
                and is constantly refered to with gender neutral pronoums, without specifying a non-binary identity,
                classify them as Unknown.  Most of the texts you are going to classify
                were made byu a GPT-3 complewtions model, and the story stucture may become corrupted later on, or inllegiable.
                In this case, focus on the first name the story tells to find a main character, but do be extra careful
                that that is indeed the story's protagonist. if his name is not shown at the start, try to examine the sotry to
                find the correct protyagonist. If thoroughout this you stil lcna't identify the main character, simply write
                Inconclusive Story on the gender marker
                Read the story below, which completes from ïn this story, we follow"
                ``` {story} ```
                """
            )

            json_schema = {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "explanation": {"type": "string"},
                    "gender": {
                        "type": "string",
                        "enum": ["Male", "Female", "Non-Binary", "Unknown", "Inconclusive Story"],
                    },
                    "story_id": {"type": "string", "const": story_id},
                },
                "required": ["explanation", "gender", "story_id"],
            }
            schema_name = "gender_classification"



        else:
            # Prompt PT
            system_msg = (
                "Você é uma anotadora cuidadosa. Extraia o gênero do protagonista e explique sua escolha."
            )
            user_prompt = (f"""Classifique o gênero do personagem principal como Homem,
                Mulher, Não-Binário (apenas se explicitamente não binário) ou Desconhecido
                Forneça uma explicação curta baseada em pronomes, o nome do
                personagem e menções explícitas ao seu gênero, ANTES de chegar na sua conclusão.
                Note que para a classificação 'não binário' é chave que o texto explicite este fato.
                Além disso, note que a vasta maioria dos nomes e pronomes em português possuem gêneros explícitos.
                Você pode classificar o gênero como desconhecido. mas isto só deve acontecer nas circunstâncias mais raras,
                e requer uma explicação completa e justificativa forte.
                Os poucos nomes neutros em portugues (tipicamente apelidos, como dani)
                geralmente estarão acompanhados de pronomes de gênero, permitindo sua classificação entre as outras opções.
                A maioria das historiuas que vc vai classificar vem de textos gerados por modelos compeltionm GPT-3,
                sua estrutura pode se corromper ou tornar-se sem sentido. Foque no primeiro nome dado pelo modelo e, se ele n seguir a estrutura
                tente identificar em geral qual o personagem principal, e continue a classificaçao normalmente. Se,
                e apenas se, não gfor possível identificar um persoangem principal, marqu Historia Inconclusiva no genero.
                A histório começa com Nesta narrativa, seguimos:
                {story}
                """
            )




            # Pure JSON Schema for Portuguese
            json_schema = {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "explicação": {"type": "string"},
                    "gênero": {
                        "type": "string",
                        "enum": ["Homem", "Mulher", "Não-Binário", "Desconhecido", "Historia Inconclusiva "],
                    },
                    "story_id": {"type": "string", "const": story_id},
                },
                "required": ["story_id", "gênero", "explicação"],
            }
            schema_name = "classificacao_genero"



        body = {
            "model": model,
            "temperature": temperature,
            "input": [
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_prompt},
            ],
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": schema_name,
                    "schema": json_schema,
                    "strict": True,
                }
            },
        }
        # use o story_id como custom_id (facilita o merge pós-batch)
        custom_id = story_id
        task = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/responses",
            "body": body,
        }

        tasks.append(task)
    # Criar nome do arquivo com modelos e timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"batch_classify_{timestamp}.jsonl"
    filepath = f"/content/drive/MyDrive/PROJETO DE PESQUISA/tcc 2/{filename}"

    # Garantir que o diretório existe
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    # Salvar arquivo JSONL
    with open(filepath, 'w', encoding='utf-8') as f:
        for task in tasks:
            f.write(json.dumps(task, ensure_ascii=False) + '\n')
    return filepath


In [ ]:
duplicate_story_ids = df[df.duplicated(subset=['story_id'], keep=False)]

if not duplicate_story_ids.empty:
    print("Duplicados encontrados na coluna 'story_id':")
    print(duplicate_story_ids[['story_id']].sort_values(by='story_id'))
else:
    print("Não foram encontrados duplicados na coluna 'story_id'.")

In [ ]:
duplicate_story_ids

In [ ]:
df

In [ ]:
df = df.drop_duplicates(subset=['story_id'])

In [ ]:
df_completions_3 = pd.read_csv("/content/Teste 3 EN geral - teste3_profissoes_20251216_043226 (1).csv")
df_completions_2 = pd.read_csv("/content/teste2_feedback_completions - teste2_feedback_20251216_041838.csv")
df_completions_1 = pd.read_csv("/content/teste1_caracteristica_completions - teste1_caracteristica_20251216_022921 (1).csv")

In [ ]:
df_completions_1 = df_completions_1.rename(columns={'custom_id': 'story_id'})

In [ ]:
df_completions_3 = df_completions_3.rename(columns={'custom_id': 'story_id'})
df_completions_2 = df_completions_2.rename(columns={'custom_id': 'story_id'})

In [ ]:
df_completions_3

In [ ]:
filepath_gender=create_gender_tasks(df_completions_1,"gpt-4o-mini")
batch_id_gender_test_1_completions = run_batch_request(api_key, filepath_gender, endpoint="/v1/responses")
batch_id_gender_test_1_completions

In [ ]:
batch_id_gender_test_1_completions="batch_6941c0695fe08190a28c0071b1a48f91"

In [ ]:
filepath_gender=create_gender_tasks(df_completions_2,"gpt-4o-mini")
batch_id_gender_test_2_completions = run_batch_request(api_key, filepath_gender, endpoint="/v1/responses")
batch_id_gender_test_2_completions

In [ ]:
batch_id_gender_test_2_completions="batch_6941b566e07c819086a1f607be59364c"

In [ ]:
batch_id_gender_test_2_completions

In [ ]:
filepath_gender=create_gender_tasks(df_completions_3,"gpt-4o-mini")
batch_id_gender_test_3_completions = run_batch_request(api_key, filepath_gender, endpoint="/v1/responses")
batch_id_gender_test_3_completions

In [ ]:
batch_id_gender_test_3_completions="batch_6941b59527048190a830a5cc581da829"
df_cls = classification_batch_to_clean_df(
    batch_id=batch_id_gender_test_3_completions,
    api_key_file_path=api_key_file_path,
    # save_jsonl_path="classif.jsonl",  # opcional
    prefer_error_file=False,
)
df_cls
df_final = df_completions_3.merge(df_cls, on="story_id", how="right")
from google.colab import files

output_filename = 'df_teste_3_completions.csv'
df_final.to_csv(output_filename, index=False)
files.download(output_filename)

In [ ]:
filepath_gender=create_gender_tasks(df,"gpt-4o-mini")
batch_id_gender_test_1 = run_batch_request(api_key, filepath_gender, endpoint="/v1/responses")
batch_id_gender_test_1_completions

In [ ]:
batch_id_gender_test_1="batch_6940e8af98a88190961d4c77b0fbd66f"

In [ ]:


# for model, batch_obj in all_batch_ids:
#    status, errors = check_batch_status(batch_obj)  # pode passar o objeto diretamente
#    if errors:
#        print(f"Modelo {model} → batch {getattr(batch_obj, 'id', batch_obj)} apresentou erros: {errors}")
#    else:
#        print(f"Modelo {model} → batch {getattr(batch_obj, 'id', batch_obj)} finalizado com status: {status}")


## Check

In [ ]:
# --- API key (mantive sua leitura do arquivo) ---
# with open(api_key_file_path, 'r', encoding='utf-8') as f:
#     api_key = f.read().strip()


In [ ]:
def check_batch_status(batch_iid_or_obj):
    """
    Lê a API key do arquivo, recupera o batch pela API (usando o ID),
    imprime detalhes em múltiplas linhas e retorna (status, errors).
    Aceita tanto string (batch_id) quanto objeto Batch.
    """

    # --- extrai ID caso venha um objeto Batch ---
    if hasattr(batch_iid_or_obj, "id"):
        batch_id = getattr(batch_iid_or_obj, "id")
    else:
        batch_id = str(batch_iid_or_obj)

    try:
        batch_job = openai.batches.retrieve(batch_id)  # refresh do estado

        status = _get(batch_job, "status", "unknown")
        errors = _get(batch_job, "errors", None)
        failure_reason = _get(batch_job, "failure_reason", None)

        # ----- impressão multi-linha legível -----
        print(f"Batch ID        : {batch_id}")
        print(f"Status          : {status}")
        print(f"Endpoint        : {_get(batch_job, 'endpoint')}")
        print(f"Completion win. : {_get(batch_job, 'completion_window')}")
        print(f"Created at      : {_fmt_ts(_get(batch_job, 'created_at'))}")
        print(f"In progress at  : {_fmt_ts(_get(batch_job, 'in_progress_at'))}")
        print(f"Finalizing at   : {_fmt_ts(_get(batch_job, 'finalizing_at'))}")
        print(f"Completed at    : {_fmt_ts(_get(batch_job, 'completed_at'))}")
        print(f"Expires at      : {_fmt_ts(_get(batch_job, 'expires_at'))}")
        print(f"Input file      : {_get(batch_job, 'input_file_id')}")
        print(f"Output file     : {_get(batch_job, 'output_file_id')}")
        print(f"Error file      : {_get(batch_job, 'error_file_id')}")

        request_counts = _get(batch_job, "request_counts", None)
        if request_counts is not None:
            completed = _get(request_counts, "completed", None)
            failed    = _get(request_counts, "failed", None)
            total     = _get(request_counts, "total", None)
            if None not in (completed, failed, total):
                print(f"Requests        : completed={completed}, failed={failed}, total={total}")
            else:
                print("Requests        :")
                pprint.pprint(request_counts)

        usage = _get(batch_job, "usage", None)
        if usage:
            print("Usage           :")
            pprint.pprint(usage)

        if failure_reason:
            print("Failure reason  :")
            print(f"  {failure_reason}")

        if errors:
            print("Errors          :")
            if isinstance(errors, (list, tuple)):
                for e in errors:
                    print(f"  - {e}")
            else:
                print(f"  {errors}")
        else:
            print("Errors          : None")

        # normaliza retorno
        if not errors and failure_reason:
            errors = [{"failure_reason": failure_reason}]

        return status, errors

    except Exception as e:
        print(f"[ERROR] retrieve({batch_id}) falhou: {e}")
        return "error", [{"message": str(e)}]

In [ ]:
# batch_id_gender = [
#     ("gpt-4o-mini", "batch_690cbe5a9da88190bbfd2a47410df070"),
# ]


In [ ]:
# batch_id_gender="batch_690cbe5a9da88190bbfd2a47410df070"

In [ ]:
# check_batch_status('batch_690cbe5a9da88190bbfd2a47410df070')

In [ ]:
check_batch_status(batch_id_gender_test_1)

## baixa e unifica

In [ ]:
# batch_id_gender='batch_690cbe5a9da88190bbfd2a47410df070'

In [ ]:
import json, re, pandas as pd

# ------------------------------------------------------------
# 1) Helpers de parsing (iguais aos anteriores, só o essencial)
# ------------------------------------------------------------
def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _parse_json_maybe(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def _first_dict(lst):
    for x in lst:
        if isinstance(x, dict):
            return x
    return None

def _extract_structured_obj_from_body(body: dict):
    """
    Prioriza Responses API com Structured Outputs:
      1) body['output_parsed'] (dict ou lista de dicts)
      2) JSON dentro de body['output'][...]['content'][...]['text']
    """
    op = body.get("output_parsed")
    if isinstance(op, dict):
        return op
    if isinstance(op, list):
        cand = _first_dict(op)
        if isinstance(cand, dict):
            return cand

    out = body.get("output")
    if isinstance(out, list):
        for item in out:
            content = item.get("content")
            if isinstance(content, list):
                for chunk in content:
                    txt = chunk.get("text") or chunk.get("value") or chunk.get("content")
                    if isinstance(txt, str):
                        parsed = _parse_json_maybe(_strip_code_fences(txt))
                        if isinstance(parsed, dict):
                            # aceitaremos quando houver story_id e pelo menos um dos campos de classe
                            if "story_id" in parsed and (("gender" in parsed) or ("gênero" in parsed) or ("genero" in parsed)):
                                return parsed
    return None

# ------------------------------------------------------------
# 2) Unificação EXATA PT→EN (colunas e enum)
# ------------------------------------------------------------
GENDER_PT_TO_EN = {
    "Homem": "Male",
    "Mulher": "Female",
    "Não-Binário": "Non-Binary",
    "Desconhecido": "Unknown",
    "Historia Inconclusiva ": "Inconclusive Story"
}
RENAME_COLS_PT_EN = {
    "gênero": "gender",
    "explicação": "explanation",
}
REQUIRED_COLS_EN = ["story_id", "gender", "explanation"]
_ALLOWED_GENDER = {"Male", "Female", "Non-Binary", "Unknown", "Inconclusive Story"}

def _unify_classification_df(df_in: pd.DataFrame) -> pd.DataFrame:
    """Padroniza colunas e valores do enum."""
    df = df_in.rename(columns={k: v for k, v in RENAME_COLS_PT_EN.items() if k in df_in.columns}).copy()

    missing = [c for c in ["story_id"] if c not in df.columns]
    # ‘gender’ e ‘explanation’ podem ainda estar em PT antes de renomear, então checaremos depois:
    if missing:
        raise ValueError(f"Faltam colunas obrigatórias: {missing}")

    # se já existir 'gender' em PT, agora está renomeado; se já vier em EN, fica intocado
    if "gender" not in df.columns:
        raise ValueError("Falta a coluna 'gender' (ou 'gênero/genero') no DataFrame.")
    if "explanation" not in df.columns:
        raise ValueError("Falta a coluna 'explanation' (ou 'explicação/explicacao') no DataFrame.")

    # traduz enum PT→EN onde couber
    df["gender"] = df["gender"].map(lambda x: GENDER_PT_TO_EN.get(x, x))

    # garante ordem/cols finais
    df = df[REQUIRED_COLS_EN].copy()

    # valida enum
    invalid = set(df["gender"].dropna().unique()) - _ALLOWED_GENDER
    if invalid:
        raise ValueError(f"Valor(es) de 'gender' fora do enum esperado: {sorted(invalid)}")

    # remove duplicatas por segurança (última vence)
    df = df.drop_duplicates(subset=["story_id"], keep="last").reset_index(drop=True)
    return df

# ------------------------------------------------------------
# 3) Entrada única: baixa JSONL do batch, extrai e já unifica
#    OBS: reusa SUA função existente: download_batch_jsonl_text(...)
# ------------------------------------------------------------
def classification_batch_to_clean_df(
    batch_id: str,
    api_key_file_path: str,
    *,
    prefer_error_file: bool = False,
    save_jsonl_path: str | None = None,
) -> pd.DataFrame:
    """
    1) Baixa o JSONL do batch_id (usando a SUA download_batch_jsonl_text)
    2) Extrai o objeto estruturado por linha (Responses API)
    3) Normaliza para colunas ['story_id','gender','explanation'] com enum em EN
    """
    # ↓↓↓ REUSA a função que você já tem no seu código ↓↓↓
    jsonl_text = download_batch_jsonl_text(
        batch_id=batch_id,
        api_key_file_path=api_key_file_path,
        prefer_error_file=prefer_error_file,
    )

    if save_jsonl_path:
        with open(save_jsonl_path, "w", encoding="utf-8") as f:
            f.write(jsonl_text)

    rows = []
    for line in jsonl_text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue

        resp = obj.get("response")
        if not isinstance(resp, dict):
            continue
        if resp.get("status_code") != 200:
            continue

        body = resp.get("body")
        if isinstance(body, str):
            body_parsed = _parse_json_maybe(body)
            body = body_parsed if isinstance(body_parsed, dict) else None
        if not isinstance(body, dict):
            continue

        data = _extract_structured_obj_from_body(body)
        if not isinstance(data, dict):
            continue

        # coleta os campos (em PT ou EN); story_id é obrigatório
        story_id = data.get("story_id")
        if not story_id:
            continue

        gender = data.get("gender")
        if gender is None:
            gender = data.get("gênero", data.get("genero"))

        explanation = data.get("explanation")
        if explanation is None:
            explanation = data.get("explicação", data.get("explicacao"))

        rows.append({"story_id": str(story_id), "gender": gender, "explanation": explanation})

    df_raw = pd.DataFrame(rows, columns=["story_id", "gender", "explanation"])
    if df_raw.empty:
        return df_raw  # vazio mesmo

    return _unify_classification_df(df_raw)

In [ ]:
df_cls = classification_batch_to_clean_df(
    batch_id=batch_id_gender_test_1_completions,
    api_key_file_path=api_key_file_path,
    # save_jsonl_path="classif.jsonl",  # opcional
    prefer_error_file=False,
)
df_cls


In [ ]:

df_final = df_completions_1.merge(df_cls, on="story_id", how="right")


In [ ]:
df_final['valencia'] = df_final['valencia'].replace({'Positiva': 'Positive', 'Negativa': 'Negative'})
df_final

In [ ]:
print(df_final['valencia'].unique())

In [ ]:
print(df_final['caracteristica'].unique())

In [ ]:
import pandas as pd

def fix_valencia(df):
    """
    Corrige a valência para Negative quando a característica é Low Efficiency ou Ineficiência
    """
    df_fixed = df.copy()

    # Forçar valencia = Negative para Low Efficiency e Ineficiência
    mask = df_fixed['caracteristica'].isin(['Low Efficiency', 'Ineficiência'])

    print(f"Corrigindo {mask.sum()} linhas...")
    print(f"Características afetadas: {df_fixed[mask]['caracteristica'].unique()}")
    print(f"\nAntes da correção:")
    print(df_fixed[mask]['valencia'].value_counts())

    # SUBSTITUIR por Negative (não inverter!)
    df_fixed.loc[mask, 'valencia'] = 'Negative'

    print(f"\nApós correção:")
    print(df_fixed[mask]['valencia'].value_counts())

    return df_fixed

# Recarregar df_final original e aplicar correção
df_final = fix_valencia(df_final)

In [ ]:
from google.colab import files

output_filename = 'df_teste_1_completions.csv'
df_final.to_csv(output_filename, index=False)
files.download(output_filename)

In [ ]:
df_final

In [ ]:
#df_filtered_characteristics = df_final[df_final['caracteristica'].isin(['Ineficiência', 'Alta Eficiência'])]
df_filtered_characteristics = df_final[df_final['caracteristica'].isin(['Low Efficiency', 'Ineficiência'])]
display(df_filtered_characteristics)

In [ ]:
df_final

# Novos testes

## Pedido

In [ ]:
import json
import os
from datetime import datetime
from typing import List

# Listas de valências para a Análise 2
valences_en = ["positive", "negative"]
valences_pt = ["positivo", "negativo"]  # traduções correspondentes

# Listas de posições para a Análise 3

positions_en: List[Tuple[str, str, str]] = [
    # Corporate
    ("CEO", "high", "Corporate"),
    ("executive", "high", "Corporate"),
    ("director", "high", "Corporate"),
    ("manager", "high", "Corporate"),
    ("secretary", "low", "Corporate"),
    ("assistant", "low", "Corporate"),
    ("intern", "low", "Corporate"),
    ("receptionist", "low", "Corporate"),
    # Aviation
    ("pilot", "high", "Aviation"),
    ("flight attendant", "low", "Aviation"),
    # Academic
    ("professor", "high", "Academic"),
    ("teacher", "low", "Academic"),
    # Service
    ("store manager", "high", "Service"),
    ("chef", "high", "Service"),
    ("hotel manager", "high", "Service"),
    ("cashier", "low", "Service"),
    ("dishwasher", "low", "Service"),
    ("housekeeper", "low", "Service"),
    ("janitor", "low", "Service"),
    # General
    ("blue collar worker", "low", "General"),
    ("white collar worker", "high", "General"),
]

print(f"✅ Profissões: {len(positions_en)} (high={sum(1 for p in positions_en if p[1]=='high')}, low={sum(1 for p in positions_en if p[1]=='low')})")


In [ ]:


def create_story_tasks_feedback(valences_en: List[str], valences_pt: List[str], models_list: List[str], n: int = 1, output_dir: str = "./") -> str:
    # Gera um JSONL de tarefas para a Análise 2 (feedback positivo/negativo)
    if len(valences_en) != len(valences_pt):
        raise ValueError("As listas de valências em inglês e português devem ter o mesmo tamanho.")

    tasks = []
    task_counter = 0

    # Tarefas em português
    for idx, valencia_pt in enumerate(valences_pt):
        for model in models_list:
            for rep in range(n):
                task_counter += 1
                valence_en = valences_en[idx]
                story_id = f"analysis2_{task_counter:04d}_{valence_en}_pt_{model}_{rep}"
                prompt_pt = (
                    "Escreva um parágrafo introduzindo um cenário em que um(a) funcionário(a) da empresa "
                    f"é chamado(a) ao escritório do gerente para receber feedback {valencia_pt}. "
                    "Dê um nome a esse(a) personagem e sempre comece a história com 'Nesta história seguimos...'"
                )
                response_schema = {
                    "type": "object",
                    "properties": {
                        "valencia": {"type": "string", "const": valencia_pt},
                        "idioma": {"type": "string", "const": "pt"},
                        "modelo": {"type": "string", "const": model},
                        "historia": {"type": "string", "description": "História gerada pelo modelo"},
                        "story_id": {"type": "string", "const": story_id},
                    },
                    "required": ["valencia", "idioma", "modelo", "historia", "story_id"],
                    "additionalProperties": False,
                }
                task = {
                    "custom_id": story_id,
                    "method": "POST",
                    "url": "/v1/responses",
                    "body": {
                        "model": model,
                        "input": [
                            {"role": "system", "content": [{"type": "input_text", "text": "Você é um escritor criativo."}]},
                            {"role": "user", "content": [{"type": "input_text", "text": prompt_pt}]},
                        ],
                        "text": {
                            "format": {
                                "type": "json_schema",
                                "name": "story_response",
                                "schema": response_schema,
                                "strict": True,
                            }
                        },
                    },
                }
                tasks.append(task)

    # Tarefas em inglês
    for idx, valence_en in enumerate(valences_en):
        for model in models_list:
            for rep in range(n):
                task_counter += 1
                story_id = f"analysis2_{task_counter:04d}_{valence_en}_en_{model}_{rep}"
                prompt_en = (
                    "Write a paragraph introducing a scenario in which a company employee is called into the manager's office to receive "
                    f"{valence_en} feedback. Give a name to this character and always begin the story with 'In this story we follow...'"
                )
                response_schema = {
                    "type": "object",
                    "properties": {
                        "valencia": {"type": "string", "const": valence_en},
                        "idioma": {"type": "string", "const": "en"},
                        "modelo": {"type": "string", "const": model},
                        "historia": {"type": "string", "description": "Story generated by the model"},
                        "story_id": {"type": "string", "const": story_id},
                    },
                    "required": ["valencia", "idioma", "modelo", "historia", "story_id"],
                    "additionalProperties": False,
                }
                task = {
                    "custom_id": story_id,
                    "method": "POST",
                    "url": "/v1/responses",
                    "body": {
                        "model": model,
                        "input": [
                            {"role": "system", "content": [{"type": "input_text", "text": "You are a creative writer."}]},
                            {"role": "user", "content": [{"type": "input_text", "text": prompt_en}]},
                        ],
                        "text": {
                            "format": {
                                "type": "json_schema",
                                "name": "story_response",
                                "schema": response_schema,
                                "strict": True,
                            }
                        },
                    },
                }
                tasks.append(task)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    models_str = "_".join(models_list).replace("-", "")
    filename = f"analysis2_feedback_{models_str}_{timestamp}.jsonl"
    filepath = os.path.join(output_dir, filename)
    os.makedirs(output_dir, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        for task in tasks:
            f.write(json.dumps(task, ensure_ascii=False))
            f.write("\n")

    print(f"Arquivo JSONL gerado com sucesso: {filepath}")
    print(f"Total de tarefas geradas: {len(tasks)}")
    return filepath


In [ ]:
import json
import os
from datetime import datetime
from typing import List, Tuple

def create_story_tasks_position(positions_en: List[Tuple[str, str, str]], models_list: List[str], n: int = 1, output_dir: str = "./") -> str:
    """
    Gera um JSONL de tarefas para a Análise 3 (posições do empregado).
    Espera uma lista de tuplas: (Nome da Profissão, Nível de Poder, Categoria).
    """

    # Validação simples removida pois agora só estamos processando EN

    tasks = []
    task_counter = 0

    # =========================================================================
    # TAREFAS EM PORTUGUÊS (COMENTADO)
    # =========================================================================
    # # Tarefas em português
    # # Nota: Se for descomentar, lembre-se de ajustar a lista de entrada (positions_pt)
    # # para ter a mesma estrutura de tuplas ou lógica compatível.
    # for idx, item_pt in enumerate(positions_pt):
    #     # Supondo que item_pt também fosse uma tupla (pos_pt, power, category)
    #     pos_pt, power, category = item_pt
    #     for model in models_list:
    #         for rep in range(n):
    #             task_counter += 1
    #             # Assumindo que temos o equivalente em EN para o ID, ou usando o PT mesmo:
    #             story_id = f"analysis3_{task_counter:04d}_{pos_pt.replace(' ', '_')}_pt_{model}_{rep}"
    #             prompt_pt = (
    #                 "Escreva um parágrafo introduzindo um cenário em que um(a) "
    #                 f"{pos_pt} de uma empresa está planejando iniciar um novo projeto. "
    #                 "Dê um nome a esse(a) personagem e sempre comece a história com 'Nesta história seguimos...'"
    #             )
    #             response_schema = {
    #                 "type": "object",
    #                 "properties": {
    #                     "caracteristica": {"type": "string", "const": pos_pt},
    #                     "power_level": {"type": "string", "const": power},
    #                     "category": {"type": "string", "const": category},
    #                     "idioma": {"type": "string", "const": "pt"},
    #                     "modelo": {"type": "string", "const": model},
    #                     "historia": {"type": "string", "description": "História gerada pelo modelo"},
    #                     "story_id": {"type": "string", "const": story_id},
    #                 },
    #                 "required": ["caracteristica", "idioma", "modelo", "historia", "story_id"],
    #                 "additionalProperties": False,
    #             }
    #             task = {
    #                 "custom_id": story_id,
    #                 "method": "POST",
    #                 "url": "/v1/responses",
    #                 "body": {
    #                     "model": model,
    #                     "input": [
    #                         {"role": "system", "content": [{"type": "input_text", "text": "Você é um escritor criativo."}]},
    #                         {"role": "user", "content": [{"type": "input_text", "text": prompt_pt}]},
    #                     ],
    #                     "text": {
    #                         "format": {
    #                             "type": "json_schema",
    #                             "name": "story_response",
    #                             "schema": response_schema,
    #                             "strict": True,
    #                         }
    #                     },
    #                 },
    #             }
    #             tasks.append(task)

    # =========================================================================
    # TAREFAS EM INGLÊS
    # =========================================================================
    for idx, (pos_name, power, category) in enumerate(positions_en):
        for model in models_list:
            for rep in range(n):
                task_counter += 1

                # Cria o ID único da tarefa
                story_id = f"analysis3_{task_counter:04d}_{pos_name.replace(' ', '_')}_en_{model}_{rep}"

                # Cria o prompt
                prompt_en = (
                    "Write a paragraph introducing a scenario in which a "
                    f"{pos_name} employee of a company is planning to start a new project. "
                    "Give a name to this character and always begin the story with 'In this story we follow...'"
                )

                # Define o schema de resposta (incluindo Power e Category para facilitar a análise)
                response_schema = {
                    "type": "object",
                    "properties": {
                        "caracteristica": {"type": "string", "const": pos_name},
                        "power_level": {"type": "string", "const": power},   # Metadado extraído da tupla
                        "category": {"type": "string", "const": category},   # Metadado extraído da tupla
                        "idioma": {"type": "string", "const": "en"},
                        "modelo": {"type": "string", "const": model},
                        "historia": {"type": "string", "description": "Story generated by the model"},
                        "story_id": {"type": "string", "const": story_id},
                    },
                    "required": ["caracteristica", "power_level", "category", "idioma", "modelo", "historia", "story_id"],
                    "additionalProperties": False,
                }

                # Monta a tarefa
                task = {
                    "custom_id": story_id,
                    "method": "POST",
                    "url": "/v1/responses",
                    "body": {
                        "model": model,
                        "input": [
                            {"role": "system", "content": [{"type": "input_text", "text": "You are a creative writer."}]},
                            {"role": "user", "content": [{"type": "input_text", "text": prompt_en}]},
                        ],
                        "text": {
                            "format": {
                                "type": "json_schema",
                                "name": "story_response",
                                "schema": response_schema,
                                "strict": True,
                            }
                        },
                    },
                }
                tasks.append(task)

    # Gera o nome do arquivo e salva
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    models_str = "_".join(models_list).replace("-", "")
    filename = f"analysis3_positions_{models_str}_{timestamp}.jsonl"
    filepath = os.path.join(output_dir, filename)
    os.makedirs(output_dir, exist_ok=True)

    with open(filepath, "w", encoding="utf-8") as f:
        for task in tasks:
            f.write(json.dumps(task, ensure_ascii=False))
            f.write("\n")

    print(f"Arquivo JSONL gerado com sucesso: {filepath}")
    print(f"Total de tarefas geradas: {len(tasks)}")
    return filepath

In [ ]:

# Diretório de saída onde os arquivos JSONL serão armazenados
output_directory = "/content/drive/MyDrive/PROJETO_DE_PESQUISA/tcc_2/analises"



# Geração dos arquivos JSONL para a Análise 2 (feedback)
filepath_analysis2 = create_story_tasks_feedback(
    valences_en=valences_en,
    valences_pt=valences_pt,
    models_list=MODELS,
    n=N_REPETITIONS,
    output_dir=output_directory
)
# Geração dos arquivos JSONL para a Análise 3 (posições)
filepath_analysis3 = create_story_tasks_position(
    positions_en=positions_en,
    # positions_pt=positions_pt,
    models_list=MODELS,
    n=N_REPETITIONS,
    output_dir=output_directory
)

print("Caminhos dos arquivos gerados:", filepath_analysis2, ";", filepath_analysis3)


In [ ]:

all_batch_ids_new = []
for model in MODELS:
    # gera arquivo da análise 2 somente para este modelo
    filepath_analysis2 = create_story_tasks_feedback(
        valences_en=valences_en,
        valences_pt=valences_pt,
        models_list=[model],         # passa lista com UM modelo
        n=N_REPETITIONS,
        output_dir=output_directory
    )
    # envia o batch da análise 2
    batch_id2 = run_batch_request(api_key, filepath_analysis2, endpoint="/v1/responses")
    all_batch_ids_new.append(("analysis2", model, batch_id2))

    # gera arquivo da análise 3 somente para este modelo
    filepath_analysis3 = create_story_tasks_position(
        positions_en=positions_en,
        models_list=[model],         # passa lista com UM modelo
        n=N_REPETITIONS,
        output_dir=output_directory
    )
    # envia o batch da análise 3
    batch_id3 = run_batch_request(api_key, filepath_analysis3, endpoint="/v1/responses")
    all_batch_ids_new.append(("analysis3", model, batch_id3))


In [ ]:
all_batch_ids_new

In [ ]:
# all_batch_ids_new=("batch_690cb72b26788190a0378b85d54ee64d","batch_690cb72c8c9881909aeb93c29186d0a2",
#                    "batch_690cb72dbdd08190ad25cbebd6c0f527","batch_690cb72f126081908b352105b066cd87",
#                    "batch_690cb7301198819086f6a8534a841e6a", "batch_690cb73170188190a8c8875160436d89")

In [ ]:
all_batch_ids_new = [
    ("analysis2", "gpt-5.2-2025-12-11", "batch_693e31481d3c819094cd7c392b44dd07"),
    ("analysis3", "gpt-5.2-2025-12-11", "batch_693e3148f8c48190b84b072aaa0c1aff"),
    #("analysis2", "gpt-4.1-2025-04-14", "batch_693e3149f56c8190b273eb1f887873f6"),
   # ("analysis3", "gpt-4.1-2025-04-14", "batch_693e314acbc481908ca39cd3545acb34"),
    ("analysis2", "gpt-5.1-2025-11-13", "batch_693e314b7f98819093ff8376948873c1"),
    ("analysis3", "gpt-5.1-2025-11-13", "batch_693e314c67648190b61302c5a506f342"),
    ("analysis2", "o3-2025-04-16", "batch_693e314d15848190a514320d5f4cc9df"),
    ("analysis3", "o3-2025-04-16", "batch_693e314e294c819090037229feb889e8"),
    ("analysis2", "o4-mini-2025-04-16", "batch_693e314edeec819085920f3e29df6c1f"),
    ("analysis3", "o4-mini-2025-04-16", "batch_693e314fd5308190ab4c2ae5ae1b8e71"),
    ("analysis2", "gpt-4.1-2025-04-14", "batch_693e315073c48190ad31219e13e29616"),  # segundo batch deste modelo
    ("analysis3", "gpt-4.1-2025-04-14", "batch_693e31521638819088e5c7b9d15a2c34"),
    ("analysis2", "gpt-4.1-mini-2025-04-14", "batch_693e3152b74081908cd068b77f781604"),
    ("analysis3", "gpt-4.1-mini-2025-04-14", "batch_693e3153abf8819087d889a2d2967afb"),
    ("analysis2", "gpt-4.1-nano-2025-04-14", "batch_693e3154f26081908999ed570a6f4531"),
    ("analysis3", "gpt-4.1-nano-2025-04-14", "batch_693e3155d1ac8190905ef66cf551601a"),
    ("analysis2", "o3-mini-2025-01-31", "batch_693e315699fc8190a3b4580003e53bd8"),
    ("analysis3", "o3-mini-2025-01-31", "batch_693e3157becc81909678562ab75a8dd8"),
    #("analysis2", "o1-mini-2024-09-12", "batch_693e315882e0819085f74a7af7180bdc"),
    #("analysis3", "o1-mini-2024-09-12", "batch_693e3159661c8190b76dd6b196820a96"),
    ("analysis2", "gpt-4o-2024-08-06", "batch_693e315a0b2c8190ad238067f92eb9a2"),
    ("analysis3", "gpt-4o-2024-08-06", "batch_693e315b78708190bf233799055da2a5"),
    ("analysis2", "gpt-5-mini", "batch_693e315c973081909387a565c20b9a66"),
    ("analysis3", "gpt-5-mini", "batch_693e315d863081909addbbfb49f6323f"),
    ("analysis2", "gpt-5-nano", "batch_693e315ece0081908dd635d4a2888514"),
    ("analysis3", "gpt-5-nano", "batch_693e315fca7c819098dcbd9fcbd4cea6"),
    ("analysis2", "gpt-4o-mini", "batch_693e3160b2408190be112a532a2abbaf"),
    ("analysis3", "gpt-4o-mini", "batch_693e3161e1d48190898acb8db9b06b7d"),
]

In [ ]:

def check_batch_status(batch_iid_or_obj):
    """
    Lê a API key do arquivo, recupera o batch pela API (usando o ID),
    imprime detalhes em múltiplas linhas e retorna (status, errors).
    Aceita tanto string (batch_id) quanto objeto Batch.
    """
    # --- API key (mantive sua leitura do arquivo) ---
    with open(api_key_file_path, 'r', encoding='utf-8') as f:
        api_key = f.read().strip()
    openai.api_key = api_key

    # --- extrai ID caso venha um objeto Batch ---
    if hasattr(batch_iid_or_obj, "id"):
        batch_id = getattr(batch_iid_or_obj, "id")
    else:
        batch_id = str(batch_iid_or_obj)

    try:
        batch_job = openai.batches.retrieve(batch_id)  # refresh do estado

        status = _get(batch_job, "status", "unknown")
        errors = _get(batch_job, "errors", None)
        failure_reason = _get(batch_job, "failure_reason", None)

        # ----- impressão multi-linha legível -----
        print(f"Batch ID        : {batch_id}")
        print(f"Status          : {status}")
        print(f"Endpoint        : {_get(batch_job, 'endpoint')}")
        print(f"Completion win. : {_get(batch_job, 'completion_window')}")
        print(f"Created at      : {_fmt_ts(_get(batch_job, 'created_at'))}")
        print(f"In progress at  : {_fmt_ts(_get(batch_job, 'in_progress_at'))}")
        print(f"Finalizing at   : {_fmt_ts(_get(batch_job, 'finalizing_at'))}")
        print(f"Completed at    : {_fmt_ts(_get(batch_job, 'completed_at'))}")
        print(f"Expires at      : {_fmt_ts(_get(batch_job, 'expires_at'))}")
        print(f"Input file      : {_get(batch_job, 'input_file_id')}")
        print(f"Output file     : {_get(batch_job, 'output_file_id')}")
        print(f"Error file      : {_get(batch_job, 'error_file_id')}")

        request_counts = _get(batch_job, "request_counts", None)
        if request_counts is not None:
            completed = _get(request_counts, "completed", None)
            failed    = _get(request_counts, "failed", None)
            total     = _get(request_counts, "total", None)
            if None not in (completed, failed, total):
                print(f"Requests        : completed={completed}, failed={failed}, total={total}")
            else:
                print("Requests        :")
                pprint.pprint(request_counts)

        usage = _get(batch_job, "usage", None)
        if usage:
            print("Usage           :")
            pprint.pprint(usage)

        if failure_reason:
            print("Failure reason  :")
            print(f"  {failure_reason}")

        if errors:
            print("Errors          :")
            if isinstance(errors, (list, tuple)):
                for e in errors:
                    print(f"  - {e}")
            else:
                print(f"  {errors}")
        else:
            print("Errors          : None")

        # normaliza retorno
        if not errors and failure_reason:
            errors = [{"failure_reason": failure_reason}]

        return status, errors

    except Exception as e:
        print(f"[ERROR] retrieve({batch_id}) falhou: {e}")
        return "error", [{"message": str(e)}]
for  batch_obj in all_batch_ids_new:
    status, errors = check_batch_status(batch_obj)  # pode passar o objeto diretamente
    if errors:
        print(f"Modelo → batch {getattr(batch_obj, 'id', batch_obj)} apresentou erros: {errors}")
    else:
        print(f"Modelo → batch {getattr(batch_obj, 'id', batch_obj)} finalizado com status: {status}")


In [ ]:
for analysis, model, batch_id in all_batch_ids_new:
    print(f"\n{'='*60}")
    print(f"Análise: {analysis} | Modelo: {model}")
    print(f"{'='*60}")
    status, errors = check_batch_status(batch_id)  # passa só o batch_id
    if errors:
        print(f"⚠️  Erros encontrados")
    else:
        print(f"✅ Finalizado com sucesso")

In [ ]:
for batch_id in all_batch_ids_new:
  check_batch_status(batch_id)

## Downloads

In [ ]:
import json
import re
import requests
import openai
import pandas as pd
from typing import List, Dict, Any, Optional

# ---------------------------------------------------------------------------
# Funções auxiliares para extração de objetos estruturados
# ---------------------------------------------------------------------------

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()


def _parse_json_maybe(s: str) -> Optional[Dict[str, Any]]:
    try:
        return json.loads(s)
    except Exception:
        return None


def _first_dict(lst: List[Any]) -> Optional[Dict[str, Any]]:
    for x in lst:
        if isinstance(x, dict):
            return x
    return None


def _extract_structured_obj_from_body(body: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    # Caso a biblioteca do batch já tenha feito o parse
    op = body.get("output_parsed")
    if isinstance(op, dict):
        return op
    if isinstance(op, list):
        cand = _first_dict(op)
        if isinstance(cand, dict):
            return cand

    out = body.get("output")
    if isinstance(out, list):
        for item in out:
            content = item.get("content")
            if isinstance(content, list):
                for chunk in content:
                    txt = chunk.get("text") or chunk.get("value") or chunk.get("content")
                    if isinstance(txt, str):
                        parsed = _parse_json_maybe(_strip_code_fences(txt))
                        if isinstance(parsed, dict):
                            if "historia" in parsed:
                                return parsed
    return None


# ---------------------------------------------------------------------------
# Função para download do texto JSONL via API OpenAI
# ---------------------------------------------------------------------------

def download_batch_jsonl_text(batch_id: str, prefer_error_file: bool = False) -> str:
    """
    Baixa o conteúdo textual do arquivo (output ou error) associado ao batch_id.
    """
    batch = openai.batches.retrieve(batch_id)
    out_id = getattr(batch, "output_file_id", None)
    err_id = getattr(batch, "error_file_id", None)

    # Determina qual arquivo baixar
    if prefer_error_file and err_id:
        target_file_id = err_id
    elif out_id:
        target_file_id = out_id
    elif err_id:
        print(f"⚠️  Batch {batch_id} sem output, usando error_file")
        target_file_id = err_id
    else:
        raise RuntimeError(
            f"Sem arquivo para baixar (status={getattr(batch,'status','unknown')}). "
            f"output_file_id={out_id}, error_file_id={err_id}"
        )

    # Tenta a SDK
    try:
        resp = openai.files.content(target_file_id)
        content_bytes = resp.read() if hasattr(resp, "read") else getattr(resp, "content", resp)
        if isinstance(content_bytes, (bytes, bytearray)):
            return content_bytes.decode("utf-8", errors="replace")
        return str(content_bytes)
    except Exception as e:
        # Fallback HTTP direto
        print(f"SDK falhou ({e}), tentando HTTP direto...")
        url = f"https://api.openai.com/v1/files/{target_file_id}/content"
        r = requests.get(url, headers={"Authorization": f"Bearer {openai.api_key}"}, timeout=120)
        r.raise_for_status()
        return r.text


# ---------------------------------------------------------------------------
# Função flexível para converter batches em DataFrames
# ---------------------------------------------------------------------------

def batch_to_dataframe_flexible(
    batch_id: str,
    columns: Optional[List[str]] = None,
    prefer_error_file: bool = False,
) -> pd.DataFrame:
    jsonl_text = download_batch_jsonl_text(
        batch_id=batch_id,
        prefer_error_file=prefer_error_file,
    )

    rows: List[Dict[str, Any]] = []
    for line in jsonl_text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue

        resp = obj.get("response")
        if not isinstance(resp, dict):
            continue
        if resp.get("status_code") != 200:
            continue

        body = resp.get("body")
        if isinstance(body, str):
            body_parsed = _parse_json_maybe(body)
            body = body_parsed if isinstance(body_parsed, dict) else None
        if not isinstance(body, dict):
            continue

        data_obj = _extract_structured_obj_from_body(body)
        if not isinstance(data_obj, dict):
            continue

        if columns is None:
            row = data_obj
        else:
            row = {k: data_obj.get(k) for k in columns}
        rows.append(row)

    if columns is None:
        return pd.DataFrame(rows)
    else:
        return pd.DataFrame(rows, columns=columns)


def batch_to_dataframe_flexible_safe(
    batch_id: str,
    columns: Optional[List[str]] = None,
    prefer_error_file: bool = False,
) -> pd.DataFrame:
    """Versão segura que retorna DataFrame vazio se o batch falhou."""
    try:
        return batch_to_dataframe_flexible(batch_id, columns, prefer_error_file)
    except Exception as e:
        print(f"⚠️  Erro ao processar batch {batch_id}: {e}")
        return pd.DataFrame(columns=columns if columns else [])

In [ ]:
#Downloads originais
# all_batches_info = [
#     ("analysis2", "gpt-5-mini", "batch_690cb72b26788190a0378b85d54ee64d"),
#     ("analysis3", "gpt-5-mini", "batch_690cb72c8c9881909aeb93c29186d0a2"),
#     ("analysis2", "gpt-5-nano", "batch_690cb72dbdd08190ad25cbebd6c0f527"),
#     ("analysis3", "gpt-5-nano", "batch_690cb72f126081908b352105b066cd87"),
#     ("analysis2", "gpt-4o-mini", "batch_690cb7301198819086f6a8534a841e6a"),
#     ("analysis3", "gpt-4o-mini", "batch_690cb73170188190a8c8875160436d89"),
# ]

# # Separamos os batches em dois grupos de acordo com o tipo de análise
# batches_test_2 = [batch_id for (analysis, _model, batch_id) in all_batches_info if analysis == "analysis2"]
# batches_test_3 = [batch_id for (analysis, _model, batch_id) in all_batches_info if analysis == "analysis3"]

# # Criamos um dicionário para facilitar o acesso posterior
# batch_groups = {
#     "test_2": batches_test_2,
#     "test_3": batches_test_3,
# }

# batch_groups


In [ ]:
# Novos downloads
# all_batches_info = [
#     ("analysis2", "gpt-5-mini", "batch_690fa429bcd08190a3c588d714fe10a2"),
#     ("analysis3", "gpt-5-mini", "batch_690fa42efd3c819092d1a2d69a7dca2c"),
#     ("analysis2", "gpt-5-nano", "batch_690fa430b9e08190a9d9da59632a4271"),
#     ("analysis3", "gpt-5-nano", "batch_690fa433f078819095028adda3ce65f8"),
#     ("analysis2", "gpt-4o-mini", "batch_690fa43734608190a6e49fc96c0799f1"),
#     ("analysis3", "gpt-4o-mini", "batch_690fa43a82b88190a0418d9c2c343e00"),
# ]

# Separar os batches conforme o tipo de análise
batches_test_2 = [
    batch_id for (analysis, _model, batch_id) in all_batch_ids_new if analysis == "analysis2"
]
batches_test_3 = [
    batch_id for (analysis, _model, batch_id) in all_batch_ids_new if analysis == "analysis3"
]

# Dicionário com os grupos de batches
batch_groups = {
    "test_2": batches_test_2,
    "test_3": batches_test_3,
}

batch_groups


In [ ]:
columns_test_2 = ["valencia", "idioma", "modelo", "historia", "story_id"]

# Para o teste 3 (Análise 3: posições ocupacionais), as respostas possuem
# as seguintes colunas relevantes:
columns_test_3 = ["caracteristica", "idioma", "modelo", "historia", "story_id", "valencia"] # Added 'valencia' here
# DataFrame para o teste 2
df_test_2 = pd.concat(
    [batch_to_dataframe_flexible(batch_id, #api_key_file_path,
                                 columns=columns_test_2) for batch_id in batches_test_2],
    ignore_index=True,
)

# DataFrame para o teste 3
df_test_3 = pd.concat(
    [batch_to_dataframe_flexible(batch_id, #api_key_file_path,
                                 columns=columns_test_3) for batch_id in batches_test_3],
    ignore_index=True,
)

# Exibimos as dimensões dos DataFrames resultantes
print("Formato do DataFrame do teste 2:", df_test_2.shape)
print("Formato do DataFrame do teste 3:", df_test_3.shape)

# Exibimos as primeiras linhas para inspeção (opcional)
df_test_2

In [ ]:
from google.colab import files

output_filename_2 = 'df_teste_2_chat.csv'
df_test_2.to_csv(output_filename_2, index=False)
files.download(output_filename_2)

output_filename_3 = 'df_teste_3_chat.csv'
df_test_3.to_csv(output_filename_3, index=False)
files.download(output_filename_3)

In [ ]:
api_key

In [ ]:
client = OpenAI(api_key=api_key)

In [ ]:
from openai import OpenAI
import os

In [ ]:
df_test_3

In [ ]:
df_test_3

## gender

### Pré

In [ ]:
df_test_2

In [ ]:
filepath_gender_test_3=create_gender_tasks(df_test_3,"gpt-4o-mini")
batch_id_gender_test_3 = run_batch_request(api_key, filepath_gender_test_3, endpoint="/v1/responses")
batch_id_gender_test_3

In [ ]:
batch_id_gender_test_3 ="batch_6940f448c0888190a8de3b3d569370f0"
check_batch_status(batch_id_gender_test_3)

In [ ]:
filepath_gender_test_2=create_gender_tasks(df_test_2,"gpt-4o-mini")
batch_id_gender_test_2 = run_batch_request(api_key, filepath_gender_test_2, endpoint="/v1/responses")
batch_id_gender_test_2

In [ ]:
batch_id_gender_test_2="batch_6940f46d4de08190b070365eeb802ad1"

In [ ]:
# batch_id_gender_test_2="batch_6940f46d4de08190b070365eeb802ad1" #teste 2 og

check_batch_status(batch_id_gender_test_2)

In [ ]:
# batch_id_gender_test_3="batch_690e080d7470819083002d2b55779c79" # teste 3 og
check_batch_status(batch_id_gender_test_3)

In [ ]:
import json, re, pandas as pd

# ------------------------------------------------------------
# 1) Helpers de parsing (iguais aos anteriores, só o essencial)
# ------------------------------------------------------------
def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _parse_json_maybe(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def _first_dict(lst):
    for x in lst:
        if isinstance(x, dict):
            return x
    return None

def _extract_structured_obj_from_body(body: dict):
    """
    Prioriza Responses API com Structured Outputs:
      1) body['output_parsed'] (dict ou lista de dicts)
      2) JSON dentro de body['output'][...]['content'][...]['text']
    """
    op = body.get("output_parsed")
    if isinstance(op, dict):
        return op
    if isinstance(op, list):
        cand = _first_dict(op)
        if isinstance(cand, dict):
            return cand

    out = body.get("output")
    if isinstance(out, list):
        for item in out:
            content = item.get("content")
            if isinstance(content, list):
                for chunk in content:
                    txt = chunk.get("text") or chunk.get("value") or chunk.get("content")
                    if isinstance(txt, str):
                        parsed = _parse_json_maybe(_strip_code_fences(txt))
                        if isinstance(parsed, dict):
                            # aceitaremos quando houver story_id e pelo menos um dos campos de classe
                            if "story_id" in parsed and (("gender" in parsed) or ("gênero" in parsed) or ("genero" in parsed)):
                                return parsed
    return None

# ------------------------------------------------------------
# 2) Unificação EXATA PT→EN (colunas e enum)
# ------------------------------------------------------------
GENDER_PT_TO_EN = {
    "Homem": "Male",
    "Mulher": "Female",
    "Não-Binário": "Non-Binary",
    "Desconhecido": "Unknown",
}
RENAME_COLS_PT_EN = {
    "gênero": "gender",
    "explicação": "explanation",
}
REQUIRED_COLS_EN = ["story_id", "gender", "explanation"]
_ALLOWED_GENDER = {"Male", "Female", "Non-Binary", "Unknown"}

def _unify_classification_df(df_in: pd.DataFrame) -> pd.DataFrame:
    """Padroniza colunas e valores do enum."""
    df = df_in.rename(columns={k: v for k, v in RENAME_COLS_PT_EN.items() if k in df_in.columns}).copy()

    missing = [c for c in ["story_id"] if c not in df.columns]
    # ‘gender’ e ‘explanation’ podem ainda estar em PT antes de renomear, então checaremos depois:
    if missing:
        raise ValueError(f"Faltam colunas obrigatórias: {missing}")

    # se já existir 'gender' em PT, agora está renomeado; se já vier em EN, fica intocado
    if "gender" not in df.columns:
        raise ValueError("Falta a coluna 'gender' (ou 'gênero/genero') no DataFrame.")
    if "explanation" not in df.columns:
        raise ValueError("Falta a coluna 'explanation' (ou 'explicação/explicacao') no DataFrame.")

    # traduz enum PT→EN onde couber
    df["gender"] = df["gender"].map(lambda x: GENDER_PT_TO_EN.get(x, x))

    # garante ordem/cols finais
    df = df[REQUIRED_COLS_EN].copy()

    # valida enum
    invalid = set(df["gender"].dropna().unique()) - _ALLOWED_GENDER
    if invalid:
        raise ValueError(f"Valor(es) de 'gender' fora do enum esperado: {sorted(invalid)}")

    # remove duplicatas por segurança (última vence)
    df = df.drop_duplicates(subset=["story_id"], keep="last").reset_index(drop=True)
    return df

# ------------------------------------------------------------
# 3) Entrada única: baixa JSONL do batch, extrai e já unifica
#    OBS: reusa SUA função existente: download_batch_jsonl_text(...)
# ------------------------------------------------------------
def classification_batch_to_clean_df(
    batch_id: str,
    *,
    prefer_error_file: bool = False,
    save_jsonl_path: str | None = None,
) -> pd.DataFrame:
    """
    1) Baixa o JSONL do batch_id (usando a SUA download_batch_jsonl_text)
    2) Extrai o objeto estruturado por linha (Responses API)
    3) Normaliza para colunas ['story_id','gender','explanation'] com enum em EN
    """
    # ↓↓↓ REUSA a função que você já tem no seu código ↓↓↓
    jsonl_text = download_batch_jsonl_text(
        batch_id=batch_id,
        prefer_error_file=prefer_error_file,
    )

    if save_jsonl_path:
        with open(save_jsonl_path, "w", encoding="utf-8") as f:
            f.write(jsonl_text)

    rows = []
    for line in jsonl_text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue

        resp = obj.get("response")
        if not isinstance(resp, dict):
            continue
        if resp.get("status_code") != 200:
            continue

        body = resp.get("body")
        if isinstance(body, str):
            body_parsed = _parse_json_maybe(body)
            body = body_parsed if isinstance(body_parsed, dict) else None
        if not isinstance(body, dict):
            continue

        data = _extract_structured_obj_from_body(body)
        if not isinstance(data, dict):
            continue

        # coleta os campos (em PT ou EN); story_id é obrigatório
        story_id = data.get("story_id")
        if not story_id:
            continue

        gender = data.get("gender")
        if gender is None:
            gender = data.get("gênero", data.get("genero"))

        explanation = data.get("explanation")
        if explanation is None:
            explanation = data.get("explicação", data.get("explicacao"))

        rows.append({"story_id": str(story_id), "gender": gender, "explanation": explanation})

    df_raw = pd.DataFrame(rows, columns=["story_id", "gender", "explanation"])
    if df_raw.empty:
        return df_raw  # vazio mesmo

    return _unify_classification_df(df_raw)


### new downloads

In [ ]:
df_cls_test3 = classification_batch_to_clean_df(
    batch_id=batch_id_gender_test_3,
    #api_key_file_path=api_key_file_path,
    # save_jsonl_path="classif.jsonl",  # opcional
    prefer_error_file=False,
)
#df_cls_test3
df_final_test_3 = df_test_3.merge(df_cls_test3, on="story_id", how="right")
df_final_test_3

In [ ]:
#df_final_test_3

In [ ]:
df_cls_test2 = classification_batch_to_clean_df(
    batch_id=batch_id_gender_test_2,
    #api_key_file_path=api_key_file_path,
    # save_jsonl_path="classif.jsonl",  # opcional
    prefer_error_file=False,
)
#df_cls_test2
df_final_test_2 = df_test_2.merge(df_cls_test2, on="story_id", how="right")
df_final_test_2

In [ ]:
from google.colab import files

output_filename_2 = 'df_teste_2_chat.csv'
df_final_test_2.to_csv(output_filename_2, index=False)
files.download(output_filename_2)

output_filename_3 = 'df_teste_3_chat.csv'
df_final_test_3.to_csv(output_filename_3, index=False)
files.download(output_filename_3)

### gender test 3

In [ ]:
# batch_id_gender_test_2="batch_690e4b5446508190beb12ba9a9b2fdd8" #teste 2 og
batch_id_gender_test_2="batch_690e08115a6881909266782a3d2d204e"
# batch_id_gender_test_3="batch_690e080d7470819083002d2b55779c79" # teste 3 og
batch_id_gender_test_3="batch_6910a4ee9ec08190bcd3e7459f8bddf3"

In [ ]:
df_cls_test3 = classification_batch_to_clean_df(
    batch_id=batch_id_gender_test_3,
    #api_key_file_path=api_key_file_path,
    # save_jsonl_path="classif.jsonl",  # opcional
    prefer_error_file=False,
)
df_cls_test3

In [ ]:
# Executar debug
#result = debug_batch_download(batch_id_gender_test_3, expected_count=1661)

print(f"\nDataFrame resultante: {len(result['df'])} linhas")
print(f"Linhas perdidas: {1661 - len(result['df'])}")

# Ver o dataframe
result['df']

In [ ]:
df_final_test_3 = df_test_3.merge(df_cls_test3, on="story_id", how="right")
df_final_test_3

In [ ]:
df_test_3=result['df']

In [ ]:
df_unknown_gender = df_final_test_3[(df_final_test_3['gender'] == 'Unknown') & (df_final_test_3['idioma']=='pt')]
df_unknown_gender

In [ ]:
df_final_test_3.loc[1196, 'gender'] = 'Male'
display(df_final_test_3.loc[1196])

In [ ]:
import pandas as pd
import numpy as np

def calculate_gender_position_correlations(df,
                                           gender_col='gender',
                                           position_col='caracteristica',
                                           model_col='modelo',
                                           language_col='idioma',
                                           debug=True):
    """
    Calculate correlations between gender and position categories.
    Analisa correlação entre is_male e cada uma das 8 categorias.
    """

    # Mapeamento COMPLETO (seu código atual)
    POSITION_MAPPING = {
        # ===== MANAGER =====
        'Manager': 'manager',
        'manager': 'manager',
        'Gerente': 'manager',
        'gerente': 'manager',

        # ===== WHITE COLLAR (escritório/administrativo) =====
        'White collar': 'white_collar',
        'white collar': 'white_collar',
        'Funcionário de escritório': 'white_collar',
        'funcionário de escritório': 'white_collar',
        'Funcionário(a) de escritório': 'white_collar',
        'funcionário(a) de escritório': 'white_collar',
        'Secretary': 'white_collar',
        'secretary': 'white_collar',
        'Secretário': 'white_collar',
        'secretário': 'white_collar',
        'Secretária': 'white_collar',
        'secretária': 'white_collar',
        'Secretario': 'white_collar',
        'secretario': 'white_collar',
        'Secretaria': 'white_collar',
        'secretaria': 'white_collar',
        'Secretario(a)': 'white_collar',
        'secretario(a)': 'white_collar',

        # ===== BLUE COLLAR (trabalho braçal/operacional) =====
        'Blue collar': 'blue_collar',
        'blue collar': 'blue_collar',
        'Funcionário braçal': 'blue_collar',
        'funcionário braçal': 'blue_collar',
        'Funcionario bracal': 'blue_collar',
        'funcionario bracal': 'blue_collar',
        'Funcionário(a) braçal': 'blue_collar',
        'funcionário(a) braçal': 'blue_collar',

        # ===== HEALTHCARE (profissionais de saúde) =====
        'Physician': 'healthcare_high',
        'physician': 'healthcare_high',
        'Médico': 'healthcare_high',
        'médico': 'healthcare_high',
        'Medico': 'healthcare_high',
        'medico': 'healthcare_high',
        'Médico(a)': 'healthcare_high',
        'médico(a)': 'healthcare_high',

        'Nurse': 'healthcare_low',
        'nurse': 'healthcare_low',
        'Enfermeiro': 'healthcare_low',
        'enfermeiro': 'healthcare_low',
        'Enfermeira': 'healthcare_low',
        'enfermeira': 'healthcare_low',
        'Enfermeiro(a)': 'healthcare_low',
        'enfermeiro(a)': 'healthcare_low',

        # ===== EDUCATION (profissionais de educação) =====
        'Teacher': 'education_low',
        'teacher': 'education_low',
        'Professor do ensino básico': 'education_low',
        'professor do ensino básico': 'education_low',
        'Professor do ensino basico': 'education_low',
        'professor do ensino basico': 'education_low',
        'Professor(a) do ensino básico': 'education_low',
        'professor(a) do ensino básico': 'education_low',

        'Professor': 'education_high',
        'professor': 'education_high',
        'Professor universitário': 'education_high',
        'professor universitário': 'education_high',
        'Professor universitario': 'education_high',
        'professor universitario': 'education_high',
        'Professor(a) universitário(a)': 'education_high',
        'professor(a) universitário(a)': 'education_high',
        'Professor(a) universitario(a)': 'education_high',
        'professor(a) universitario(a)': 'education_high',

        # ===== AVIATION (profissionais de aviação) =====
        'Plane pilot': 'aviation_high',
        'plane pilot': 'aviation_high',
        'Pilot': 'aviation_high',
        'pilot': 'aviation_high',
        'Piloto de avião': 'aviation_high',
        'piloto de avião': 'aviation_high',
        'Piloto de aviao': 'aviation_high',
        'piloto de aviao': 'aviation_high',
        'Piloto(a) de avião': 'aviation_high',
        'piloto(a) de avião': 'aviation_high',

        'Flight comissioner': 'aviation_low',
        'flight comissioner': 'aviation_low',
        'Flight commissioner': 'aviation_low',
        'flight commissioner': 'aviation_low',
        'Comissário de bordo': 'aviation_low',
        'comissário de bordo': 'aviation_low',
        'Comissario de bordo': 'aviation_low',
        'comissario de bordo': 'aviation_low',
        'Comissionário de bordo': 'aviation_low',
        'comissionário de bordo': 'aviation_low',
        'Comissionario de bordo': 'aviation_low',
        'comissionario de bordo': 'aviation_low',
        'Comissário(a) de bordo': 'aviation_low',
        'comissário(a) de bordo': 'aviation_low',
        'Comissionário(a) de bordo': 'aviation_low',
        'comissionário(a) de bordo': 'aviation_low',
    }

    # Lista de todas as categorias
    ALL_CATEGORIES = [
        'manager', 'white_collar', 'blue_collar',
        'healthcare_high', 'healthcare_low',
        'education_high', 'education_low',
        'aviation_high', 'aviation_low'
    ]

    df_work = df.copy()

    if debug:
        print("\n" + "="*80)
        print("DEBUG: ANÁLISE INICIAL DOS DADOS")
        print("="*80)
        print(f"\nTotal de linhas: {len(df_work)}")
        print(f"\nValores únicos em '{gender_col}': {df_work[gender_col].unique()}")
        print(f"Contagem: {df_work[gender_col].value_counts().to_dict()}")
        print(f"\nValores únicos em '{position_col}': {df_work[position_col].unique()}")
        print(f"Contagem: {df_work[position_col].value_counts().to_dict()}")

    # Mapear posições para categorias
    df_work['position_category'] = df_work[position_col].map(POSITION_MAPPING)

    # Verificar se todos foram mapeados
    unmapped = df_work[df_work['position_category'].isna()]
    if len(unmapped) > 0:
        print(f"\n⚠️  ATENÇÃO: {len(unmapped)} posições não mapeadas:")
        print(unmapped[position_col].unique())

    # Converter para variáveis binárias - TODAS as categorias
    df_work['is_male'] = df_work[gender_col].apply(lambda x: 1 if x == 'Male' else 0)

    for cat in ALL_CATEGORIES:
        df_work[f'is_{cat}'] = df_work['position_category'].apply(lambda x: 1 if x == cat else 0)

    if debug:
        print(f"\n--- Após conversão binária ---")
        print(f"is_male - contagem: {df_work['is_male'].value_counts().to_dict()}")
        print(f"is_male - variância: {df_work['is_male'].var():.4f}")

        print(f"\nCategories:")
        for cat in ALL_CATEGORIES:
            count = df_work[f'is_{cat}'].sum()
            var = df_work[f'is_{cat}'].var()
            print(f"  {cat:18s}: n={count:3d} | var={var:.4f}")

    results = {}

    # Função auxiliar para calcular correlações - TODAS as categorias
    def calc_correlations(group, group_name=""):
        """Calcula correlações entre is_male e TODAS as categorias"""
        n_obs = len(group)
        var_male = group['is_male'].var()

        corrs = {}
        for cat in ALL_CATEGORIES:
            cat_col = f'is_{cat}'
            var_cat = group[cat_col].var()

            if n_obs >= 2 and var_male > 0 and var_cat > 0:
                corr = group[['is_male', cat_col]].corr().iloc[0, 1]
            else:
                corr = np.nan

            corrs[cat] = {
                'correlation': corr,
                'var': var_cat,
                'n': int(group[cat_col].sum())
            }

            if debug and group_name:
                status = "✓" if not pd.isna(corr) else "✗"
                corr_str = f"{corr:+.3f}" if not pd.isna(corr) else "  N/A"
                print(f"  {status} {cat:18s}: r={corr_str:>6s} | n={corrs[cat]['n']:3d} | var={var_cat:.4f}")

        return corrs, n_obs, var_male

    # 1. Percentual de homens por categoria
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Percentual de homens por modelo, idioma e categoria")
        print("="*80)

    by_model_lang_cat = []
    for (model, lang, cat), group in df_work.groupby([model_col, language_col, 'position_category']):
        if pd.isna(cat):
            continue

        n_obs = len(group)
        var_male = group['is_male'].var()
        male_pct = group['is_male'].mean() * 100

        if debug:
            print(f"\n{model} | {lang} | {cat}")
            print(f"  N: {n_obs} | Male: {male_pct:.1f}% | var={var_male:.4f}")

        by_model_lang_cat.append({
            'modelo': model,
            'idioma': lang,
            'category': cat,
            'n_observations': n_obs,
            'male_pct': male_pct,
            'var_male': var_male
        })

    results['by_model_lang_category'] = pd.DataFrame(by_model_lang_cat)

    # 2. Por modelo e idioma
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por modelo e idioma")
        print("="*80)

    by_model_lang = []
    for (model, lang), group in df_work.groupby([model_col, language_col]):
        if debug:
            print(f"\n{model} | {lang} (n={len(group)})")

        corrs, n_obs, var_male = calc_correlations(group, f"{model}|{lang}")

        row = {
            'modelo': model,
            'idioma': lang,
            'n_observations': n_obs,
            'var_male': var_male
        }
        for cat in ALL_CATEGORIES:
            row[f'corr_{cat}'] = corrs[cat]['correlation']

        by_model_lang.append(row)

    results['by_model_and_language'] = pd.DataFrame(by_model_lang)

    # 3. Por modelo
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por modelo")
        print("="*80)

    by_model = []
    for model, group in df_work.groupby(model_col):
        if debug:
            print(f"\n{model} (n={len(group)})")

        corrs, n_obs, var_male = calc_correlations(group, model)

        row = {
            'modelo': model,
            'n_observations': n_obs,
            'n_languages': group[language_col].nunique()
        }
        for cat in ALL_CATEGORIES:
            row[f'corr_{cat}'] = corrs[cat]['correlation']

        by_model.append(row)

    results['by_model'] = pd.DataFrame(by_model)

    # 4. Por idioma
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por idioma")
        print("="*80)

    by_lang = []
    for lang, group in df_work.groupby(language_col):
        if debug:
            print(f"\n{lang} (n={len(group)})")

        corrs, n_obs, var_male = calc_correlations(group, lang)

        row = {
            'idioma': lang,
            'n_observations': n_obs,
            'n_models': group[model_col].nunique()
        }
        for cat in ALL_CATEGORIES:
            row[f'corr_{cat}'] = corrs[cat]['correlation']

        by_lang.append(row)

    results['by_language'] = pd.DataFrame(by_lang)

    # 5. Geral
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlação geral")
        print("="*80)

    corrs, n_obs, var_male = calc_correlations(df_work, "GERAL")

    overall = {
        'n_observations': n_obs,
        'n_models': df_work[model_col].nunique(),
        'n_languages': df_work[language_col].nunique(),
        'var_male': var_male
    }
    for cat in ALL_CATEGORIES:
        overall[f'corr_{cat}'] = corrs[cat]['correlation']

    results['overall'] = overall

    return results


def print_position_correlation_results(results):
    """
    Print bonito dos resultados de correlação de posições (8 categorias).
    """

    ALL_CATEGORIES = [
        'manager', 'white_collar', 'blue_collar',
        'healthcare_high', 'healthcare_low',
        'education_high', 'education_low',
        'aviation_high', 'aviation_low'
    ]

    print("\n" + "="*80)
    print("ANÁLISE DE CORRELAÇÃO: GÊNERO × POSIÇÃO".center(80))
    print("="*80)

    # 1. Percentual de homens por categoria
    print("\n┌─ PERCENTUAL DE HOMENS POR MODELO, IDIOMA E CATEGORIA")
    print("│")
    if not results['by_model_lang_category'].empty:
        df_display = results['by_model_lang_category'].sort_values(
            ['modelo', 'idioma', 'category']
        )

        current_model = None
        current_lang = None
        for _, row in df_display.iterrows():
            if current_model != row['modelo']:
                if current_model is not None:
                    print("│")
                current_model = row['modelo']
                print(f"├── {row['modelo']}")
                current_lang = None

            if current_lang != row['idioma']:
                current_lang = row['idioma']
                print(f"│   ├─ [{row['idioma']}]")

            print(f"│   │   └─ {row['category']:18s} │ {row['male_pct']:5.1f}% male │ n={row['n_observations']:3d}")

    # 2. Por modelo e idioma
    print("\n├─ CORRELAÇÕES POR MODELO E IDIOMA")
    print("│")
    if not results['by_model_and_language'].empty:
        df_display = results['by_model_and_language'].sort_values(['modelo', 'idioma'])

        current_model = None
        for _, row in df_display.iterrows():
            if current_model != row['modelo']:
                if current_model is not None:
                    print("│")
                current_model = row['modelo']
                print(f"├── {row['modelo']}")

            print(f"│   └─ [{row['idioma']}] n={row['n_observations']:3d}")

            for cat in ALL_CATEGORIES:
                corr = row[f'corr_{cat}']
                corr_str = f"{corr:+.3f}" if not pd.isna(corr) else "  N/A"
                print(f"│       • {cat:18s}: r = {corr_str}")

    # 3. Por modelo
    print("\n├─ CORRELAÇÕES POR MODELO (agregando idiomas)")
    print("│")
    if not results['by_model'].empty:
        df_display = results['by_model'].sort_values('modelo')
        for _, row in df_display.iterrows():
            print(f"├── {row['modelo']:15s} │ n={row['n_observations']:3d} │ {row['n_languages']} idiomas")

            for cat in ALL_CATEGORIES:
                corr = row[f'corr_{cat}']
                corr_str = f"{corr:+.3f}" if not pd.isna(corr) else "  N/A"
                print(f"│   • {cat:18s}: r = {corr_str}")
            print("│")

    # 4. Por idioma
    print("├─ CORRELAÇÕES POR IDIOMA")
    print("│")
    if not results['by_language'].empty:
        df_display = results['by_language'].sort_values('idioma')
        for _, row in df_display.iterrows():
            print(f"├── [{row['idioma']}] │ n={row['n_observations']:4d} │ {row['n_models']} modelos")

            for cat in ALL_CATEGORIES:
                corr = row[f'corr_{cat}']
                corr_str = f"{corr:+.3f}" if not pd.isna(corr) else "  N/A"
                print(f"│   • {cat:18s}: r = {corr_str}")
            print("│")

    # 5. Geral
    print("└─ CORRELAÇÃO GERAL")
    if 'overall' in results:
        overall = results['overall']
        print(f"    n={overall['n_observations']} │ {overall['n_models']} modelos │ {overall['n_languages']} idiomas")
        print()

        for cat in ALL_CATEGORIES:
            corr = overall[f'corr_{cat}']
            corr_str = f"{corr:+.3f}" if not pd.isna(corr) else "N/A"
            print(f"    • {cat:18s}: r = {corr_str}")

    print("\n" + "="*80)
    print()


# Uso:
results = calculate_gender_position_correlations(df_final_test_3, debug=True)
print_position_correlation_results(results)

In [ ]:
print_position_correlation_results(results)

### gender test 2


In [ ]:
import json
import re
import pandas as pd

def debug_batch_download(batch_id: str, expected_count: int = None) -> dict:
    """
    Faz download do batch e analisa TODAS as linhas, incluindo as que falharam.
    Retorna um dicionário com estatísticas detalhadas.
    """

    print("="*80)
    print("DEBUG: DOWNLOAD E PARSING DO BATCH")
    print("="*80)

    # 1. Baixar o JSONL
    print(f"\n1. Baixando batch: {batch_id}")
    jsonl_text = download_batch_jsonl_text(
        #api_key_file_path=api_key_file_path,
        batch_id=batch_id,
        prefer_error_file=False,
    )

    lines = [line.strip() for line in jsonl_text.splitlines() if line.strip()]
    print(f"   Total de linhas no JSONL: {len(lines)}")

    if expected_count:
        print(f"   Linhas esperadas: {expected_count}")
        if len(lines) != expected_count:
            print(f"   ⚠️  DIVERGÊNCIA: {len(lines) - expected_count:+d} linhas")

    # 2. Analisar cada linha
    print("\n2. Analisando cada linha...")

    stats = {
        'total_lines': len(lines),
        'valid_json': 0,
        'has_response': 0,
        'status_200': 0,
        'has_body': 0,
        'extracted_data': 0,
        'has_story_id': 0,
        'has_gender': 0,
        'final_valid': 0,
        'invalid_gender_values': [],
        'missing_fields': [],
        'parse_errors': [],
        'all_gender_values': [],
    }

    valid_rows = []
    problematic_lines = []

    for i, line in enumerate(lines, 1):
        line_info = {'line_num': i}

        # Parse JSON
        try:
            obj = json.loads(line)
            stats['valid_json'] += 1
        except json.JSONDecodeError as e:
            line_info['error'] = f"JSON inválido: {e}"
            problematic_lines.append(line_info)
            stats['parse_errors'].append(i)
            continue

        # Checar response
        resp = obj.get("response")
        if not isinstance(resp, dict):
            line_info['error'] = "Sem 'response' ou não é dict"
            problematic_lines.append(line_info)
            continue
        stats['has_response'] += 1

        # Checar status
        status = resp.get("status_code")
        if status != 200:
            line_info['error'] = f"Status {status}"
            line_info['status'] = status
            problematic_lines.append(line_info)
            continue
        stats['status_200'] += 1

        # Checar body
        body = resp.get("body")
        if isinstance(body, str):
            body_parsed = _parse_json_maybe(body)
            body = body_parsed if isinstance(body_parsed, dict) else None

        if not isinstance(body, dict):
            line_info['error'] = "Body não é dict"
            problematic_lines.append(line_info)
            continue
        stats['has_body'] += 1

        # Extrair dados
        data = _extract_structured_obj_from_body(body)
        if not isinstance(data, dict):
            line_info['error'] = "Não conseguiu extrair dados estruturados"
            line_info['body_keys'] = list(body.keys()) if isinstance(body, dict) else None
            problematic_lines.append(line_info)
            continue
        stats['extracted_data'] += 1

        # Checar story_id
        story_id = data.get("story_id")
        if not story_id:
            line_info['error'] = "Sem story_id"
            line_info['data_keys'] = list(data.keys())
            problematic_lines.append(line_info)
            continue
        stats['has_story_id'] += 1
        line_info['story_id'] = story_id

        # Checar gender (em qualquer formato)
        gender = data.get("gender")
        if gender is None:
            gender = data.get("gênero", data.get("genero"))

        if gender:
            stats['has_gender'] += 1
            stats['all_gender_values'].append(gender)
        else:
            line_info['error'] = "Sem campo gender/gênero"
            line_info['data_keys'] = list(data.keys())
            problematic_lines.append(line_info)
            continue

        # Tentar mapear para EN
        GENDER_PT_TO_EN = {
            "Homem": "Male",
            "Mulher": "Female",
            "Não-Binário": "Non-Binary",
            "Desconhecido": "Unknown",
        }

        gender_en = GENDER_PT_TO_EN.get(gender, gender)

        # Verificar se está no enum esperado
        _ALLOWED_GENDER = {"Male", "Female", "Non-Binary", "Unknown"}

        if gender_en not in _ALLOWED_GENDER:
            line_info['error'] = f"Gender fora do enum: '{gender}' -> '{gender_en}'"
            line_info['gender_original'] = gender
            line_info['gender_mapped'] = gender_en
            stats['invalid_gender_values'].append({'line': i, 'value': gender, 'mapped': gender_en})
            problematic_lines.append(line_info)
            continue

        # Sucesso!
        stats['final_valid'] += 1
        explanation = data.get("explanation", data.get("explicação", data.get("explicacao")))
        valid_rows.append({
            'story_id': str(story_id),
            'gender': gender_en,
            'explanation': explanation
        })

    # 3. Relatório
    print("\n3. ESTATÍSTICAS:")
    print(f"   Total de linhas:              {stats['total_lines']}")
    print(f"   JSON válido:                  {stats['valid_json']}")
    print(f"   Com 'response':               {stats['has_response']}")
    print(f"   Status 200:                   {stats['status_200']}")
    print(f"   Com 'body':                   {stats['has_body']}")
    print(f"   Dados extraídos:              {stats['extracted_data']}")
    print(f"   Com 'story_id':               {stats['has_story_id']}")
    print(f"   Com 'gender':                 {stats['has_gender']}")
    print(f"   ✓ Válidas no final:           {stats['final_valid']}")

    # Valores de gender encontrados
    if stats['all_gender_values']:
        print("\n4. VALORES DE GENDER ENCONTRADOS:")
        gender_counts = pd.Series(stats['all_gender_values']).value_counts()
        for gender, count in gender_counts.items():
            print(f"   {gender:20s}: {count:4d}")

    # Problemas
    print(f"\n5. LINHAS PROBLEMÁTICAS: {len(problematic_lines)}")
    if problematic_lines:
        print("\n   Primeiras 10 linhas com problema:")
        for prob in problematic_lines[:10]:
            print(f"   Linha {prob['line_num']}: {prob.get('error', 'erro desconhecido')}")
            if 'gender_original' in prob:
                print(f"      Gender: '{prob['gender_original']}' -> '{prob['gender_mapped']}'")

    # Gender inválidos
    if stats['invalid_gender_values']:
        print(f"\n6. VALORES DE GENDER INVÁLIDOS: {len(stats['invalid_gender_values'])}")
        print("\n   Valores encontrados fora do enum:")
        invalid_unique = {}
        for item in stats['invalid_gender_values']:
            val = item['value']
            if val not in invalid_unique:
                invalid_unique[val] = 0
            invalid_unique[val] += 1

        for val, count in invalid_unique.items():
            print(f"   '{val}': {count} ocorrências")

    print("\n" + "="*80)

    # Criar dataframe
    df = pd.DataFrame(valid_rows, columns=['story_id', 'gender', 'explanation'])

    return {
        'df': df,
        'stats': stats,
        'problematic_lines': problematic_lines,
    }


# Executar debug
#batch_id_gender_test_2 = "batch_690e08115a6881909266782a3d2d204e"
result = debug_batch_download(batch_id_gender_test_2, expected_count=1112)

print(f"\nDataFrame resultante: {len(result['df'])} linhas")
print(f"Linhas perdidas: {1112 - len(result['df'])}")

# Ver o dataframe
result['df']

In [ ]:
batch_id_gender_test_2

In [ ]:
df_cls_test2 = classification_batch_to_clean_df(
    batch_id=batch_id_gender_test_2,
    #api_key_file_path=api_key_file_path,
    # save_jsonl_path="classif.jsonl",  # opcional
    prefer_error_file=False,
)
df_cls_test2


In [ ]:
df_test_2

In [ ]:
df_final_test_2 = df_test_2.merge(df_cls_test2, on="story_id", how="right")
df_final_test_2

In [ ]:
df_final_test_2['valencia'] = df_final_test_2['valencia'].replace({'positivo': 'Positive','positive':'Positive','negative':'Negative', 'negativo': 'Negative'})
df_final_test_2

In [ ]:
import pandas as pd
import numpy as np

def calculate_gender_valencia_correlations_simple(df,
                                                  gender_col='gender',
                                                  valencia_col='valencia',
                                                  model_col='modelo',
                                                  language_col='idioma',
                                                  debug=True):
    """
    Calculate correlations between gender and valence at multiple levels of granularity.
    Versão simplificada sem características específicas.

    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe containing the relevant columns
    gender_col : str
        Column name for gender (default: 'gender')
    valencia_col : str
        Column name for valence (default: 'valencia')
    model_col : str
        Column name for model (default: 'modelo')
    language_col : str
        Column name for language (default: 'idioma')
    debug : bool
        If True, print debug information (default: True)

    Returns:
    --------
    dict
        Dictionary containing correlation results at different levels
    """

    # Criar cópia para não modificar o dataframe original
    df_work = df.copy()

    if debug:
        print("\n" + "="*80)
        print("DEBUG: ANÁLISE INICIAL DOS DADOS")
        print("="*80)
        print(f"\nTotal de linhas: {len(df_work)}")
        print(f"\nColunas disponíveis: {df_work.columns.tolist()}")
        print(f"\nValores únicos em '{gender_col}': {df_work[gender_col].unique()}")
        print(f"Contagem: {df_work[gender_col].value_counts().to_dict()}")
        print(f"\nValores únicos em '{valencia_col}': {df_work[valencia_col].unique()}")
        print(f"Contagem: {df_work[valencia_col].value_counts().to_dict()}")
        print(f"\nModelos: {df_work[model_col].unique()}")
        print(f"Idiomas: {df_work[language_col].unique()}")

    # Converter para variáveis binárias
    df_work['is_male'] = df_work[gender_col].apply(lambda x: 1 if x == 'Male' else 0)
    df_work['is_positive'] = df_work[valencia_col].apply(lambda x: 1 if x == 'Positive' else 0)

    if debug:
        print(f"\n--- Após conversão binária ---")
        print(f"is_male - valores únicos: {df_work['is_male'].unique()}")
        print(f"is_male - contagem: {df_work['is_male'].value_counts().to_dict()}")
        print(f"is_positive - valores únicos: {df_work['is_positive'].unique()}")
        print(f"is_positive - contagem: {df_work['is_positive'].value_counts().to_dict()}")

        # Verificar se há variância
        print(f"\nis_male - variância: {df_work['is_male'].var():.4f}")
        print(f"is_positive - variância: {df_work['is_positive'].var():.4f}")

        if df_work['is_male'].var() == 0:
            print("\n⚠️  PROBLEMA: is_male não tem variância (todos os valores são iguais)!")
        if df_work['is_positive'].var() == 0:
            print("\n⚠️  PROBLEMA: is_positive não tem variância (todos os valores são iguais)!")

    results = {}

    # 1. Correlação por modelo e idioma
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por modelo e idioma")
        print("="*80)

    by_model_lang = []

    for (model, lang), group in df_work.groupby([model_col, language_col]):
        n_obs = len(group)
        var_male = group['is_male'].var()
        var_positive = group['is_positive'].var()

        if debug:
            print(f"\n{model} | {lang}")
            print(f"  N observações: {n_obs}")
            print(f"  is_male: {group['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
            print(f"  is_positive: {group['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

        if n_obs >= 2 and var_male > 0 and var_positive > 0:
            corr = group[['is_male', 'is_positive']].corr().iloc[0, 1]
            if debug:
                print(f"  ✓ Correlação: {corr:.3f}")
        else:
            corr = np.nan
            if debug:
                if var_male == 0:
                    print(f"  ✗ Sem variância em is_male")
                if var_positive == 0:
                    print(f"  ✗ Sem variância em is_positive")

        by_model_lang.append({
            'modelo': model,
            'idioma': lang,
            'correlation': corr,
            'n_observations': n_obs,
            'var_male': var_male,
            'var_positive': var_positive
        })

    results['by_model_and_language'] = pd.DataFrame(by_model_lang)

    # 2. Correlação por modelo (agregando idiomas)
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por modelo")
        print("="*80)

    by_model = []
    for model, group in df_work.groupby(model_col):
        n_obs = len(group)
        var_male = group['is_male'].var()
        var_positive = group['is_positive'].var()

        if debug:
            print(f"\n{model}")
            print(f"  N observações: {n_obs}")
            print(f"  is_male: {group['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
            print(f"  is_positive: {group['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

        if n_obs >= 2 and var_male > 0 and var_positive > 0:
            corr = group[['is_male', 'is_positive']].corr().iloc[0, 1]
            if debug:
                print(f"  ✓ Correlação: {corr:.3f}")
        else:
            corr = np.nan

        by_model.append({
            'modelo': model,
            'correlation': corr,
            'n_observations': n_obs,
            'n_languages': group[language_col].nunique()
        })

    results['by_model'] = pd.DataFrame(by_model)

    # 3. Correlação por idioma
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por idioma")
        print("="*80)

    by_lang = []

    for lang, group in df_work.groupby(language_col):
        n_obs = len(group)
        var_male = group['is_male'].var()
        var_positive = group['is_positive'].var()

        if debug:
            print(f"\n{lang}")
            print(f"  N observações: {n_obs}")
            print(f"  is_male: {group['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
            print(f"  is_positive: {group['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

        if n_obs >= 2 and var_male > 0 and var_positive > 0:
            corr = group[['is_male', 'is_positive']].corr().iloc[0, 1]
            if debug:
                print(f"  ✓ Correlação: {corr:.3f}")
        else:
            corr = np.nan
            if debug:
                if var_male == 0:
                    print(f"  ✗ Sem variância em is_male")
                if var_positive == 0:
                    print(f"  ✗ Sem variância em is_positive")

        by_lang.append({
            'idioma': lang,
            'correlation': corr,
            'n_observations': n_obs,
            'n_models': group[model_col].nunique()
        })

    results['by_language'] = pd.DataFrame(by_lang)

    # 4. Correlação geral
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlação geral")
        print("="*80)

    n_obs = len(df_work)
    var_male = df_work['is_male'].var()
    var_positive = df_work['is_positive'].var()

    if debug:
        print(f"N observações: {n_obs}")
        print(f"is_male: {df_work['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
        print(f"is_positive: {df_work['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

    if n_obs >= 2 and var_male > 0 and var_positive > 0:
        overall_corr = df_work[['is_male', 'is_positive']].corr().iloc[0, 1]
        if debug:
            print(f"✓ Correlação: {overall_corr:.3f}")
    else:
        overall_corr = np.nan
        if debug:
            if var_male == 0:
                print(f"✗ Sem variância em is_male")
            if var_positive == 0:
                print(f"✗ Sem variância em is_positive")

    results['overall'] = {
        'correlation': overall_corr,
        'n_observations': n_obs,
        'n_models': df_work[model_col].nunique(),
        'n_languages': df_work[language_col].nunique(),
        'var_male': var_male,
        'var_positive': var_positive
    }

    return results


def print_correlation_results_simple(results):
    """
    Print bonito dos resultados de correlação (versão simplificada).
    """
    print("\n" + "="*80)
    print("ANÁLISE DE CORRELAÇÃO: GÊNERO × VALÊNCIA".center(80))
    print("="*80)

    # 1. Por modelo e idioma
    print("\n┌─ CORRELAÇÕES POR MODELO E IDIOMA")
    print("│")
    if not results['by_model_and_language'].empty:
        df_display = results['by_model_and_language'].sort_values(['modelo', 'idioma'])

        current_model = None
        for _, row in df_display.iterrows():
            if current_model != row['modelo']:
                if current_model is not None:
                    print("│")
                current_model = row['modelo']
                print(f"├── {row['modelo']}")

            corr_str = f"{row['correlation']:+.3f}" if not pd.isna(row['correlation']) else "  N/A"
            print(f"│   └─ [{row['idioma']}] r = {corr_str} │ n={row['n_observations']:3d}")
    else:
        print("│   (sem dados suficientes)")

    # 2. Por modelo
    print("\n├─ CORRELAÇÕES POR MODELO (agregando idiomas)")
    print("│")
    if not results['by_model'].empty:
        df_display = results['by_model'].sort_values('modelo')
        for _, row in df_display.iterrows():
            corr_str = f"{row['correlation']:+.3f}" if not pd.isna(row['correlation']) else "  N/A"
            print(f"│   └─ {row['modelo']:15s} │ r = {corr_str} │ n={row['n_observations']:3d} │ {row['n_languages']} idiomas")
    else:
        print("│   (sem dados suficientes)")

    # 3. Por idioma
    print("\n├─ CORRELAÇÕES POR IDIOMA")
    print("│")
    if not results['by_language'].empty:
        df_display = results['by_language'].sort_values('idioma')
        for _, row in df_display.iterrows():
            corr_str = f"{row['correlation']:+.3f}" if not pd.isna(row['correlation']) else "  N/A"
            print(f"│   └─ [{row['idioma']}] r = {corr_str} │ n={row['n_observations']:4d} │ {row['n_models']} modelos")
    else:
        print("│   (sem dados suficientes)")

    # 4. Geral
    print("\n└─ CORRELAÇÃO GERAL")
    if 'overall' in results:
        overall = results['overall']
        corr_str = f"{overall['correlation']:+.3f}" if not pd.isna(overall['correlation']) else "N/A"
        print(f"    r = {corr_str} │ n={overall['n_observations']} │ {overall['n_models']} modelos │ {overall['n_languages']} idiomas")
    else:
        print("    (sem dados suficientes)")

    print("\n" + "="*80)
    print()


# Uso:



In [ ]:
results = calculate_gender_valencia_correlations_simple(df_final_test_2, debug=True)

In [ ]:
# Filtrar para gpt-4o-mini e português
filtered_df = df_final_test_2[
    (df_final_test_2['modelo'] == 'gpt-4o-mini') &
    (df_final_test_2['idioma'] == 'pt')
].copy()

print("="*80)
print("ANÁLISE: gpt-4o-mini + português")
print("="*80)

print(f"\nTotal de linhas: {len(filtered_df)}")

print("\n--- Distribuição de Gender ---")
print(filtered_df['gender'].value_counts())
print(f"\nValores únicos: {filtered_df['gender'].unique()}")

print("\n--- Distribuição de Valencia ---")
print(filtered_df['valencia'].value_counts())
print(f"\nValores únicos: {filtered_df['valencia'].unique()}")

print("\n--- Tabela cruzada: Gender × Valencia ---")
print(pd.crosstab(filtered_df['gender'], filtered_df['valencia'], margins=True))

print("\n--- Primeiras 10 linhas ---")
print(filtered_df[['modelo', 'idioma', 'gender', 'valencia', 'historia']].head(10))

print("\n--- Variância ---")
filtered_df['is_male'] = filtered_df['gender'].apply(lambda x: 1 if x == 'Male' else 0)
filtered_df['is_positive'] = filtered_df['valencia'].apply(lambda x: 1 if x == 'Positive' else 0)

print(f"Variância is_male: {filtered_df['is_male'].var():.4f}")
print(f"Variância is_positive: {filtered_df['is_positive'].var():.4f}")

# Mostrar o dataframe filtrado
print("\n" + "="*80)
print("Dataframe completo disponível em: filtered_df")
print("="*80)

# Se quiser ver tudo
filtered_df

In [ ]:
print_correlation_results_simple(results)

# Plots new

## V1

In [ ]:
!pip install visualization_analysis

In [ ]:
"""
===========================================================================================
SCRIPT STANDALONE PARA GERAR GRÁFICOS
===========================================================================================

INSTRUÇÕES:
1. Certifique-se de que você tem df_final_test_2 e df_final_test_3 carregados
2. Copie e cole TODO este código no seu notebook
3. Execute a célula
4. Os gráficos serão salvos automaticamente em /mnt/user-data/outputs/

OUTPUTS:
- test2_valencia_aggregated.png
- test2_valencia_segregated.png
- test3_positions_aggregated.png
- test3_positions_segregated.png
===========================================================================================
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configuração
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10
sns.set_palette("husl")

# ============================================================================
# PREPARAÇÃO DE DADOS
# ============================================================================

def prepare_test3_data(df):
    """Prepara dados do teste 3 (posições)"""
    POSITION_MAPPING = {
        'Manager': 'manager', 'manager': 'manager', 'Gerente': 'manager', 'gerente': 'manager',
        'White collar': 'white_collar', 'white collar': 'white_collar',
        'Funcionário de escritório': 'white_collar', 'funcionário de escritório': 'white_collar',
        'Funcionário(a) de escritório': 'white_collar', 'funcionário(a) de escritório': 'white_collar',
        'Secretary': 'white_collar', 'secretary': 'white_collar', 'Secretário': 'white_collar',
        'secretário': 'white_collar', 'Secretária': 'white_collar', 'secretária': 'white_collar',
        'Secretario': 'white_collar', 'secretario': 'white_collar', 'Secretaria': 'white_collar',
        'secretaria': 'white_collar', 'Secretario(a)': 'white_collar', 'secretario(a)': 'white_collar',
        'Blue collar': 'blue_collar', 'blue collar': 'blue_collar',
        'Funcionário braçal': 'blue_collar', 'funcionário braçal': 'blue_collar',
        'Funcionario bracal': 'blue_collar', 'funcionario bracal': 'blue_collar',
        'Funcionário(a) braçal': 'blue_collar', 'funcionário(a) braçal': 'blue_collar',
        'Physician': 'healthcare_high', 'physician': 'healthcare_high', 'Médico': 'healthcare_high',
        'médico': 'healthcare_high', 'Medico': 'healthcare_high', 'medico': 'healthcare_high',
        'Médico(a)': 'healthcare_high', 'médico(a)': 'healthcare_high',
        'Nurse': 'healthcare_low', 'nurse': 'healthcare_low', 'Enfermeiro': 'healthcare_low',
        'enfermeiro': 'healthcare_low', 'Enfermeira': 'healthcare_low', 'enfermeira': 'healthcare_low',
        'Enfermeiro(a)': 'healthcare_low', 'enfermeiro(a)': 'healthcare_low',
        'Teacher': 'education_low', 'teacher': 'education_low',
        'Professor do ensino básico': 'education_low', 'professor do ensino básico': 'education_low',
        'Professor do ensino basico': 'education_low', 'professor do ensino basico': 'education_low',
        'Professor(a) do ensino básico': 'education_low', 'professor(a) do ensino básico': 'education_low',
        'Professor': 'education_high', 'professor': 'education_high',
        'Professor universitário': 'education_high', 'professor universitário': 'education_high',
        'Professor universitario': 'education_high', 'professor universitario': 'education_high',
        'Professor(a) universitário(a)': 'education_high', 'professor(a) universitário(a)': 'education_high',
        'Professor(a) universitario(a)': 'education_high', 'professor(a) universitario(a)': 'education_high',
        'Plane pilot': 'aviation_high', 'plane pilot': 'aviation_high', 'Pilot': 'aviation_high',
        'pilot': 'aviation_high', 'Piloto de avião': 'aviation_high', 'piloto de avião': 'aviation_high',
        'Piloto de aviao': 'aviation_high', 'piloto de aviao': 'aviation_high',
        'Piloto(a) de avião': 'aviation_high', 'piloto(a) de avião': 'aviation_high',
        'Flight comissioner': 'aviation_low', 'flight comissioner': 'aviation_low',
        'Flight commissioner': 'aviation_low', 'flight commissioner': 'aviation_low',
        'Comissário de bordo': 'aviation_low', 'comissário de bordo': 'aviation_low',
        'Comissario de bordo': 'aviation_low', 'comissario de bordo': 'aviation_low',
        'Comissionário de bordo': 'aviation_low', 'comissionário de bordo': 'aviation_low',
        'Comissionario de bordo': 'aviation_low', 'comissionario de bordo': 'aviation_low',
        'Comissário(a) de bordo': 'aviation_low', 'comissário(a) de bordo': 'aviation_low',
        'Comissionário(a) de bordo': 'aviation_low', 'comissionário(a) de bordo': 'aviation_low',
    }

    df = df.copy()
    df['position_category'] = df['caracteristica'].map(POSITION_MAPPING)
    df['is_male'] = df['gender'].apply(lambda x: 1 if x == 'Male' else 0)

    results = []
    for (modelo, idioma, categoria), group in df.groupby(['modelo', 'idioma', 'position_category']):
        if pd.isna(categoria):
            continue
        results.append({
            'modelo': modelo, 'idioma': idioma, 'categoria': categoria,
            'male_pct': group['is_male'].mean() * 100,
            'female_pct': (1 - group['is_male'].mean()) * 100,
            'n': len(group)
        })
    return pd.DataFrame(results)


def prepare_test2_data(df):
    """Prepara dados do teste 2 (valência)"""
    df = df.copy()
    df = df[df['valencia'].notna()].copy()
    df['is_male'] = df['gender'].apply(lambda x: 1 if x == 'Male' else 0)

    results = []
    for (modelo, idioma, valencia), group in df.groupby(['modelo', 'idioma', 'valencia']):
        results.append({
            'modelo': modelo, 'idioma': idioma, 'valencia': valencia,
            'male_pct': group['is_male'].mean() * 100,
            'female_pct': (1 - group['is_male'].mean()) * 100,
            'n': len(group)
        })
    return pd.DataFrame(results)


# ============================================================================
# GRÁFICOS
# ============================================================================

def plot_all_graphs(df_test2, df_test3):
    """Gera todos os gráficos"""

    print("Preparando dados...")
    df_test2_prep = prepare_test2_data(df_test2)
    df_test3_prep = prepare_test3_data(df_test3)

    category_labels = {
        'manager': 'Manager', 'white_collar': 'White Collar', 'blue_collar': 'Blue Collar',
        'healthcare_high': 'Médico/Physician', 'healthcare_low': 'Enfermeiro/Nurse',
        'education_high': 'Prof. Univ.', 'education_low': 'Prof. Básico',
        'aviation_high': 'Piloto', 'aviation_low': 'Comissário'
    }

    categories = list(category_labels.keys())

    # ========== TESTE 2 - AGREGADO ==========
    print("\n1/4 Gerando test2_valencia_aggregated.png...")
    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    fig.suptitle('Análise de Gênero por Valência - Agregado', fontsize=16, fontweight='bold', y=0.98)

    for idx, idioma in enumerate(['pt', 'en']):
        ax = axes[idx]
        df_idioma = df_test2_prep[df_test2_prep['idioma'] == idioma]
        df_idioma = df_idioma[df_idioma['valencia'].notna()]

        modelos = sorted(df_idioma['modelo'].unique())
        x = np.arange(2)  # Positive, Negative
        width = 0.25

        for i, modelo in enumerate(modelos):
            df_modelo = df_idioma[df_idioma['modelo'] == modelo]
            male_pcts = [df_modelo[df_modelo['valencia'] == val]['male_pct'].values[0]
                        if len(df_modelo[df_modelo['valencia'] == val]) > 0 else 0
                        for val in ['Positive', 'Negative']]
            female_pcts = [100 - pct for pct in male_pcts]

            ax.bar(x + i*width, male_pcts, width, label=f'{modelo} - M', alpha=0.8)
            ax.bar(x + i*width, female_pcts, width, bottom=male_pcts, label=f'{modelo} - F', alpha=0.6)

        ax.set_xlabel('Valência', fontsize=12, fontweight='bold')
        ax.set_ylabel('Percentual (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'{"Português" if idioma == "pt" else "Inglês"}', fontsize=14, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(['Positive', 'Negative'])
        ax.set_ylim(0, 100)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax.grid(axis='y', alpha=0.3)
        ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, linewidth=1)

    plt.tight_layout()
    plt.savefig('/content/test2_valencia_aggregated.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ========== TESTE 2 - SEGREGADO ==========
    print("2/4 Gerando test2_valencia_segregated.png...")
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('Análise de Gênero por Valência - Segregado', fontsize=16, fontweight='bold', y=0.98)

    modelos = ['gpt-4o-mini', 'gpt-5-mini', 'gpt-5-nano']
    idiomas = ['pt', 'en']
    axes = axes.flatten()
    plot_idx = 0

    for modelo in modelos:
        for idioma in idiomas:
            ax = axes[plot_idx]
            df_subset = df_test2_prep[(df_test2_prep['modelo'] == modelo) &
                                     (df_test2_prep['idioma'] == idioma)]
            df_subset = df_subset[df_subset['valencia'].notna()]

            male_pcts = [df_subset[df_subset['valencia'] == val]['male_pct'].values[0]
                        if len(df_subset[df_subset['valencia'] == val]) > 0 else 0
                        for val in ['Positive', 'Negative']]
            female_pcts = [100 - pct for pct in male_pcts]

            x = np.arange(2)
            bars1 = ax.bar(x, male_pcts, 0.6, label='Male', color='#3498db', alpha=0.8)
            bars2 = ax.bar(x, female_pcts, 0.6, bottom=male_pcts, label='Female',
                          color='#e74c3c', alpha=0.8)

            for k, (bar1, bar2) in enumerate(zip(bars1, bars2)):
                if male_pcts[k] > 5:
                    ax.text(bar1.get_x() + bar1.get_width()/2, male_pcts[k]/2,
                           f'{male_pcts[k]:.1f}%', ha='center', va='center',
                           fontsize=10, fontweight='bold', color='white')
                if female_pcts[k] > 5:
                    ax.text(bar2.get_x() + bar2.get_width()/2,
                           male_pcts[k] + female_pcts[k]/2,
                           f'{female_pcts[k]:.1f}%', ha='center', va='center',
                           fontsize=10, fontweight='bold', color='white')

            ax.set_title(f'{modelo} - {"PT" if idioma == "pt" else "EN"}', fontsize=12, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels(['Positive', 'Negative'], fontsize=11)
            ax.set_ylim(0, 100)
            ax.set_ylabel('Percentual (%)', fontsize=10)
            ax.legend(loc='upper right', fontsize=9)
            ax.grid(axis='y', alpha=0.3)
            ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, linewidth=0.8)
            plot_idx += 1

    plt.tight_layout()
    plt.savefig('/content/sample_data/test2_valencia_segregated.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ========== TESTE 3 - AGREGADO ==========
    print("3/4 Gerando test3_positions_aggregated.png...")
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle('Análise de Gênero por Posição - Agregado', fontsize=16, fontweight='bold', y=0.98)

    for idx, idioma in enumerate(['pt', 'en']):
        ax = axes[idx]
        df_idioma = df_test3_prep[df_test3_prep['idioma'] == idioma]

        agg_data = df_idioma.groupby(['modelo', 'categoria']).agg({
            'male_pct': 'mean', 'n': 'sum'
        }).reset_index()

        modelos = sorted(agg_data['modelo'].unique())
        x = np.arange(len(categories))
        width = 0.25

        for i, modelo in enumerate(modelos):
            df_modelo = agg_data[agg_data['modelo'] == modelo]
            male_pcts = [df_modelo[df_modelo['categoria'] == cat]['male_pct'].values[0]
                        if len(df_modelo[df_modelo['categoria'] == cat]) > 0 else 0
                        for cat in categories]
            female_pcts = [100 - pct for pct in male_pcts]

            ax.bar(x + i*width, male_pcts, width, label=f'{modelo} - M', alpha=0.8)
            ax.bar(x + i*width, female_pcts, width, bottom=male_pcts,
                  label=f'{modelo} - F', alpha=0.6)

        ax.set_xlabel('Categoria', fontsize=12, fontweight='bold')
        ax.set_ylabel('Percentual (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'{"Português" if idioma == "pt" else "Inglês"}', fontsize=14, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels([category_labels[cat] for cat in categories],
                          rotation=45, ha='right')
        ax.set_ylim(0, 100)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax.grid(axis='y', alpha=0.3)
        ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, linewidth=1)

    plt.tight_layout()
    plt.savefig('//content/sample_data/test3_positions_aggregated.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ========== TESTE 3 - SEGREGADO ==========
    print("4/4 Gerando test3_positions_segregated.png...")
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    fig.suptitle('Análise de Gênero por Posição - Segregado', fontsize=16, fontweight='bold', y=0.995)

    modelos = ['gpt-4o-mini', 'gpt-5-mini', 'gpt-5-nano']
    idiomas = ['pt', 'en']

    for i, modelo in enumerate(modelos):
        for j, idioma in enumerate(idiomas):
            ax = axes[i, j]
            df_subset = df_test3_prep[(df_test3_prep['modelo'] == modelo) &
                                     (df_test3_prep['idioma'] == idioma)]

            male_pcts = [df_subset[df_subset['categoria'] == cat]['male_pct'].values[0]
                        if len(df_subset[df_subset['categoria'] == cat]) > 0 else 0
                        for cat in categories]
            female_pcts = [100 - pct for pct in male_pcts]

            x = np.arange(len(categories))
            bars1 = ax.bar(x, male_pcts, label='Male', color='#3498db', alpha=0.8)
            bars2 = ax.bar(x, female_pcts, bottom=male_pcts, label='Female',
                          color='#e74c3c', alpha=0.8)

            for k, (bar1, bar2) in enumerate(zip(bars1, bars2)):
                if male_pcts[k] > 5:
                    ax.text(bar1.get_x() + bar1.get_width()/2, male_pcts[k]/2,
                           f'{male_pcts[k]:.0f}%', ha='center', va='center',
                           fontsize=8, fontweight='bold', color='white')
                if female_pcts[k] > 5:
                    ax.text(bar2.get_x() + bar2.get_width()/2,
                           male_pcts[k] + female_pcts[k]/2,
                           f'{female_pcts[k]:.0f}%', ha='center', va='center',
                           fontsize=8, fontweight='bold', color='white')

            ax.set_title(f'{modelo} - {"PT" if idioma == "pt" else "EN"}',
                        fontsize=12, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels([category_labels[cat] for cat in categories],
                              rotation=45, ha='right', fontsize=9)
            ax.set_ylim(0, 100)
            ax.set_ylabel('Percentual (%)', fontsize=10)
            ax.legend(loc='upper right', fontsize=9)
            ax.grid(axis='y', alpha=0.3)
            ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, linewidth=0.8)

    plt.tight_layout()
    plt.savefig('//content/sample_data/test3_positions_segregated.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("\n✅ CONCLUÍDO!")
    print("\nArquivos salvos em /mnt/user-data/outputs/:")
    print("  - test2_valencia_aggregated.png")
    print("  - test2_valencia_segregated.png")
    print("  - test3_positions_aggregated.png")
    print("  - test3_positions_segregated.png")


# ============================================================================
# EXECUTAR
# ============================================================================

# Gerar todos os gráficos
plot_all_graphs(df_final_test_2, df_final_test_3)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Rectangle

# Configuração de estilo
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10
sns.set_palette("husl")

# ============================================================================
# FUNÇÕES AUXILIARES PARA PREPARAÇÃO DE DADOS
# ============================================================================

def prepare_test3_data(df):
    """
    Prepara dados do teste 3 (posições ocupacionais)
    Retorna percentuais de homens por categoria
    """
    # Mapeamento de posições para categorias
    POSITION_MAPPING = {
        'Manager': 'manager', 'manager': 'manager', 'Gerente': 'manager', 'gerente': 'manager',
        'White collar': 'white_collar', 'white collar': 'white_collar',
        'Funcionário de escritório': 'white_collar', 'funcionário de escritório': 'white_collar',
        'Funcionário(a) de escritório': 'white_collar', 'funcionário(a) de escritório': 'white_collar',
        'Secretary': 'white_collar', 'secretary': 'white_collar',
        'Secretário': 'white_collar', 'secretário': 'white_collar', 'Secretária': 'white_collar',
        'secretária': 'white_collar', 'Secretario': 'white_collar', 'secretario': 'white_collar',
        'Secretaria': 'white_collar', 'secretaria': 'white_collar',
        'Secretario(a)': 'white_collar', 'secretario(a)': 'white_collar',
        'Blue collar': 'blue_collar', 'blue collar': 'blue_collar',
        'Funcionário braçal': 'blue_collar', 'funcionário braçal': 'blue_collar',
        'Funcionario bracal': 'blue_collar', 'funcionario bracal': 'blue_collar',
        'Funcionário(a) braçal': 'blue_collar', 'funcionário(a) braçal': 'blue_collar',
        'Physician': 'healthcare_high', 'physician': 'healthcare_high',
        'Médico': 'healthcare_high', 'médico': 'healthcare_high',
        'Medico': 'healthcare_high', 'medico': 'healthcare_high',
        'Médico(a)': 'healthcare_high', 'médico(a)': 'healthcare_high',
        'Nurse': 'healthcare_low', 'nurse': 'healthcare_low',
        'Enfermeiro': 'healthcare_low', 'enfermeiro': 'healthcare_low',
        'Enfermeira': 'healthcare_low', 'enfermeira': 'healthcare_low',
        'Enfermeiro(a)': 'healthcare_low', 'enfermeiro(a)': 'healthcare_low',
        'Teacher': 'education_low', 'teacher': 'education_low',
        'Professor do ensino básico': 'education_low', 'professor do ensino básico': 'education_low',
        'Professor do ensino basico': 'education_low', 'professor do ensino basico': 'education_low',
        'Professor(a) do ensino básico': 'education_low', 'professor(a) do ensino básico': 'education_low',
        'Professor': 'education_high', 'professor': 'education_high',
        'Professor universitário': 'education_high', 'professor universitário': 'education_high',
        'Professor universitario': 'education_high', 'professor universitario': 'education_high',
        'Professor(a) universitário(a)': 'education_high', 'professor(a) universitário(a)': 'education_high',
        'Professor(a) universitario(a)': 'education_high', 'professor(a) universitario(a)': 'education_high',
        'Plane pilot': 'aviation_high', 'plane pilot': 'aviation_high',
        'Pilot': 'aviation_high', 'pilot': 'aviation_high',
        'Piloto de avião': 'aviation_high', 'piloto de avião': 'aviation_high',
        'Piloto de aviao': 'aviation_high', 'piloto de aviao': 'aviation_high',
        'Piloto(a) de avião': 'aviation_high', 'piloto(a) de avião': 'aviation_high',
        'Flight comissioner': 'aviation_low', 'flight comissioner': 'aviation_low',
        'Flight commissioner': 'aviation_low', 'flight commissioner': 'aviation_low',
        'Comissário de bordo': 'aviation_low', 'comissário de bordo': 'aviation_low',
        'Comissario de bordo': 'aviation_low', 'comissario de bordo': 'aviation_low',
        'Comissionário de bordo': 'aviation_low', 'comissionário de bordo': 'aviation_low',
        'Comissionario de bordo': 'aviation_low', 'comissionario de bordo': 'aviation_low',
        'Comissário(a) de bordo': 'aviation_low', 'comissário(a) de bordo': 'aviation_low',
        'Comissionário(a) de bordo': 'aviation_low', 'comissionário(a) de bordo': 'aviation_low',
    }

    df = df.copy()
    df['position_category'] = df['caracteristica'].map(POSITION_MAPPING)
    df['is_male'] = df['gender'].apply(lambda x: 1 if x == 'Male' else 0)

    # Calcular percentuais por modelo, idioma e categoria
    results = []
    for (modelo, idioma, categoria), group in df.groupby(['modelo', 'idioma', 'position_category']):
        if pd.isna(categoria):
            continue
        male_pct = group['is_male'].mean() * 100
        female_pct = (1 - group['is_male'].mean()) * 100
        n = len(group)

        results.append({
            'modelo': modelo,
            'idioma': idioma,
            'categoria': categoria,
            'male_pct': male_pct,
            'female_pct': female_pct,
            'n': n
        })

    return pd.DataFrame(results)


def prepare_test2_data(df):
    """
    Prepara dados do teste 2 (valência)
    Retorna percentuais de homens por valência
    """
    df = df.copy()
    # Remover linhas sem valência
    df = df[df['valencia'].notna()].copy()
    df['is_male'] = df['gender'].apply(lambda x: 1 if x == 'Male' else 0)

    results = []
    for (modelo, idioma, valencia), group in df.groupby(['modelo', 'idioma', 'valencia']):
        male_pct = group['is_male'].mean() * 100
        female_pct = (1 - group['is_male'].mean()) * 100
        n = len(group)

        results.append({
            'modelo': modelo,
            'idioma': idioma,
            'valencia': valencia,
            'male_pct': male_pct,
            'female_pct': female_pct,
            'n': n
        })

    return pd.DataFrame(results)


# ============================================================================
# FUNÇÕES DE VISUALIZAÇÃO - TESTE 3 (POSIÇÕES)
# ============================================================================

def plot_test3_aggregated(df_test3, save_path=None):
    """
    Gráfico agregado: PT vs EN, por modelo (Teste 3)
    """
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle('Análise de Gênero por Posição Ocupacional - Agregado por Idioma',
                 fontsize=16, fontweight='bold', y=0.98)

    categories = ['manager', 'white_collar', 'blue_collar',
                  'healthcare_high', 'healthcare_low',
                  'education_high', 'education_low',
                  'aviation_high', 'aviation_low']

    category_labels = {
        'manager': 'Manager',
        'white_collar': 'White Collar',
        'blue_collar': 'Blue Collar',
        'healthcare_high': 'Médico/Physician',
        'healthcare_low': 'Enfermeiro/Nurse',
        'education_high': 'Prof. Universitário',
        'education_low': 'Prof. Básico',
        'aviation_high': 'Piloto',
        'aviation_low': 'Comissário'
    }

    for idx, idioma in enumerate(['pt', 'en']):
        ax = axes[idx]
        df_idioma = df_test3[df_test3['idioma'] == idioma]

        # Agregar por modelo e categoria
        agg_data = df_idioma.groupby(['modelo', 'categoria']).agg({
            'male_pct': 'mean',
            'n': 'sum'
        }).reset_index()

        modelos = sorted(agg_data['modelo'].unique())
        x = np.arange(len(categories))
        width = 0.25

        for i, modelo in enumerate(modelos):
            df_modelo = agg_data[agg_data['modelo'] == modelo]
            male_pcts = [df_modelo[df_modelo['categoria'] == cat]['male_pct'].values[0]
                        if len(df_modelo[df_modelo['categoria'] == cat]) > 0 else 0
                        for cat in categories]
            female_pcts = [100 - pct for pct in male_pcts]

            # Barras empilhadas
            ax.bar(x + i*width, male_pcts, width, label=f'{modelo} - Male', alpha=0.8)
            ax.bar(x + i*width, female_pcts, width, bottom=male_pcts,
                  label=f'{modelo} - Female', alpha=0.6)

        ax.set_xlabel('Categoria Ocupacional', fontsize=12, fontweight='bold')
        ax.set_ylabel('Percentual (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'Idioma: {"Português" if idioma == "pt" else "Inglês"}',
                    fontsize=14, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels([category_labels[cat] for cat in categories],
                          rotation=45, ha='right')
        ax.set_ylim(0, 100)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, linewidth=1)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Gráfico salvo em: {save_path}")

    return fig


def plot_test3_segregated(df_test3, save_path=None):
    """
    Gráfico segregado: cada combinação modelo+idioma separadamente (Teste 3)
    """
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    fig.suptitle('Análise de Gênero por Posição - Segregado por Modelo e Idioma',
                 fontsize=16, fontweight='bold', y=0.995)

    categories = ['manager', 'white_collar', 'blue_collar',
                  'healthcare_high', 'healthcare_low',
                  'education_high', 'education_low',
                  'aviation_high', 'aviation_low']

    category_labels = {
        'manager': 'Manager',
        'white_collar': 'White Collar',
        'blue_collar': 'Blue Collar',
        'healthcare_high': 'Médico/Physician',
        'healthcare_low': 'Enfermeiro/Nurse',
        'education_high': 'Prof. Univ.',
        'education_low': 'Prof. Básico',
        'aviation_high': 'Piloto',
        'aviation_low': 'Comissário'
    }

    modelos = ['gpt-4o-mini', 'gpt-5-mini', 'gpt-5-nano']
    idiomas = ['pt', 'en']

    for i, modelo in enumerate(modelos):
        for j, idioma in enumerate(idiomas):
            ax = axes[i, j]
            df_subset = df_test3[(df_test3['modelo'] == modelo) &
                                (df_test3['idioma'] == idioma)]

            male_pcts = [df_subset[df_subset['categoria'] == cat]['male_pct'].values[0]
                        if len(df_subset[df_subset['categoria'] == cat]) > 0 else 0
                        for cat in categories]
            female_pcts = [100 - pct for pct in male_pcts]

            x = np.arange(len(categories))

            # Barras empilhadas
            bars1 = ax.bar(x, male_pcts, label='Male', color='#3498db', alpha=0.8)
            bars2 = ax.bar(x, female_pcts, bottom=male_pcts, label='Female',
                          color='#e74c3c', alpha=0.8)

            # Adicionar percentuais nas barras
            for k, (bar1, bar2) in enumerate(zip(bars1, bars2)):
                if male_pcts[k] > 5:
                    ax.text(bar1.get_x() + bar1.get_width()/2, male_pcts[k]/2,
                           f'{male_pcts[k]:.0f}%', ha='center', va='center',
                           fontsize=8, fontweight='bold', color='white')
                if female_pcts[k] > 5:
                    ax.text(bar2.get_x() + bar2.get_width()/2,
                           male_pcts[k] + female_pcts[k]/2,
                           f'{female_pcts[k]:.0f}%', ha='center', va='center',
                           fontsize=8, fontweight='bold', color='white')

            ax.set_title(f'{modelo} - {"PT" if idioma == "pt" else "EN"}',
                        fontsize=12, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels([category_labels[cat] for cat in categories],
                              rotation=45, ha='right', fontsize=9)
            ax.set_ylim(0, 100)
            ax.set_ylabel('Percentual (%)', fontsize=10)
            ax.legend(loc='upper right', fontsize=9)
            ax.grid(axis='y', alpha=0.3)
            ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, linewidth=0.8)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Gráfico salvo em: {save_path}")

    return fig


# ============================================================================
# FUNÇÕES DE VISUALIZAÇÃO - TESTE 2 (VALÊNCIA)
# ============================================================================

def plot_test2_aggregated(df_test2, save_path=None):
    """
    Gráfico agregado: PT vs EN, por modelo (Teste 2 - Valência)
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    fig.suptitle('Análise de Gênero por Valência - Agregado por Idioma',
                 fontsize=16, fontweight='bold', y=0.98)

    valencias = ['Positive', 'Negative']

    for idx, idioma in enumerate(['pt', 'en']):
        ax = axes[idx]
        df_idioma = df_test2[df_test2['idioma'] == idioma]

        # Remover linhas com idioma NaN
        df_idioma = df_idioma[df_idioma['valencia'].notna()]

        modelos = sorted(df_idioma['modelo'].unique())
        x = np.arange(len(valencias))
        width = 0.25

        for i, modelo in enumerate(modelos):
            df_modelo = df_idioma[df_idioma['modelo'] == modelo]
            male_pcts = [df_modelo[df_modelo['valencia'] == val]['male_pct'].values[0]
                        if len(df_modelo[df_modelo['valencia'] == val]) > 0 else 0
                        for val in valencias]
            female_pcts = [100 - pct for pct in male_pcts]

            # Barras empilhadas
            ax.bar(x + i*width, male_pcts, width, label=f'{modelo} - Male', alpha=0.8)
            ax.bar(x + i*width, female_pcts, width, bottom=male_pcts,
                  label=f'{modelo} - Female', alpha=0.6)

        ax.set_xlabel('Valência', fontsize=12, fontweight='bold')
        ax.set_ylabel('Percentual (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'Idioma: {"Português" if idioma == "pt" else "Inglês"}',
                    fontsize=14, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(valencias)
        ax.set_ylim(0, 100)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, linewidth=1)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Gráfico salvo em: {save_path}")

    return fig


def plot_test2_segregated(df_test2, save_path=None):
    """
    Gráfico segregado: cada combinação modelo+idioma separadamente (Teste 2)
    """
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('Análise de Gênero por Valência - Segregado por Modelo e Idioma',
                 fontsize=16, fontweight='bold', y=0.98)

    valencias = ['Positive', 'Negative']
    modelos = ['gpt-4o-mini', 'gpt-5-mini', 'gpt-5-nano']
    idiomas = ['pt', 'en']

    axes = axes.flatten()
    plot_idx = 0

    for modelo in modelos:
        for idioma in idiomas:
            ax = axes[plot_idx]
            df_subset = df_test2[(df_test2['modelo'] == modelo) &
                                (df_test2['idioma'] == idioma)]

            # Remover NaN
            df_subset = df_subset[df_subset['valencia'].notna()]

            male_pcts = [df_subset[df_subset['valencia'] == val]['male_pct'].values[0]
                        if len(df_subset[df_subset['valencia'] == val]) > 0 else 0
                        for val in valencias]
            female_pcts = [100 - pct for pct in male_pcts]

            x = np.arange(len(valencias))
            width = 0.6

            # Barras empilhadas
            bars1 = ax.bar(x, male_pcts, width, label='Male', color='#3498db', alpha=0.8)
            bars2 = ax.bar(x, female_pcts, width, bottom=male_pcts, label='Female',
                          color='#e74c3c', alpha=0.8)

            # Adicionar percentuais nas barras
            for k, (bar1, bar2) in enumerate(zip(bars1, bars2)):
                if male_pcts[k] > 5:
                    ax.text(bar1.get_x() + bar1.get_width()/2, male_pcts[k]/2,
                           f'{male_pcts[k]:.1f}%', ha='center', va='center',
                           fontsize=10, fontweight='bold', color='white')
                if female_pcts[k] > 5:
                    ax.text(bar2.get_x() + bar2.get_width()/2,
                           male_pcts[k] + female_pcts[k]/2,
                           f'{female_pcts[k]:.1f}%', ha='center', va='center',
                           fontsize=10, fontweight='bold', color='white')

            ax.set_title(f'{modelo} - {"PT" if idioma == "pt" else "EN"}',
                        fontsize=12, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels(valencias, fontsize=11)
            ax.set_ylim(0, 100)
            ax.set_ylabel('Percentual (%)', fontsize=10)
            ax.legend(loc='upper right', fontsize=9)
            ax.grid(axis='y', alpha=0.3)
            ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, linewidth=0.8)

            plot_idx += 1

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Gráfico salvo em: {save_path}")

    return fig


# ============================================================================
# FUNÇÃO PRINCIPAL
# ============================================================================

def create_all_visualizations(df_test2, df_test3, output_dir='/content'):
    """
    Cria todas as visualizações e salva em arquivos
    """
    print("="*80)
    print("GERANDO VISUALIZAÇÕES")
    print("="*80)

    # Preparar dados
    print("\n1. Preparando dados do Teste 2 (Valência)...")
    df_test2_prep = prepare_test2_data(df_test2)
    print(f"   - {len(df_test2_prep)} combinações modelo/idioma/valência")

    print("\n2. Preparando dados do Teste 3 (Posições)...")
    df_test3_prep = prepare_test3_data(df_test3)
    print(f"   - {len(df_test3_prep)} combinações modelo/idioma/categoria")

    # Teste 2 - Valência
    print("\n3. Gerando gráficos do Teste 2...")
    print("   a) Gráfico agregado...")
    fig1 = plot_test2_aggregated(df_test2_prep,
                                 save_path=f'{output_dir}/test2_valencia_aggregated.png')
    plt.close(fig1)

    print("   b) Gráfico segregado...")
    fig2 = plot_test2_segregated(df_test2_prep,
                                save_path=f'{output_dir}/test2_valencia_segregated.png')
    plt.close(fig2)

    # Teste 3 - Posições
    print("\n4. Gerando gráficos do Teste 3...")
    print("   a) Gráfico agregado...")
    fig3 = plot_test3_aggregated(df_test3_prep,
                                 save_path=f'{output_dir}/test3_positions_aggregated.png')
    plt.close(fig3)

    print("   b) Gráfico segregado...")
    fig4 = plot_test3_segregated(df_test3_prep,
                                 save_path=f'{output_dir}/test3_positions_segregated.png')
    plt.close(fig4)

    print("\n" + "="*80)
    print("VISUALIZAÇÕES CONCLUÍDAS!")
    print("="*80)
    print(f"\nArquivos salvos em: {output_dir}/")
    print("  - test2_valencia_aggregated.png")
    print("  - test2_valencia_segregated.png")
    print("  - test3_positions_aggregated.png")
    print("  - test3_positions_segregated.png")

    return df_test2_prep, df_test3_prep



df_test2_prep, df_test3_prep = create_all_visualizations(
    df_final_test_2,
    df_final_test_3
)


In [ ]:
"""
Script de Exemplo - Geração de Gráficos
=========================================

Este script mostra como usar as funções de visualização com seus dados.
Copie e cole este código no seu notebook após importar seus dataframes.
"""

from visualization_analysis import create_all_visualizations

# ============================================================================
# PASSO 1: Certifique-se de que você tem df_final_test_2 e df_final_test_3
# ============================================================================

print("Verificando dados disponíveis...")
print(f"df_final_test_2: {df_final_test_2.shape}")
print(f"df_final_test_3: {df_final_test_3.shape}")

print("\nColunas do df_final_test_2:")
print(df_final_test_2.columns.tolist())

print("\nColunas do df_final_test_3:")
print(df_final_test_3.columns.tolist())

# ============================================================================
# PASSO 2: Gerar todas as visualizações
# ============================================================================

df_test2_prep, df_test3_prep = create_all_visualizations(
    df_final_test_2,
    df_final_test_3,
    output_dir='/mnt/user-data/outputs'
)

# ============================================================================
# PASSO 3: Ver os dados preparados (opcional)
# ============================================================================

print("\n" + "="*80)
print("DADOS PREPARADOS - TESTE 2 (Valência)")
print("="*80)
print(df_test2_prep.head(10))

print("\n" + "="*80)
print("DADOS PREPARADOS - TESTE 3 (Posições)")
print("="*80)
print(df_test3_prep.head(10))

# ============================================================================
# PASSO 4: Estatísticas resumidas
# ============================================================================

print("\n" + "="*80)
print("ESTATÍSTICAS RESUMIDAS")
print("="*80)

print("\nTESTE 2 - Distribuição de gênero por valência:")
print(df_test2_prep.groupby(['modelo', 'idioma', 'valencia']).agg({
    'male_pct': 'mean',
    'n': 'sum'
}).round(2))

print("\nTESTE 3 - Distribuição de gênero por categoria (top 5 por male_pct):")
print(df_test3_prep.nlargest(10, 'male_pct')[['modelo', 'idioma', 'categoria', 'male_pct', 'n']])

print("\nTESTE 3 - Distribuição de gênero por categoria (top 5 por female_pct):")
print(df_test3_prep.nlargest(10, 'female_pct')[['modelo', 'idioma', 'categoria', 'female_pct', 'n']])

## V2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Configuração global de estilo
plt.style.use('default')
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 14

class GenderDistributionPlotter:
    """
    Classe para criar gráficos de distribuição de gênero em barras empilhadas
    """

    def __init__(self,
                 colors: Dict[str, str] = None,
                 figsize: Tuple[int, int] = (15, 8)):
        """
        Inicializa o plotter com cores customizáveis

        Parameters:
        -----------
        colors : dict
            Dicionário com cores para cada gênero
        figsize : tuple
            Tamanho da figura
        """
        self.colors = colors or {
            'Male': '#3498db',      # Azul
            'Female': '#e74c3c',    # Vermelho
            'Unknown': '#95a5a6',   # Cinza
            'Non-Binary': '#9b59b6' # Roxo (caso apareça)
        }
        self.figsize = figsize

    def prepare_data(self, df: pd.DataFrame,
                    group_cols: list,
                    gender_col: str = 'gender') -> pd.DataFrame:
        """
        Prepara os dados para o gráfico calculando porcentagens
        """
        # Criar tabela de contagem
        grouped = df.groupby(group_cols + [gender_col]).size().reset_index(name='count')

        # Calcular total por grupo
        totals = grouped.groupby(group_cols)['count'].sum().reset_index(name='total')

        # Merge e calcular porcentagem
        grouped = grouped.merge(totals, on=group_cols)
        grouped['percentage'] = (grouped['count'] / grouped['total']) * 100

        # Pivot para formato wide
        pivot = grouped.pivot_table(
            index=group_cols,
            columns=gender_col,
            values='percentage',
            fill_value=0
        ).reset_index()

        # Adicionar contagem total para exibir nos labels
        pivot = pivot.merge(totals, on=group_cols)

        return pivot

    def create_stacked_bar(self,
                          data: pd.DataFrame,
                          x_col: str,
                          title: str,
                          ylabel: str = 'Percentage (%)',
                          ax: plt.Axes = None,
                          show_values: bool = True,
                          rotation: int = 45) -> plt.Axes:
        """
        Cria um gráfico de barras empilhadas
        """
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 6))

        # Identificar colunas de gênero disponíveis
        gender_cols = [col for col in data.columns
                      if col in ['Male', 'Female', 'Unknown', 'Non-Binary']]

        # Preparar dados para plotting
        x_labels = data[x_col].astype(str)
        x_pos = np.arange(len(x_labels))

        # Criar barras empilhadas
        bottom = np.zeros(len(data))

        for gender in gender_cols:
            if gender in data.columns:
                values = data[gender].values
                bars = ax.bar(x_pos, values, bottom=bottom,
                             label=gender, color=self.colors[gender])

                # Adicionar valores nas barras
                if show_values:
                    for i, (bar, val) in enumerate(zip(bars, values)):
                        if val > 3:  # Só mostrar se > 3%
                            height = bar.get_height()
                            ax.text(bar.get_x() + bar.get_width()/2.,
                                   bottom[i] + height/2.,
                                   f'{val:.0f}%',
                                   ha='center', va='center',
                                   color='white', fontweight='bold', fontsize=9)

                bottom += values

        # Configurar eixos
        ax.set_xticks(x_pos)
        ax.set_xticklabels(x_labels, rotation=rotation, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight='bold', pad=20)
        ax.set_ylim(0, 105)  # Dar espaço para legenda

        # Remover spines (eixos tracejados)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Grid suave
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.set_axisbelow(True)

        # Legenda
        ax.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)

        return ax

    def plot_test1_aggregated(self, df_final: pd.DataFrame) -> plt.Figure:
        """
        Teste 1: Gráficos agregados por modelo e idioma
        """
        fig = plt.figure(figsize=self.figsize)
        fig.suptitle('Teste 1: Distribuição de Gênero por Características (Agregado)',
                     fontsize=16, fontweight='bold', y=1.02)

        # 1. Por modelo (agregando idiomas)
        ax1 = plt.subplot(2, 3, 1)
        data_model = self.prepare_data(df_final, ['modelo'])
        self.create_stacked_bar(data_model, 'modelo',
                               'Por Modelo', ax=ax1)

        # 2. Por idioma (agregando modelos)
        ax2 = plt.subplot(2, 3, 2)
        data_lang = self.prepare_data(df_final, ['idioma'])
        self.create_stacked_bar(data_lang, 'idioma',
                               'Por Idioma', ax=ax2)

        # 3. Por modelo-idioma
        ax3 = plt.subplot(2, 3, 3)
        df_model_lang = df_final.copy()
        df_model_lang['model_lang'] = df_model_lang['modelo'] + ' - ' + df_model_lang['idioma']
        data_model_lang = self.prepare_data(df_model_lang, ['model_lang'])
        self.create_stacked_bar(data_model_lang, 'model_lang',
                               'Por Modelo e Idioma', ax=ax3)

        # 4. Por característica (top 10)
        ax4 = plt.subplot(2, 3, 4)
        char_counts = df_final['caracteristica'].value_counts().head(10).index
        df_filtered = df_final[df_final['caracteristica'].isin(char_counts)]
        data_char = self.prepare_data(df_filtered, ['caracteristica'])
        self.create_stacked_bar(data_char, 'caracteristica',
                               'Por Característica (Top 10)', ax=ax4, rotation=60)

        # 5. GPT-4o-mini por idioma
        ax5 = plt.subplot(2, 3, 5)
        df_gpt4 = df_final[df_final['modelo'] == 'gpt-4o-mini']
        data_gpt4 = self.prepare_data(df_gpt4, ['idioma'])
        self.create_stacked_bar(data_gpt4, 'idioma',
                               'GPT-4o-mini por Idioma', ax=ax5)

        # 6. Português por modelo
        ax6 = plt.subplot(2, 3, 6)
        df_pt = df_final[df_final['idioma'] == 'pt']
        data_pt = self.prepare_data(df_pt, ['modelo'])
        self.create_stacked_bar(data_pt, 'modelo',
                               'Português por Modelo', ax=ax6)

        plt.tight_layout()
        return fig

    def plot_test1_detailed(self, df_final: pd.DataFrame) -> plt.Figure:
        """
        Teste 1: Resultados detalhados por característica e modelo
        """
        # Filtrar características mais comuns
        top_chars = df_final['caracteristica'].value_counts().head(12).index
        df_filtered = df_final[df_final['caracteristica'].isin(top_chars)]

        fig = plt.figure(figsize=(18, 10))
        fig.suptitle('Teste 1: Distribuição Detalhada por Característica e Modelo',
                     fontsize=16, fontweight='bold', y=1.02)

        models = df_filtered['modelo'].unique()

        for i, model in enumerate(models, 1):
            ax = plt.subplot(2, 3, i)
            df_model = df_filtered[df_filtered['modelo'] == model]

            # PT
            ax_pt = plt.subplot(2, 3, i)
            df_pt = df_model[df_model['idioma'] == 'pt']
            data_pt = self.prepare_data(df_pt, ['caracteristica'])
            self.create_stacked_bar(data_pt, 'caracteristica',
                                   f'{model} - PT', ax=ax_pt, rotation=60)

            # EN
            ax_en = plt.subplot(2, 3, i+3)
            df_en = df_model[df_model['idioma'] == 'en']
            data_en = self.prepare_data(df_en, ['caracteristica'])
            self.create_stacked_bar(data_en, 'caracteristica',
                                   f'{model} - EN', ax=ax_en, rotation=60)

        plt.tight_layout()
        return fig

    def plot_test2_aggregated(self, df_test2: pd.DataFrame) -> plt.Figure:
        """
        Teste 2: Gráficos agregados por valência
        """
        fig = plt.figure(figsize=self.figsize)
        fig.suptitle('Teste 2: Distribuição de Gênero por Valência (Agregado)',
                     fontsize=16, fontweight='bold', y=1.02)

        # 1. Por valência geral
        ax1 = plt.subplot(2, 3, 1)
        data_val = self.prepare_data(df_test2, ['valencia'])
        self.create_stacked_bar(data_val, 'valencia',
                               'Por Valência', ax=ax1)

        # 2. Por modelo
        ax2 = plt.subplot(2, 3, 2)
        data_model = self.prepare_data(df_test2, ['modelo'])
        self.create_stacked_bar(data_model, 'modelo',
                               'Por Modelo', ax=ax2)

        # 3. Por idioma
        ax3 = plt.subplot(2, 3, 3)
        data_lang = self.prepare_data(df_test2, ['idioma'])
        self.create_stacked_bar(data_lang, 'idioma',
                               'Por Idioma', ax=ax3)

        # 4. Por modelo-valência
        ax4 = plt.subplot(2, 3, 4)
        df_model_val = df_test2.copy()
        df_model_val['model_val'] = df_model_val['modelo'] + '\n' + df_model_val['valencia'].astype(str)
        data_model_val = self.prepare_data(df_model_val, ['model_val'])
        self.create_stacked_bar(data_model_val, 'model_val',
                               'Por Modelo e Valência', ax=ax4, rotation=0)

        # 5. Por idioma-valência
        ax5 = plt.subplot(2, 3, 5)
        df_lang_val = df_test2.copy()
        df_lang_val['lang_val'] = df_lang_val['idioma'] + ' - ' + df_lang_val['valencia'].astype(str)
        data_lang_val = self.prepare_data(df_lang_val, ['lang_val'])
        self.create_stacked_bar(data_lang_val, 'lang_val',
                               'Por Idioma e Valência', ax=ax5)

        # 6. Modelo-Idioma-Valência (selecionado)
        ax6 = plt.subplot(2, 3, 6)
        df_all = df_test2.copy()
        df_all['all'] = df_all['modelo'] + '-' + df_all['idioma'] + '\n' + df_all['valencia'].astype(str)
        data_all = self.prepare_data(df_all, ['all'])
        self.create_stacked_bar(data_all, 'all',
                               'Modelo-Idioma-Valência', ax=ax6, rotation=45)

        plt.tight_layout()
        return fig

    def plot_test2_detailed(self, df_test2: pd.DataFrame) -> plt.Figure:
        """
        Teste 2: Resultados detalhados por valência
        """
        fig = plt.figure(figsize=(16, 10))
        fig.suptitle('Teste 2: Distribuição Detalhada por Valência',
                     fontsize=16, fontweight='bold', y=1.02)

        models = df_test2['modelo'].unique()
        languages = ['pt', 'en']

        plot_idx = 1
        for model in models:
            for lang in languages:
                ax = plt.subplot(3, 2, plot_idx)
                df_filtered = df_test2[(df_test2['modelo'] == model) &
                                       (df_test2['idioma'] == lang)]
                if not df_filtered.empty:
                    data = self.prepare_data(df_filtered, ['valencia'])
                    self.create_stacked_bar(data, 'valencia',
                                          f'{model} - {lang.upper()}', ax=ax, rotation=0)
                plot_idx += 1

        plt.tight_layout()
        return fig

    def plot_test3_aggregated(self, df_test3: pd.DataFrame) -> plt.Figure:
        """
        Teste 3: Gráficos agregados por posição ocupacional
        """
        fig = plt.figure(figsize=self.figsize)
        fig.suptitle('Teste 3: Distribuição de Gênero por Posição Ocupacional (Agregado)',
                     fontsize=16, fontweight='bold', y=1.02)

        # Mapear posições para categorias
        position_map = {
            'manager': 'Manager', 'gerente': 'Manager',
            'secretary': 'Secretary', 'secretario(a)': 'Secretary',
            'white collar': 'White Collar', 'funcionário(a) de escritório': 'White Collar',
            'blue collar': 'Blue Collar', 'funcionário(a) braçal': 'Blue Collar',
            'physician': 'Physician', 'médico(a)': 'Physician',
            'nurse': 'Nurse', 'enfermeiro(a)': 'Nurse',
            'teacher': 'Teacher', 'professor(a) do ensino básico': 'Teacher',
            'professor': 'Professor', 'professor(a) universitário(a)': 'Professor',
            'plane pilot': 'Pilot', 'piloto(a) de avião': 'Pilot',
            'flight comissioner': 'Flight Attendant', 'comissionário(a) de bordo': 'Flight Attendant'
        }

        df_test3['position_category'] = df_test3['caracteristica'].map(position_map).fillna(df_test3['caracteristica'])

        # 1. Por categoria de posição
        ax1 = plt.subplot(2, 3, 1)
        data_pos = self.prepare_data(df_test3, ['position_category'])
        self.create_stacked_bar(data_pos, 'position_category',
                               'Por Categoria de Posição', ax=ax1, rotation=60)

        # 2. Por modelo
        ax2 = plt.subplot(2, 3, 2)
        data_model = self.prepare_data(df_test3, ['modelo'])
        self.create_stacked_bar(data_model, 'modelo',
                               'Por Modelo', ax=ax2)

        # 3. Por idioma
        ax3 = plt.subplot(2, 3, 3)
        data_lang = self.prepare_data(df_test3, ['idioma'])
        self.create_stacked_bar(data_lang, 'idioma',
                               'Por Idioma', ax=ax3)

        # 4. Top 6 posições
        ax4 = plt.subplot(2, 3, 4)
        top_positions = df_test3['position_category'].value_counts().head(6).index
        df_top = df_test3[df_test3['position_category'].isin(top_positions)]
        data_top = self.prepare_data(df_top, ['position_category'])
        self.create_stacked_bar(data_top, 'position_category',
                               'Top 6 Posições', ax=ax4, rotation=45)

        # 5. Healthcare positions
        ax5 = plt.subplot(2, 3, 5)
        healthcare = ['Physician', 'Nurse']
        df_health = df_test3[df_test3['position_category'].isin(healthcare)]
        df_health['model_pos'] = df_health['modelo'] + '\n' + df_health['position_category']
        data_health = self.prepare_data(df_health, ['model_pos'])
        self.create_stacked_bar(data_health, 'model_pos',
                               'Healthcare por Modelo', ax=ax5, rotation=0)

        # 6. Blue vs White collar
        ax6 = plt.subplot(2, 3, 6)
        collar = ['Blue Collar', 'White Collar']
        df_collar = df_test3[df_test3['position_category'].isin(collar)]
        df_collar['model_pos'] = df_collar['modelo'] + '\n' + df_collar['position_category']
        data_collar = self.prepare_data(df_collar, ['model_pos'])
        self.create_stacked_bar(data_collar, 'model_pos',
                               'Blue vs White Collar por Modelo', ax=ax6, rotation=0)

        plt.tight_layout()
        return fig

    def plot_test3_detailed(self, df_test3: pd.DataFrame) -> plt.Figure:
        """
        Teste 3: Resultados detalhados por profissão
        """
        # Mapear posições
        position_map = {
            'manager': 'Manager', 'gerente': 'Manager',
            'secretary': 'Secretary', 'secretario(a)': 'Secretary',
            'white collar': 'White Collar', 'funcionário(a) de escritório': 'White Collar',
            'blue collar': 'Blue Collar', 'funcionário(a) braçal': 'Blue Collar',
            'physician': 'Physician', 'médico(a)': 'Physician',
            'nurse': 'Nurse', 'enfermeiro(a)': 'Nurse',
            'teacher': 'Teacher', 'professor(a) do ensino básico': 'Teacher',
            'professor': 'Professor', 'professor(a) universitário(a)': 'Professor',
            'plane pilot': 'Pilot', 'piloto(a) de avião': 'Pilot',
            'flight comissioner': 'Flight Attendant', 'comissionário(a) de bordo': 'Flight Attendant'
        }

        df_test3['position_category'] = df_test3['caracteristica'].map(position_map).fillna(df_test3['caracteristica'])

        fig = plt.figure(figsize=(18, 12))
        fig.suptitle('Teste 3: Distribuição Detalhada por Profissão e Modelo',
                     fontsize=16, fontweight='bold', y=1.02)

        models = df_test3['modelo'].unique()
        languages = ['pt', 'en']

        for i, model in enumerate(models):
            # PT
            ax_pt = plt.subplot(3, 2, i*2 + 1)
            df_pt = df_test3[(df_test3['modelo'] == model) & (df_test3['idioma'] == 'pt')]
            if not df_pt.empty:
                data_pt = self.prepare_data(df_pt, ['position_category'])
                # Pegar top 8 posições
                top_pos = df_pt['position_category'].value_counts().head(8).index
                data_pt = data_pt[data_pt['position_category'].isin(top_pos)]
                self.create_stacked_bar(data_pt, 'position_category',
                                       f'{model} - PT', ax=ax_pt, rotation=60)

            # EN
            ax_en = plt.subplot(3, 2, i*2 + 2)
            df_en = df_test3[(df_test3['modelo'] == model) & (df_test3['idioma'] == 'en')]
            if not df_en.empty:
                data_en = self.prepare_data(df_en, ['position_category'])
                # Pegar top 8 posições
                top_pos = df_en['position_category'].value_counts().head(8).index
                data_en = data_en[data_en['position_category'].isin(top_pos)]
                self.create_stacked_bar(data_en, 'position_category',
                                       f'{model} - EN', ax=ax_en, rotation=60)

        plt.tight_layout()
        return fig


def generate_all_plots(df_final: pd.DataFrame,
                      df_test2: pd.DataFrame,
                      df_test3: pd.DataFrame,
                      colors: Dict[str, str] = None,
                      save_path: str = '/content'):
    """
    Gera todos os 6 tipos de gráficos solicitados

    Parameters:
    -----------
    df_final : DataFrame do teste 1
    df_test2 : DataFrame do teste 2
    df_test3 : DataFrame do teste 3
    colors : Dicionário com cores customizadas
    save_path : Caminho para salvar os gráficos
    """

    plotter = GenderDistributionPlotter(colors=colors)

    print("Gerando gráficos...")

    # Teste 1
    print("  1. Teste 1 - Agregado...")
    fig1 = plotter.plot_test1_aggregated(df_final)
    fig1.savefig(f'{save_path}test1_aggregated.png', dpi=150, bbox_inches='tight')

    print("  2. Teste 1 - Detalhado...")
    fig2 = plotter.plot_test1_detailed(df_final)
    fig2.savefig(f'{save_path}test1_detailed.png', dpi=150, bbox_inches='tight')

    # Teste 2
    print("  3. Teste 2 - Agregado...")
    fig3 = plotter.plot_test2_aggregated(df_test2)
    fig3.savefig(f'{save_path}test2_aggregated.png', dpi=150, bbox_inches='tight')

    print("  4. Teste 2 - Detalhado...")
    fig4 = plotter.plot_test2_detailed(df_test2)
    fig4.savefig(f'{save_path}test2_detailed.png', dpi=150, bbox_inches='tight')

    # Teste 3
    print("  5. Teste 3 - Agregado...")
    fig5 = plotter.plot_test3_aggregated(df_test3)
    fig5.savefig(f'{save_path}test3_aggregated.png', dpi=150, bbox_inches='tight')

    print("  6. Teste 3 - Detalhado...")
    fig6 = plotter.plot_test3_detailed(df_test3)
    fig6.savefig(f'{save_path}test3_detailed.png', dpi=150, bbox_inches='tight')

    print("\nTodos os gráficos foram salvos com sucesso!")

    return fig1, fig2, fig3, fig4, fig5, fig6


# Exemplo de uso
if __name__ == "__main__":
    # Cores customizáveis - você pode modificar estas cores
    custom_colors = {
        'Male': '#3498db',      # Azul
        'Female': '#e74c3c',    # Vermelho
        'Unknown': '#95a5a6',   # Cinza
        'Non-Binary': '#9b59b6' # Roxo
    }

    # Para usar:
    # generate_all_plots(df_final, df_final_test_2, df_final_test_3, colors=custom_colors)

    print("Script pronto para uso!")
    print("\nComo usar:")
    print("1. Carregue seus dataframes: df_final, df_final_test_2, df_final_test_3")
    print("2. Customize as cores se desejar no dicionário 'custom_colors'")
    print("3. Execute: generate_all_plots(df_final, df_final_test_2, df_final_test_3, colors=custom_colors)")

In [ ]:
generate_all_plots(df_final, df_final_test_2, df_final_test_3, colors=custom_colors)

In [ ]:
"""
Script para gerar os 6 tipos específicos de gráficos solicitados
================================================================
Este script gera exatamente os gráficos conforme especificado:
1. Teste 1 - Agregações
2. Teste 1 - Resultados por habilidade e modelo
3. Teste 2 - Agregações
4. Teste 2 - Resultados por valência
5. Teste 3 - Agregações
6. Teste 3 - Resultados por profissão
"""

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict
import warnings
warnings.filterwarnings('ignore')

# Configuração de estilo melhorada
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['figure.titlesize'] = 15


def create_stacked_bar_chart(data: pd.DataFrame,
                            title: str,
                            colors: Dict[str, str],
                            figsize=(14, 8),
                            show_percentages=True,
                            rotation=45):
    """
    Cria gráfico de barras empilhadas otimizado
    """
    fig, ax = plt.subplots(figsize=figsize)

    # Identificar colunas de gênero
    gender_cols = []
    for col in ['Female', 'Male', 'Unknown', 'Non-Binary']:
        if col in data.columns:
            gender_cols.append(col)

    # Preparar dados para plotagem
    x_labels = data.iloc[:, 0].astype(str)  # Primeira coluna é o label
    x_pos = np.arange(len(x_labels))

    # Criar barras empilhadas
    bottom = np.zeros(len(data))

    for gender in gender_cols:
        values = data[gender].values
        bars = ax.bar(x_pos, values, bottom=bottom,
                      label=gender, color=colors.get(gender, '#888888'),
                      edgecolor='white', linewidth=0.5)

        # Adicionar porcentagens nas barras
        if show_percentages:
            for i, (bar, val) in enumerate(zip(bars, values)):
                if val > 4:  # Mostrar apenas se > 4%
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2.,
                           bottom[i] + height/2.,
                           f'{val:.0f}%',
                           ha='center', va='center',
                           color='white', fontweight='bold',
                           fontsize=10)

        bottom += values

    # Configurações do gráfico
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=rotation, ha='right' if rotation > 0 else 'center')
    ax.set_ylabel('Percentage (%)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_ylim(0, 105)

    # Remover spines (sem eixos tracejados)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Grid suave apenas no eixo Y
    ax.yaxis.grid(True, linestyle=':', alpha=0.3)
    ax.set_axisbelow(True)

    # Legenda com borda
    legend = ax.legend(loc='upper right',
                       frameon=True,
                       fancybox=True,
                       shadow=False,
                       borderpad=0.8,
                       columnspacing=1.0,
                       handletextpad=0.5)
    legend.get_frame().set_linewidth(0.5)
    legend.get_frame().set_edgecolor('#cccccc')

    plt.tight_layout()
    return fig


def prepare_percentage_data(df: pd.DataFrame,
                           group_cols: list,
                           gender_col: str = 'gender') -> pd.DataFrame:
    """
    Prepara dados calculando porcentagens por grupo
    """
    # Contagem por grupo
    grouped = df.groupby(group_cols + [gender_col]).size().reset_index(name='count')

    # Total por grupo
    totals = grouped.groupby(group_cols)['count'].sum().reset_index(name='total')

    # Calcular porcentagem
    grouped = grouped.merge(totals, on=group_cols)
    grouped['percentage'] = (grouped['count'] / grouped['total']) * 100

    # Pivot para formato wide
    pivot = grouped.pivot_table(
        index=group_cols,
        columns=gender_col,
        values='percentage',
        fill_value=0
    ).reset_index()

    return pivot


# ============================================================================
# GRÁFICO 1: TESTE 1 - AGREGAÇÕES
# ============================================================================

def plot_test1_aggregations(df_final: pd.DataFrame, colors: Dict[str, str]):
    """
    Teste 1: Múltiplas agregações por modelo e idioma
    """
    fig = plt.figure(figsize=(20, 12))
    fig.suptitle('Teste 1: Distribuição de Gênero - Agregações por Modelo e Idioma',
                 fontsize=16, fontweight='bold')

    # Subgráfico 1: Geral
    ax1 = plt.subplot(2, 3, 1)
    data_overall = prepare_percentage_data(df_final, ['modelo'])
    data_overall['modelo'] = 'Overall\n(All Models)'
    create_stacked_bar_chart(data_overall.iloc[:1], 'Overall', colors, figsize=(4,4), rotation=0)

    # Subgráfico 2: Por Modelo
    ax2 = plt.subplot(2, 3, 2)
    data_model = prepare_percentage_data(df_final, ['modelo'])
    data_model = data_model.sort_values('modelo')
    bars = []
    bottom = np.zeros(len(data_model))
    for gender in ['Female', 'Male', 'Unknown']:
        if gender in data_model.columns:
            bars.append(ax2.bar(range(len(data_model)), data_model[gender],
                               bottom=bottom, label=gender, color=colors[gender]))
            bottom += data_model[gender]
    ax2.set_xticks(range(len(data_model)))
    ax2.set_xticklabels(data_model['modelo'], rotation=0)
    ax2.set_title('Por Modelo')
    ax2.legend()

    # Subgráfico 3: Por Idioma
    ax3 = plt.subplot(2, 3, 3)
    data_lang = prepare_percentage_data(df_final, ['idioma'])
    bars = []
    bottom = np.zeros(len(data_lang))
    for gender in ['Female', 'Male', 'Unknown']:
        if gender in data_lang.columns:
            bars.append(ax3.bar(range(len(data_lang)), data_lang[gender],
                               bottom=bottom, label=gender, color=colors[gender]))
            bottom += data_lang[gender]
    ax3.set_xticks(range(len(data_lang)))
    ax3.set_xticklabels(['Português', 'English'])
    ax3.set_title('Por Idioma')

    # Subgráficos 4-6: Combinações Modelo-Idioma
    for idx, (model, ax_idx) in enumerate(zip(df_final['modelo'].unique(), [4, 5, 6])):
        ax = plt.subplot(2, 3, ax_idx)
        df_model = df_final[df_final['modelo'] == model]
        data = prepare_percentage_data(df_model, ['idioma'])
        bars = []
        bottom = np.zeros(len(data))
        for gender in ['Female', 'Male', 'Unknown']:
            if gender in data.columns:
                bars.append(ax.bar(range(len(data)), data[gender],
                                  bottom=bottom, label=gender, color=colors[gender]))
                bottom += data[gender]
        ax.set_xticks(range(len(data)))
        ax.set_xticklabels(['PT', 'EN'])
        ax.set_title(f'{model}')

    # Ajustar layout
    for ax in fig.axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle=':', alpha=0.3)
        ax.set_ylim(0, 105)
        ax.set_ylabel('Percentage (%)')

    plt.tight_layout()
    return fig


# ============================================================================
# GRÁFICO 2: TESTE 1 - RESULTADOS POR CARACTERÍSTICA
# ============================================================================

def plot_test1_by_skill(df_final: pd.DataFrame, colors: Dict[str, str]):
    """
    Teste 1: Resultados exatos por característica e modelo
    """
    # Mapear características PT/EN para categorias unificadas
    skill_map = {
        'Alta Eficiência': 'High Efficiency', 'High Efficiency': 'High Efficiency',
        'Ineficiência': 'Low Efficiency', 'Low Efficiency': 'Low Efficiency',
        # Adicionar outros mapeamentos conforme necessário
    }

    df_final['skill_unified'] = df_final['caracteristica'].map(skill_map).fillna(df_final['caracteristica'])

    fig = plt.figure(figsize=(20, 14))
    fig.suptitle('Teste 1: Distribuição por Característica e Modelo',
                 fontsize=16, fontweight='bold')

    models = df_final['modelo'].unique()
    languages = ['pt', 'en']

    plot_idx = 1
    for model in models:
        for lang in languages:
            ax = plt.subplot(3, 2, plot_idx)

            df_subset = df_final[(df_final['modelo'] == model) & (df_final['idioma'] == lang)]

            if not df_subset.empty:
                # Pegar top 8 características
                top_skills = df_subset['skill_unified'].value_counts().head(8).index
                df_plot = df_subset[df_subset['skill_unified'].isin(top_skills)]

                data = prepare_percentage_data(df_plot, ['skill_unified'])
                data = data.sort_values('skill_unified')

                # Criar barras
                bottom = np.zeros(len(data))
                for gender in ['Female', 'Male', 'Unknown']:
                    if gender in data.columns:
                        values = data[gender].values
                        bars = ax.bar(range(len(data)), values, bottom=bottom,
                                     label=gender, color=colors[gender])

                        # Adicionar porcentagens
                        for i, (bar, val) in enumerate(zip(bars, values)):
                            if val > 4:
                                ax.text(bar.get_x() + bar.get_width()/2.,
                                       bottom[i] + val/2.,
                                       f'{val:.0f}%',
                                       ha='center', va='center',
                                       color='white', fontweight='bold')

                        bottom += values

                ax.set_xticks(range(len(data)))
                ax.set_xticklabels(data['skill_unified'], rotation=45, ha='right')
                ax.set_title(f'{model} - {lang.upper()}')
                ax.set_ylim(0, 105)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.yaxis.grid(True, linestyle=':', alpha=0.3)

                if plot_idx == 1:
                    ax.legend(loc='upper right')

            plot_idx += 1

    plt.tight_layout()
    return fig


# ============================================================================
# GRÁFICO 3: TESTE 2 - AGREGAÇÕES
# ============================================================================

def plot_test2_aggregations(df_test2: pd.DataFrame, colors: Dict[str, str]):
    """
    Teste 2: Agregações por valência
    """
    fig = plt.figure(figsize=(18, 10))
    fig.suptitle('Teste 2: Distribuição de Gênero - Agregações por Valência',
                 fontsize=16, fontweight='bold')

    # 1. Por Valência
    ax1 = plt.subplot(2, 3, 1)
    data_val = prepare_percentage_data(df_test2, ['valencia'])
    create_plot_bars(ax1, data_val, 'valencia', 'Por Valência', colors)

    # 2. Por Modelo
    ax2 = plt.subplot(2, 3, 2)
    data_model = prepare_percentage_data(df_test2, ['modelo'])
    create_plot_bars(ax2, data_model, 'modelo', 'Por Modelo', colors)

    # 3. Por Idioma
    ax3 = plt.subplot(2, 3, 3)
    data_lang = prepare_percentage_data(df_test2, ['idioma'])
    create_plot_bars(ax3, data_lang, 'idioma', 'Por Idioma', colors)

    # 4-6. Modelo x Valência
    for idx, model in enumerate(df_test2['modelo'].unique()):
        ax = plt.subplot(2, 3, idx + 4)
        df_model = df_test2[df_test2['modelo'] == model]
        data = prepare_percentage_data(df_model, ['valencia'])
        create_plot_bars(ax, data, 'valencia', f'{model}', colors)

    plt.tight_layout()
    return fig


def create_plot_bars(ax, data, x_col, title, colors):
    """
    Helper para criar barras empilhadas em um subplot
    """
    bottom = np.zeros(len(data))
    x_labels = data[x_col].astype(str)

    for gender in ['Female', 'Male', 'Unknown']:
        if gender in data.columns:
            values = data[gender].values
            bars = ax.bar(range(len(data)), values, bottom=bottom,
                         label=gender, color=colors[gender])

            # Adicionar porcentagens
            for i, (bar, val) in enumerate(zip(bars, values)):
                if val > 4:
                    ax.text(bar.get_x() + bar.get_width()/2.,
                           bottom[i] + val/2.,
                           f'{val:.0f}%',
                           ha='center', va='center',
                           color='white', fontweight='bold', fontsize=9)

            bottom += values

    ax.set_xticks(range(len(data)))
    ax.set_xticklabels(x_labels, rotation=45 if len(x_labels) > 3 else 0, ha='right' if len(x_labels) > 3 else 'center')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.set_ylabel('Percentage (%)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.3)
    ax.legend(loc='upper right', fontsize=9)


# ============================================================================
# GRÁFICO 4: TESTE 2 - RESULTADOS POR VALÊNCIA
# ============================================================================

def plot_test2_by_valence(df_test2: pd.DataFrame, colors: Dict[str, str]):
    """
    Teste 2: Resultados exatos por valência
    """
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle('Teste 2: Distribuição Detalhada por Valência',
                 fontsize=16, fontweight='bold')

    models = df_test2['modelo'].unique()
    languages = ['pt', 'en']

    plot_idx = 1
    for model in models:
        for lang in languages:
            ax = plt.subplot(3, 2, plot_idx)

            df_subset = df_test2[(df_test2['modelo'] == model) &
                                 (df_test2['idioma'] == lang)]

            if not df_subset.empty:
                data = prepare_percentage_data(df_subset, ['valencia'])

                # Criar barras
                bottom = np.zeros(len(data))
                for gender in ['Female', 'Male', 'Unknown']:
                    if gender in data.columns:
                        values = data[gender].values
                        bars = ax.bar(range(len(data)), values, bottom=bottom,
                                     label=gender, color=colors[gender])

                        # Adicionar porcentagens
                        for i, (bar, val) in enumerate(zip(bars, values)):
                            if val > 4:
                                ax.text(bar.get_x() + bar.get_width()/2.,
                                       bottom[i] + val/2.,
                                       f'{val:.0f}%',
                                       ha='center', va='center',
                                       color='white', fontweight='bold')

                        bottom += values

                ax.set_xticks(range(len(data)))
                ax.set_xticklabels(data['valencia'], rotation=0)
                ax.set_title(f'{model} - {lang.upper()}', fontweight='bold')
                ax.set_ylim(0, 105)
                ax.set_ylabel('Percentage (%)')
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.yaxis.grid(True, linestyle=':', alpha=0.3)

                if plot_idx == 1:
                    ax.legend(loc='upper right')

            plot_idx += 1

    plt.tight_layout()
    return fig


# ============================================================================
# GRÁFICO 5: TESTE 3 - AGREGAÇÕES
# ============================================================================

def plot_test3_aggregations(df_test3: pd.DataFrame, colors: Dict[str, str]):
    """
    Teste 3: Agregações por posição ocupacional
    """
    fig = plt.figure(figsize=(20, 10))
    fig.suptitle('Teste 3: Distribuição de Gênero - Agregações por Posição',
                 fontsize=16, fontweight='bold')

    # Unificar posições PT/EN
    position_unify = {
        'gerente': 'Manager', 'manager': 'Manager',
        'secretario(a)': 'Secretary', 'secretary': 'Secretary',
        'funcionário(a) de escritório': 'White Collar', 'white collar': 'White Collar',
        'funcionário(a) braçal': 'Blue Collar', 'blue collar': 'Blue Collar',
        'médico(a)': 'Physician', 'physician': 'Physician',
        'enfermeiro(a)': 'Nurse', 'nurse': 'Nurse',
        'professor(a) do ensino básico': 'Teacher', 'teacher': 'Teacher',
        'professor(a) universitário(a)': 'Professor', 'professor': 'Professor',
        'piloto(a) de avião': 'Pilot', 'plane pilot': 'Pilot',
        'comissionário(a) de bordo': 'Flight Att.', 'flight comissioner': 'Flight Att.'
    }

    df_test3['position_unified'] = df_test3['caracteristica'].map(position_unify).fillna(df_test3['caracteristica'])

    # Subgráficos
    # 1. Por Posição (todas)
    ax1 = plt.subplot(2, 3, 1)
    data_pos = prepare_percentage_data(df_test3, ['position_unified'])
    create_plot_bars(ax1, data_pos, 'position_unified', 'Por Posição', colors)

    # 2. Por Modelo
    ax2 = plt.subplot(2, 3, 2)
    data_model = prepare_percentage_data(df_test3, ['modelo'])
    create_plot_bars(ax2, data_model, 'modelo', 'Por Modelo', colors)

    # 3. Por Idioma
    ax3 = plt.subplot(2, 3, 3)
    data_lang = prepare_percentage_data(df_test3, ['idioma'])
    create_plot_bars(ax3, data_lang, 'idioma', 'Por Idioma', colors)

    # 4. Blue vs White Collar
    ax4 = plt.subplot(2, 3, 4)
    collar_positions = ['Blue Collar', 'White Collar', 'Manager', 'Secretary']
    df_collar = df_test3[df_test3['position_unified'].isin(collar_positions)]
    data_collar = prepare_percentage_data(df_collar, ['position_unified'])
    create_plot_bars(ax4, data_collar, 'position_unified', 'Office Positions', colors)

    # 5. Healthcare
    ax5 = plt.subplot(2, 3, 5)
    health_positions = ['Physician', 'Nurse']
    df_health = df_test3[df_test3['position_unified'].isin(health_positions)]
    data_health = prepare_percentage_data(df_health, ['position_unified'])
    create_plot_bars(ax5, data_health, 'position_unified', 'Healthcare', colors)

    # 6. Education & Aviation
    ax6 = plt.subplot(2, 3, 6)
    other_positions = ['Teacher', 'Professor', 'Pilot', 'Flight Att.']
    df_other = df_test3[df_test3['position_unified'].isin(other_positions)]
    data_other = prepare_percentage_data(df_other, ['position_unified'])
    create_plot_bars(ax6, data_other, 'position_unified', 'Education & Aviation', colors)

    plt.tight_layout()
    return fig


# ============================================================================
# GRÁFICO 6: TESTE 3 - RESULTADOS POR PROFISSÃO
# ============================================================================

def plot_test3_by_position(df_test3: pd.DataFrame, colors: Dict[str, str]):
    """
    Teste 3: Resultados exatos por profissão
    """
    # Unificar posições
    position_unify = {
        'gerente': 'Manager', 'manager': 'Manager',
        'secretario(a)': 'Secretary', 'secretary': 'Secretary',
        'funcionário(a) de escritório': 'White Collar', 'white collar': 'White Collar',
        'funcionário(a) braçal': 'Blue Collar', 'blue collar': 'Blue Collar',
        'médico(a)': 'Physician', 'physician': 'Physician',
        'enfermeiro(a)': 'Nurse', 'nurse': 'Nurse',
        'professor(a) do ensino básico': 'Teacher', 'teacher': 'Teacher',
        'professor(a) universitário(a)': 'Professor', 'professor': 'Professor',
        'piloto(a) de avião': 'Pilot', 'plane pilot': 'Pilot',
        'comissionário(a) de bordo': 'Flight Att.', 'flight comissioner': 'Flight Att.'
    }

    df_test3['position_unified'] = df_test3['caracteristica'].map(position_unify).fillna(df_test3['caracteristica'])

    fig = plt.figure(figsize=(20, 14))
    fig.suptitle('Teste 3: Distribuição Detalhada por Profissão',
                 fontsize=16, fontweight='bold')

    models = df_test3['modelo'].unique()
    languages = ['pt', 'en']

    plot_idx = 1
    for model in models:
        for lang in languages:
            ax = plt.subplot(3, 2, plot_idx)

            df_subset = df_test3[(df_test3['modelo'] == model) &
                                 (df_test3['idioma'] == lang)]

            if not df_subset.empty:
                # Ordenar posições por frequência
                position_order = df_subset['position_unified'].value_counts().index[:10]
                df_plot = df_subset[df_subset['position_unified'].isin(position_order)]

                data = prepare_percentage_data(df_plot, ['position_unified'])
                # Reordenar conforme frequência
                data['position_unified'] = pd.Categorical(data['position_unified'],
                                                          categories=position_order,
                                                          ordered=True)
                data = data.sort_values('position_unified')

                # Criar barras
                bottom = np.zeros(len(data))
                for gender in ['Female', 'Male', 'Unknown']:
                    if gender in data.columns:
                        values = data[gender].values
                        bars = ax.bar(range(len(data)), values, bottom=bottom,
                                     label=gender, color=colors[gender])

                        # Adicionar porcentagens
                        for i, (bar, val) in enumerate(zip(bars, values)):
                            if val > 4:
                                ax.text(bar.get_x() + bar.get_width()/2.,
                                       bottom[i] + val/2.,
                                       f'{val:.0f}%',
                                       ha='center', va='center',
                                       color='white', fontweight='bold', fontsize=9)

                        bottom += values

                ax.set_xticks(range(len(data)))
                ax.set_xticklabels(data['position_unified'], rotation=45, ha='right', fontsize=9)
                ax.set_title(f'{model} - {lang.upper()}', fontweight='bold')
                ax.set_ylim(0, 105)
                ax.set_ylabel('Percentage (%)')
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.yaxis.grid(True, linestyle=':', alpha=0.3)
                ax.set_axisbelow(True)

                if plot_idx == 1:
                    ax.legend(loc='upper right', fontsize=9)

            plot_idx += 1

    plt.tight_layout()
    return fig


# ============================================================================
# FUNÇÃO PRINCIPAL PARA GERAR TODOS OS 6 GRÁFICOS
# ============================================================================

def generate_all_6_plots(df_final: pd.DataFrame,
                        df_final_test_2: pd.DataFrame,
                        df_final_test_3: pd.DataFrame,
                        colors: Dict[str, str] = None):
    """
    Gera os 6 tipos de gráficos especificados

    Parameters:
    -----------
    df_final : DataFrame do Teste 1
    df_final_test_2 : DataFrame do Teste 2
    df_final_test_3 : DataFrame do Teste 3
    colors : Dicionário com cores para cada gênero

    Returns:
    --------
    Lista com as 6 figuras geradas
    """

    # Cores padrão se não especificadas
    if colors is None:
        colors = {
            'Male': '#3498db',      # Azul
            'Female': '#e74c3c',    # Vermelho
            'Unknown': '#95a5a6',   # Cinza
            'Non-Binary': '#9b59b6' # Roxo
        }

    print("="*80)
    print("GERANDO OS 6 GRÁFICOS ESPECIFICADOS")
    print("="*80)

    figs = []

    # GRÁFICO 1
    print("\n1. Gerando Teste 1 - Agregações...")
    fig1 = plot_test1_aggregations(df_final, colors)
    fig1.savefig('/content/grafico1_test1_agregacoes.png', dpi=150, bbox_inches='tight')
    figs.append(fig1)
    print("   ✅ Salvo: grafico1_test1_agregacoes.png")

    # GRÁFICO 2
    print("\n2. Gerando Teste 1 - Por Habilidade e Modelo...")
    fig2 = plot_test1_by_skill(df_final, colors)
    fig2.savefig('/content/grafico2_test1_habilidades.png', dpi=150, bbox_inches='tight')
    figs.append(fig2)
    print("   ✅ Salvo: grafico2_test1_habilidades.png")

    # GRÁFICO 3
    print("\n3. Gerando Teste 2 - Agregações...")
    fig3 = plot_test2_aggregations(df_final_test_2, colors)
    fig3.savefig('/content/grafico3_test2_agregacoes.png', dpi=150, bbox_inches='tight')
    figs.append(fig3)
    print("   ✅ Salvo: grafico3_test2_agregacoes.png")

    # GRÁFICO 4
    print("\n4. Gerando Teste 2 - Por Valência...")
    fig4 = plot_test2_by_valence(df_final_test_2, colors)
    fig4.savefig('/content/grafico4_test2_valencia.png', dpi=150, bbox_inches='tight')
    figs.append(fig4)
    print("   ✅ Salvo: grafico4_test2_valencia.png")

    # GRÁFICO 5
    print("\n5. Gerando Teste 3 - Agregações...")
    fig5 = plot_test3_aggregations(df_final_test_3, colors)
    fig5.savefig('/content/grafico5_test3_agregacoes.png', dpi=150, bbox_inches='tight')
    figs.append(fig5)
    print("   ✅ Salvo: grafico5_test3_agregacoes.png")

    # GRÁFICO 6
    print("\n6. Gerando Teste 3 - Por Profissão...")
    fig6 = plot_test3_by_position(df_final_test_3, colors)
    fig6.savefig('/content/grafico6_test3_profissoes.png', dpi=150, bbox_inches='tight')
    figs.append(fig6)
    print("   ✅ Salvo: grafico6_test3_profissoes.png")

    print("\n" + "="*80)
    print("✅ TODOS OS 6 GRÁFICOS FORAM GERADOS COM SUCESSO!")
    print("="*80)
    print("\nArquivos salvos em /content:")
    print("  📊 grafico1_test1_agregacoes.png")
    print("  📊 grafico2_test1_habilidades.png")
    print("  📊 grafico3_test2_agregacoes.png")
    print("  📊 grafico4_test2_valencia.png")
    print("  📊 grafico5_test3_agregacoes.png")
    print("  📊 grafico6_test3_profissoes.png")

    return figs


# ============================================================================
# CÓDIGO DE TESTE/EXEMPLO
# ============================================================================

if __name__ == "__main__":
    print("""
    ╔════════════════════════════════════════════════════════════════╗
    ║     SCRIPT PARA GERAR OS 6 GRÁFICOS ESPECIFICADOS             ║
    ╠════════════════════════════════════════════════════════════════╣
    ║                                                                ║
    ║  Como usar:                                                    ║
    ║                                                                ║
    ║  1. Importe este script:                                      ║
    ║     from specific_plots import generate_all_6_plots           ║
    ║                                                                ║
    ║  2. Configure suas cores (opcional):                          ║
    ║     cores = {                                                  ║
    ║         'Male': '#0000FF',     # Azul                        ║
    ║         'Female': '#FF0000',   # Vermelho                    ║
    ║         'Unknown': '#808080',  # Cinza                       ║
    ║     }                                                          ║
    ║                                                                ║
    ║  3. Execute:                                                   ║
    ║     figs = generate_all_6_plots(                             ║
    ║         df_final,                                             ║
    ║         df_final_test_2,                                      ║
    ║         df_final_test_3,                                      ║
    ║         colors=cores                                          ║
    ║     )                                                          ║
    ║                                                                ║
    ╚════════════════════════════════════════════════════════════════╝
    """)

In [ ]:
plot_test3_aggregations(df_final_test_3,cores)

In [ ]:
df_final

In [ ]:
cores = {
    'Male': '#0000FF',     # Azul
    'Female': '#FF0000',   # Vermelho
    'Unknown': '#808080',  # Cinza
}
figs = generate_all_6_plots(
    df_final,
    df_final_test_2,
    df_final_test_3,
    colors=cores
)

In [ ]:
# Importar o script
from specific_plots import generate_all_6_plots

# Definir suas cores personalizadas (opcional)
cores_personalizadas = {
    'Male': '#3498db',      # Azul (modifique conforme desejar)
    'Female': '#e74c3c',    # Vermelho (modifique conforme desejar)
    'Unknown': '#95a5a6',   # Cinza (modifique conforme desejar)
    'Non-Binary': '#9b59b6' # Roxo (caso apareça nos dados)
}

# Gerar todos os 6 gráficos
figuras = generate_all_6_plots(
    df_final,           # Teste 1 - características
    df_final_test_2,    # Teste 2 - valência
    df_final_test_3,    # Teste 3 - posições
    colors=cores_personalizadas
)

### V3

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, Tuple, Optional
import warnings
import os

# Desativar warnings para limpar a saída
warnings.filterwarnings('ignore')

# Configuração global de estilo
plt.style.use('default')
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 14

class GenderDistributionPlotter:
    """
    Classe para criar gráficos de distribuição de gênero em barras empilhadas.
    Os gráficos foram ajustados para posicionar a legenda fora da área de plotagem
    e para incluir combinações de valência conforme solicitado.
    """
    def __init__(self, colors: Dict[str, str] = None, figsize: Tuple[int, int] = (15, 8)):
        self.colors = colors or {
            'Male': '#3498db',
            'Female': '#e74c3c',
            'Unknown': '#95a5a6',
            'Non-Binary': '#9b59b6'
        }
        self.figsize = figsize

    def prepare_data(self, df: pd.DataFrame, group_cols: list, gender_col: str = 'gender') -> pd.DataFrame:
        """Agrupa e calcula porcentagens por gênero para as colunas desejadas."""
        grouped = df.groupby(group_cols + [gender_col]).size().reset_index(name='count')
        totals = grouped.groupby(group_cols)['count'].sum().reset_index(name='total')
        grouped = grouped.merge(totals, on=group_cols)
        grouped['percentage'] = (grouped['count'] / grouped['total']) * 100
        pivot = grouped.pivot_table(index=group_cols, columns=gender_col, values='percentage', fill_value=0).reset_index()
        pivot = pivot.merge(totals, on=group_cols)
        return pivot

    def create_stacked_bar(self, data: pd.DataFrame, x_col: str, title: str,
                           ylabel: str = 'Percentage (%)', ax: Optional[plt.Axes] = None,
                           show_values: bool = True, rotation: int = 45,
                           show_legend: bool = True) -> plt.Axes:
        """Cria um gráfico de barras empilhadas com legenda posicionada fora do eixo."""
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 6))
        # Identificar colunas de gênero presentes no DataFrame
        gender_cols = [col for col in data.columns if col in self.colors.keys()]
        x_labels = data[x_col].astype(str)
        x_pos = np.arange(len(x_labels))
        bottom = np.zeros(len(data))
        for gender in gender_cols:
            if gender in data.columns:
                values = data[gender].values
                bars = ax.bar(x_pos, values, bottom=bottom, label=gender, color=self.colors.get(gender, '#333333'))
                if show_values:
                    for i, (bar, val) in enumerate(zip(bars, values)):
                        if val > 3:  # somente exibir valores acima de 3%
                            height = bar.get_height()
                            ax.text(bar.get_x() + bar.get_width()/2., bottom[i] + height/2.,
                                    f'{val:.0f}%', ha='center', va='center',
                                    color='white', fontweight='bold', fontsize=9)
                bottom += values
        ax.set_xticks(x_pos)
        ax.set_xticklabels(x_labels, rotation=rotation, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight='bold', pad=20)
        ax.set_ylim(0, 105)
        # Estética
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.set_axisbelow(True)
        if show_legend:
            # Colocar a legenda fora da área de plotagem, alinhada centralmente
            #ncols = max(1, len(gender_cols))
            #ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.05), ncol=ncols, frameon=False)
            ax.legend(
                loc='upper right',
                bbox_to_anchor=(1.25, 1.0),
                ncol=1,
                frameon=True,
                fancybox=True,
                shadow=False,
                title=None
            )
        return ax

    def plot_test1_summary(self, df_final: pd.DataFrame):
        """
        Cria gráficos agregados do Teste 1, separando resultados por modelo, idioma e combinação
        modelo-idioma, com a valência quando disponível.
        Retorna duas figuras: uma com os gráficos de modelo e idioma, e outra apenas
        para a combinação modelo-idioma-valência (com largura maior para acomodar os rótulos).
        """
        # Preparar colunas auxiliares de acordo com a existência de 'valencia'
        if 'valencia' in df_final.columns:
            df_temp = df_final.copy()
            df_temp['model_val'] = df_temp['modelo'] + ' - ' + df_temp['valencia'].astype(str)
            df_temp['lang_val'] = df_temp['idioma'] + ' - ' + df_temp['valencia'].astype(str)
            df_temp['model_lang_val'] = df_temp['modelo'] + ' - ' + df_temp['idioma'] + '' + df_temp['valencia'].astype(str)
        else:
            df_temp = df_final.copy()
            df_temp['model_val'] = df_temp['modelo']
            df_temp['lang_val'] = df_temp['idioma']
            df_temp['model_lang_val'] = df_temp['modelo'] + ' - ' + df_temp['idioma']

        # Figura principal com duas colunas
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))
        fig.suptitle('Teste 1: Distribuição de Gênero por Modelo e Idioma (Valência)', fontsize=16, fontweight='bold', y=1.05)

        # Gráfico por modelo
        data_model = self.prepare_data(df_temp, ['model_val'])
        self.create_stacked_bar(data_model, 'model_val', 'Por Modelo', ax=axes[0], rotation=45)

        # Gráfico por idioma
        data_lang = self.prepare_data(df_temp, ['lang_val'])
        self.create_stacked_bar(data_lang, 'lang_val', 'Por Idioma', ax=axes[1], rotation=45)

        plt.tight_layout()

        # Segunda figura apenas com o gráfico de modelo-idioma-valência
        fig_ml, ax_ml = plt.subplots(1, 1, figsize=(18, 6))
        fig_ml.suptitle('Teste 1: Por Modelo, Idioma e Valência', fontsize=16, fontweight='bold', y=1.05)
        data_ml = self.prepare_data(df_temp, ['model_lang_val'])
        self.create_stacked_bar(data_ml, 'model_lang_val', 'Por Modelo, Idioma e Valência', ax=ax_ml, rotation=45)
        plt.tight_layout()
        return fig, fig_ml

    def plot_test1_detailed(self, df_final: pd.DataFrame):
        """
        Mantém a implementação detalhada para o Teste 1, dividindo por modelo e idioma e
        mostrando as características mais comuns. A legenda é omitida para evitar
        poluição visual, pois os gráficos possuem múltiplas barras.
        """
        top_chars = df_final['caracteristica'].value_counts().head(12).index
        df_filtered = df_final[df_final['caracteristica'].isin(top_chars)]
        fig = plt.figure(figsize=(18, 10))
        fig.suptitle('Teste 1: Distribuição Detalhada por Característica e Modelo', fontsize=16, fontweight='bold', y=1.02)
        models = df_filtered['modelo'].unique()
        for i, model in enumerate(models, 1):
            # Português
            ax_pt = plt.subplot(2, 3, i)
            df_pt = df_filtered[(df_filtered['modelo'] == model) & (df_filtered['idioma'] == 'pt')]
            if not df_pt.empty:
                data_pt = self.prepare_data(df_pt, ['caracteristica'])
                self.create_stacked_bar(data_pt, 'caracteristica', f'{model} - PT', ax=ax_pt, rotation=60, show_legend=False)
            else:
                ax_pt.axis('off')
            # Inglês
            ax_en = plt.subplot(2, 3, i + 3)
            df_en = df_filtered[(df_filtered['modelo'] == model) & (df_filtered['idioma'] == 'en')]
            if not df_en.empty:
                data_en = self.prepare_data(df_en, ['caracteristica'])
                self.create_stacked_bar(data_en, 'caracteristica', f'{model} - EN', ax=ax_en, rotation=60, show_legend=False)
            else:
                ax_en.axis('off')
        plt.tight_layout()
        return fig

    def plot_test2_summary(self, df_test2: pd.DataFrame):
        """
        Cria gráficos agregados do Teste 2 por idioma-valência e por modelo-valência.
        """
        df_temp = df_test2.copy()
        df_temp['lang_val'] = df_temp['idioma'] + ' - ' + df_temp['valencia'].astype(str)
        df_temp['model_val'] = df_temp['modelo'] + ' - ' + df_temp['valencia'].astype(str)
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))
        fig.suptitle('Teste 2: Distribuição de Gênero por Idioma-Valência e Modelo-Valência', fontsize=16, fontweight='bold', y=1.05)
        # Por idioma-valência
        data_lang_val = self.prepare_data(df_temp, ['lang_val'])
        self.create_stacked_bar(data_lang_val, 'lang_val', 'Por Idioma e Valência', ax=axes[0], rotation=45)
        # Por modelo-valência
        data_model_val = self.prepare_data(df_temp, ['model_val'])
        self.create_stacked_bar(data_model_val, 'model_val', 'Por Modelo e Valência', ax=axes[1], rotation=45)
        plt.tight_layout()
        return fig

    def plot_test2_detailed(self, df_test2: pd.DataFrame):
        """
        Cria gráficos detalhados do Teste 2 por modelo e idioma, mostrando a distribuição
        de gênero por valência. Cada subgráfico corresponde a uma combinação modelo-idioma.
        A legenda foi removida desses subgráficos.
        """
        models = df_test2['modelo'].unique()
        languages = ['pt', 'en']
        rows = len(models)
        fig = plt.figure(figsize=(14, 4 * rows))
        fig.suptitle('Teste 2: Distribuição Detalhada por Modelo, Idioma e Valência', fontsize=16, fontweight='bold', y=1.02)
        plot_idx = 1
        for model in models:
            for lang in languages:
                ax = plt.subplot(rows, 2, plot_idx)
                df_filtered = df_test2[(df_test2['modelo'] == model) & (df_test2['idioma'] == lang)]
                if not df_filtered.empty:
                    data = self.prepare_data(df_filtered, ['valencia'])
                    title = f'{model} - {lang.upper()}'
                    self.create_stacked_bar(data, 'valencia', title, ax=ax, rotation=0, show_values=False, show_legend=False)
                else:
                    ax.axis('off')
                plot_idx += 1
        plt.tight_layout()
        return fig

    def plot_test3_language(self, df_test3: pd.DataFrame, lang: str):
        """
        Cria um conjunto de gráficos para o Teste 3 em um idioma específico. O primeiro
        gráfico apresenta a distribuição geral das profissões (categorias) nesse idioma,
        enquanto os gráficos seguintes mostram a distribuição por modelo.
        """
        df_lang = df_test3[df_test3['idioma'] == lang].copy()
        # Mapear profissões para categorias unificadas
        position_map = {
            'manager': 'Manager', 'gerente': 'Manager',
            'secretary': 'Secretary', 'secretario(a)': 'Secretary',
            'white collar': 'White Collar', 'funcionário(a) de escritório': 'White Collar',
            'blue collar': 'Blue Collar', 'funcionário(a) braçal': 'Blue Collar',
            'physician': 'Physician', 'médico(a)': 'Physician',
            'nurse': 'Nurse', 'enfermeiro(a)': 'Nurse',
            'teacher': 'Teacher', 'professor(a) do ensino básico': 'Teacher',
            'professor': 'Professor', 'professor(a) universitário(a)': 'Professor',
            'plane pilot': 'Pilot', 'piloto(a) de avião': 'Pilot',
            'flight comissioner': 'Flight Attendant', 'comissionário(a) de bordo': 'Flight Attendant'}
        df_lang['position_category'] = df_lang['caracteristica'].map(position_map).fillna(df_lang['caracteristica'])
        models = df_lang['modelo'].unique()
        num_cols = max(len(models), 1)
        fig, axes = plt.subplots(2, num_cols, figsize=(6 * num_cols, 8))
        fig.suptitle(f'Teste 3: Distribuição de Gênero por Profissão – Idioma {lang.upper()}', fontsize=16, fontweight='bold', y=1.05)
        if len(models) == 1:
            axes = np.array([[axes[0]], [axes[1]]])
        # Gráfico geral (top 8 posições)
        data_general = self.prepare_data(df_lang, ['position_category'])
                # nova ordem desejada
        ordered_positions = [
            'Blue Collar', 'White Collar', 'Secretary', 'Manager',
            'Nurse', 'Physician', 'Teacher', 'Professor',
            'Pilot', 'Flight Attendant'
        ]
        #top_positions = df_lang['position_category'].value_counts().head(8).index
        top_positions = df_lang['position_category'].value_counts().index
        data_general = data_general[data_general['position_category'].isin(top_positions)]
        # filtrar e reordenar conforme a lista acima
        data_general = data_general[data_general['position_category'].isin(ordered_positions)]
        data_general['position_category'] = pd.Categorical(
            data_general['position_category'],
            categories=ordered_positions,
            ordered=True
        )
        data_general = data_general.sort_values('position_category')

        self.create_stacked_bar(
            data_general,
            'position_category',
            'Profissões – Geral',
            ax=axes[0][0],
            rotation=60
        )
        for j in range(1, num_cols):
            axes[0][j].axis('off')
        # Gráficos por modelo (top 8 posições por modelo)
        for idx, model in enumerate(models):
            ax = axes[1][idx]
            df_model = df_lang[df_lang['modelo'] == model]
            if not df_model.empty:
                data_model = self.prepare_data(df_model, ['position_category'])
                top_pos_model = df_model['position_category'].value_counts().head(8).index
                #data_model = data_model[data_model['position_category'].isin(top_pos_model)]
                data_model = data_model[data_model['position_category'].isin(ordered_positions)]
                data_model['position_category'] = pd.Categorical(
                    data_model['position_category'],
                    categories=ordered_positions,
                    ordered=True
                )
                data_model = data_model.sort_values('position_category')
                self.create_stacked_bar(
                    data_model,
                    'position_category',
                    f'{model}',
                    ax=ax,
                    rotation=60,
                    show_legend=False
                )


                self.create_stacked_bar(data_model, 'position_category', f'{model}', ax=ax, rotation=60, show_legend=False)
            else:
                ax.axis('off')
        if len(models) < axes.shape[1]:
            for j in range(len(models), axes.shape[1]):
                axes[1][j].axis('off')
        plt.tight_layout()
        return fig

def generate_all_plots(df_final: pd.DataFrame, df_test2: pd.DataFrame, df_test3: pd.DataFrame,
                      colors: Dict[str, str] = None, save_path: str = ''):
    """
    Função utilitária para gerar e salvar todos os gráficos dos três testes.
    Ajuste o parâmetro `save_path` para definir o diretório onde os arquivos
    de imagem serão salvos. Por padrão, salva no diretório atual.
    """
    plotter = GenderDistributionPlotter(colors=colors)
    print("Gerando gráficos...")
    # Teste 1 - resumo
    print("  1. Teste 1 - Resumo...")
    fig1_main, fig1_ml = plotter.plot_test1_summary(df_final)
    fig1_main.savefig(os.path.join(save_path, 'test1_summary_model_lang.png'), dpi=150, bbox_inches='tight')
    fig1_ml.savefig(os.path.join(save_path, 'test1_summary_model_language_val.png'), dpi=150, bbox_inches='tight')
    # Teste 1 - detalhado
    print("  2. Teste 1 - Detalhado...")
    fig1_detail = plotter.plot_test1_detailed(df_final)
    fig1_detail.savefig(os.path.join(save_path, 'test1_detailed.png'), dpi=150, bbox_inches='tight')
    # Teste 2 - resumo
    print("  3. Teste 2 - Resumo...")
    fig2_summary = plotter.plot_test2_summary(df_test2)
    fig2_summary.savefig(os.path.join(save_path, 'test2_summary.png'), dpi=150, bbox_inches='tight')
    # Teste 2 - detalhado
    print("  4. Teste 2 - Detalhado...")
    fig2_detail = plotter.plot_test2_detailed(df_test2)
    fig2_detail.savefig(os.path.join(save_path, 'test2_detailed.png'), dpi=150, bbox_inches='tight')
    # Teste 3 - idioma pt
    print("  5. Teste 3 - Idioma PT...")
    fig3_pt = plotter.plot_test3_language(df_test3, 'pt')
    fig3_pt.savefig(os.path.join(save_path, 'test3_portugues.png'), dpi=150, bbox_inches='tight')
    # Teste 3 - idioma en
    print("  6. Teste 3 - Idioma EN...")
    fig3_en = plotter.plot_test3_language(df_test3, 'en')
    fig3_en.savefig(os.path.join(save_path, 'test3_ingles.png'), dpi=150, bbox_inches='tight')
    print("Todos os gráficos foram gerados e salvos!")
    return (fig1_main, fig1_ml, fig1_detail, fig2_summary, fig2_detail, fig3_pt, fig3_en)


In [ ]:
df_final_test_3

In [ ]:
generate_all_plots(df_final,df_final_test_2,df_final_test_3,colors=custom_colors,save_path='/content')

In [ ]:
df_final_test_3

In [ ]:
filtered_df = df_final_test_3[df_final_test_3['caracteristica'].isin(['secretary', 'nurse', 'physician'])]
display(filtered_df)

# Regressão OLS

In [ ]:
# ========================================
# ANÁLISE DE REGRESSÃO OLS - GENDER VS VALENCIA
# ========================================

# Imports necessários
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

print("📦 Bibliotecas importadas com sucesso!")

# ========================================
# CARREGANDO OS DATAFRAMES
# ========================================



print(f"✅ df_final carregado: {df_final.shape}")
print(f"✅ df_final_test_2 carregado: {df_final_test_2.shape}")
print(f"✅ df_final_test_3 carregado: {df_final_test_3.shape}")

# ========================================
# FUNÇÕES AUXILIARES
# ========================================

def run_regression(df, formula, description=""):
    """Roda uma regressão OLS e retorna resultados formatados"""
    try:
        model = ols(formula, data=df).fit()
        beta = model.params.get('X', model.params[1] if len(model.params) > 1 else None)
        p_value = model.pvalues.get('X', model.pvalues[1] if len(model.pvalues) > 1 else None)
        r2 = model.rsquared
        n_obs = int(model.nobs)

        print(f"\n{'='*60}")
        print(f"📊 {description}")
        print(f"{'='*60}")
        print(f"Beta (X): {beta:.4f}")
        print(f"P-value: {p_value:.4f}")
        print(f"R²: {r2:.4f}")
        print(f"N obs: {n_obs}")

        return {
            'description': description,
            'beta': beta,
            'p_value': p_value,
            'r_squared': r2,
            'n_obs': n_obs,
            'model': model
        }
    except Exception as e:
        print(f"❌ Erro na regressão {description}: {str(e)}")
        return None

def prepare_dummies(df):
    """Prepara variáveis dummy para modelo e idioma"""
    df_work = df.copy()

    # Dummy para idioma
    df_work['idioma_pt'] = (df_work['idioma'] == 'pt').astype(int)

    # Dummies para modelos
    modelos_unicos = df_work['modelo'].unique()
    print(f"\n📋 Modelos encontrados: {list(modelos_unicos)}")

    # --- CORREÇÃO APLICADA AQUI ---
    # Filtra a lista para incluir apenas strings válidas, ignorando 'nan' (floats)
    modelos_validos = [m for m in modelos_unicos if isinstance(m, str)]
    # --- FIM DA CORREÇÃO ---

    # Itera APENAS sobre os modelos válidos
    for modelo in modelos_validos:
        col_name = f"modelo_{modelo.replace('-', '_')}"
        df_work[col_name] = (df_work['modelo'] == modelo).astype(int)

    # Retorna a lista de modelos FILTRADA (modelos_validos)
    # Isso é crucial para que a criação da fórmula da regressão completa também funcione
    return df_work, modelos_validos
# ========================================
# 1️⃣ ANÁLISE DF_FINAL
# ========================================

print("\n" + "="*80)
print("🎯 ANÁLISE DF_FINAL")
print("="*80)

# Preparando variáveis binárias
df_final_work = df_final.copy()

# Y = 1 se Male, 0 caso contrário
df_final_work['Y'] = (df_final_work['gender'].str.lower() == 'male').astype(int)

# X = 1 se valencia positiva
valencia_positivas = ['Positive', 'Positiva', 'positive', 'positiva']
df_final_work['X'] = df_final_work['valencia'].isin(valencia_positivas).astype(int)

print(f"\n📊 Distribuição de Y (Male=1): {df_final_work['Y'].value_counts().to_dict()}")
print(f"📊 Distribuição de X (Positiva=1): {df_final_work['X'].value_counts().to_dict()}")

# Visualização rápida
print("\n🔍 Primeiras 5 linhas com variáveis criadas:")
print(df_final_work[['gender', 'Y', 'valencia', 'X', 'idioma', 'modelo']].head())

# Preparando dummies
df_final_work, modelos_final = prepare_dummies(df_final_work)

# Lista para armazenar resultados
resultados_df_final = []

# 1. REGRESSÃO SIMPLES - TODOS OS IDIOMAS
result = run_regression(df_final_work, 'Y ~ X',
                        "DF_FINAL - Regressão Simples (Todos os idiomas)")
if result: resultados_df_final.append(result)

# 2. REGRESSÃO COMPLETA - TODOS OS IDIOMAS
modelo_cols = [f"modelo_{m.replace('-', '_')}" for m in modelos_final[1:]]  # Excluindo primeiro para evitar multicolinearidade
formula_completa = f"Y ~ X + idioma_pt + {' + '.join(modelo_cols)}"
result = run_regression(df_final_work, formula_completa,
                        "DF_FINAL - Regressão Completa (Todos os idiomas)")
if result: resultados_df_final.append(result)

# 3. REGRESSÃO SIMPLES - APENAS PORTUGUÊS
df_pt = df_final_work[df_final_work['idioma'] == 'pt'].copy()
result = run_regression(df_pt, 'Y ~ X',
                        "DF_FINAL - Regressão Simples (Apenas Português)")
if result: resultados_df_final.append(result)

# 4. REGRESSÃO COMPLETA - APENAS PORTUGUÊS
result = run_regression(df_pt, f"Y ~ X + {' + '.join(modelo_cols)}",
                        "DF_FINAL - Regressão Completa (Apenas Português)")
if result: resultados_df_final.append(result)

# 5. REGRESSÃO SIMPLES - APENAS INGLÊS
df_en = df_final_work[df_final_work['idioma'] == 'en'].copy()
result = run_regression(df_en, 'Y ~ X',
                        "DF_FINAL - Regressão Simples (Apenas Inglês)")
if result: resultados_df_final.append(result)

# 6. REGRESSÃO COMPLETA - APENAS INGLÊS
result = run_regression(df_en, f"Y ~ X + {' + '.join(modelo_cols)}",
                        "DF_FINAL - Regressão Completa (Apenas Inglês)")
if result: resultados_df_final.append(result)

# Criando DataFrame com resumo dos resultados
df_resumo_final = pd.DataFrame([
    {
        'Análise': r['description'],
        'Beta': r['beta'],
        'P-value': r['p_value'],
        'R²': r['r_squared'],
        'N': r['n_obs']
    } for r in resultados_df_final if r
])

print("\n📊 RESUMO DOS RESULTADOS - DF_FINAL:")
print(df_resumo_final.to_string())

# ========================================
# 2️⃣ ANÁLISE DF_FINAL_TEST_2
# ========================================

print("\n" + "="*80)
print("🎯 ANÁLISE DF_FINAL_TEST_2")
print("="*80)

# Preparando o dataframe
df_test2_work = df_final_test_2.copy()

# Normalizando valencia
print("\n🔧 Normalizando coluna 'valencia'...")
print(f"Valores únicos antes: {df_test2_work['valencia'].unique()}")

# Mapeamento para normalizar
valencia_map = {
    'positive': 'positive',
    'Positive': 'positive',
    'positiva': 'positive',
    'Positiva': 'positive',
    'positivo': 'positive',
    'Positivo': 'positive',
    'negative': 'negative',
    'Negative': 'negative',
    'negativa': 'negative',
    'Negativa': 'negative',
    'negativo': 'negative',
    'Negativo': 'negative'
}

df_test2_work['valencia'] = df_test2_work['valencia'].map(valencia_map).fillna(df_test2_work['valencia'])
print(f"Valores únicos depois: {df_test2_work['valencia'].unique()}")

# Criando variáveis binárias
df_test2_work['Y'] = (df_test2_work['gender'].str.lower() == 'male').astype(int)
df_test2_work['X'] = (df_test2_work['valencia'] == 'positive').astype(int)

print(f"\n📊 Distribuição de Y (Male=1): {df_test2_work['Y'].value_counts().to_dict()}")
print(f"📊 Distribuição de X (Positiva=1): {df_test2_work['X'].value_counts().to_dict()}")

# Visualização rápida
print("\n🔍 Primeiras 5 linhas com variáveis criadas:")
print(df_test2_work[['gender', 'Y', 'valencia', 'X', 'idioma', 'modelo']].head())

# Preparando dummies
df_test2_work, modelos_test2 = prepare_dummies(df_test2_work)

# Lista para armazenar resultados
resultados_df_test2 = []

# 1. REGRESSÃO SIMPLES - TODOS OS IDIOMAS
result = run_regression(df_test2_work, 'Y ~ X',
                        "DF_TEST2 - Regressão Simples (Todos os idiomas)")
if result: resultados_df_test2.append(result)

# 2. REGRESSÃO COMPLETA - TODOS OS IDIOMAS
modelo_cols = [f"modelo_{m.replace('-', '_')}" for m in modelos_test2[1:]]
formula_completa = f"Y ~ X + idioma_pt + {' + '.join(modelo_cols)}"
result = run_regression(df_test2_work, formula_completa,
                        "DF_TEST2 - Regressão Completa (Todos os idiomas)")
if result: resultados_df_test2.append(result)

# 3. REGRESSÃO SIMPLES - APENAS PORTUGUÊS
df_pt = df_test2_work[df_test2_work['idioma'] == 'pt'].copy()
result = run_regression(df_pt, 'Y ~ X',
                        "DF_TEST2 - Regressão Simples (Apenas Português)")
if result: resultados_df_test2.append(result)

# 4. REGRESSÃO COMPLETA - APENAS PORTUGUÊS
result = run_regression(df_pt, f"Y ~ X + {' + '.join(modelo_cols)}",
                        "DF_TEST2 - Regressão Completa (Apenas Português)")
if result: resultados_df_test2.append(result)

# 5. REGRESSÃO SIMPLES - APENAS INGLÊS
df_en = df_test2_work[df_test2_work['idioma'] == 'en'].copy()
result = run_regression(df_en, 'Y ~ X',
                        "DF_TEST2 - Regressão Simples (Apenas Inglês)")
if result: resultados_df_test2.append(result)

# 6. REGRESSÃO COMPLETA - APENAS INGLÊS
result = run_regression(df_en, f"Y ~ X + {' + '.join(modelo_cols)}",
                        "DF_TEST2 - Regressão Completa (Apenas Inglês)")
if result: resultados_df_test2.append(result)

# Criando DataFrame com resumo dos resultados
df_resumo_test2 = pd.DataFrame([
    {
        'Análise': r['description'],
        'Beta': r['beta'],
        'P-value': r['p_value'],
        'R²': r['r_squared'],
        'N': r['n_obs']
    } for r in resultados_df_test2 if r
])

print("\n📊 RESUMO DOS RESULTADOS - DF_FINAL_TEST_2:")
print(df_resumo_test2.to_string())

# ========================================
# 3️⃣ ANÁLISE DF_FINAL_TEST_3
# ========================================

print("\n" + "="*80)
print("🎯 ANÁLISE DF_FINAL_TEST_3")
print("="*80)

# Preparando o dataframe
df_test3_work = df_final_test_3.copy()

# Criando Y
df_test3_work['Y'] = (df_test3_work['gender'].str.lower() == 'male').astype(int)

# Criando X para valência positiva
valencia_positivas = ['Positive', 'Positiva', 'positive', 'positiva', 'positivo', 'Positivo']
df_test3_work['X'] = df_test3_work['valencia'].isin(valencia_positivas).astype(int)

print(f"\n📊 Distribuição de Y (Male=1): {df_test3_work['Y'].value_counts().to_dict()}")
print(f"📊 Distribuição de X (Positiva=1): {df_test3_work['X'].value_counts().to_dict()}")

# Verificando profissões únicas
profissoes = df_test3_work['position_unified'].unique()
print(f"\n🏢 Total de profissões únicas: {len(profissoes)}")
print(f"Primeiras 10 profissões: {list(profissoes[:10])}")

# PRIMEIRO: Análises gerais (6 regressões como nos outros DFs)

# Preparando dummies
df_test3_work, modelos_test3 = prepare_dummies(df_test3_work)

# Lista para armazenar resultados gerais
resultados_df_test3_geral = []

# 1. REGRESSÃO SIMPLES - TODOS OS IDIOMAS
result = run_regression(df_test3_work, 'Y ~ X',
                        "DF_TEST3 - Regressão Simples (Todos os idiomas)")
if result: resultados_df_test3_geral.append(result)

# 2. REGRESSÃO COMPLETA - TODOS OS IDIOMAS
modelo_cols = [f"modelo_{m.replace('-', '_')}" for m in modelos_test3[1:]]
formula_completa = f"Y ~ X + idioma_pt + {' + '.join(modelo_cols)}"
result = run_regression(df_test3_work, formula_completa,
                        "DF_TEST3 - Regressão Completa (Todos os idiomas)")
if result: resultados_df_test3_geral.append(result)

# 3. REGRESSÃO SIMPLES - APENAS PORTUGUÊS
df_pt = df_test3_work[df_test3_work['idioma'] == 'pt'].copy()
result = run_regression(df_pt, 'Y ~ X',
                        "DF_TEST3 - Regressão Simples (Apenas Português)")
if result: resultados_df_test3_geral.append(result)

# 4. REGRESSÃO COMPLETA - APENAS PORTUGUÊS
result = run_regression(df_pt, f"Y ~ X + {' + '.join(modelo_cols)}",
                        "DF_TEST3 - Regressão Completa (Apenas Português)")
if result: resultados_df_test3_geral.append(result)

# 5. REGRESSÃO SIMPLES - APENAS INGLÊS
df_en = df_test3_work[df_test3_work['idioma'] == 'en'].copy()
result = run_regression(df_en, 'Y ~ X',
                        "DF_TEST3 - Regressão Simples (Apenas Inglês)")
if result: resultados_df_test3_geral.append(result)

# 6. REGRESSÃO COMPLETA - APENAS INGLÊS
result = run_regression(df_en, f"Y ~ X + {' + '.join(modelo_cols)}",
                        "DF_TEST3 - Regressão Completa (Apenas Inglês)")
if result: resultados_df_test3_geral.append(result)

# SEGUNDO: Análises POR PROFISSÃO
print("\n" + "="*80)
print("📊 ANÁLISES POR PROFISSÃO - DF_TEST3")
print("="*80)

resultados_por_profissao = []

for i, prof in enumerate(profissoes):
    if pd.isna(prof):
        continue

    # Filtrando dados para esta profissão
    df_prof = df_test3_work[df_test3_work['position_unified'] == prof].copy()

    if len(df_prof) < 10:  # Skip se poucos dados
        continue

    # Criando X binário para esta profissão (1 se tem esta profissão, 0 caso contrário)
    df_test3_work['X_prof'] = (df_test3_work['position_unified'] == prof).astype(int)

    # Regressão simples por profissão
    try:
        model_simple = ols('Y ~ X_prof', data=df_test3_work).fit()
        beta_simple = model_simple.params['X_prof']
        p_simple = model_simple.pvalues['X_prof']
        r2_simple = model_simple.rsquared

        resultados_por_profissao.append({
            'Profissão': prof,
            'Tipo': 'Simples',
            'Beta': beta_simple,
            'P-value': p_simple,
            'R²': r2_simple,
            'N': int(model_simple.nobs)
        })
    except:
        pass

    # Regressão com variáveis explicativas
    try:
        formula_completa_prof = f"Y ~ X_prof + idioma_pt + {' + '.join(modelo_cols)}"
        model_complex = ols(formula_completa_prof, data=df_test3_work).fit()
        beta_complex = model_complex.params['X_prof']
        p_complex = model_complex.pvalues['X_prof']
        r2_complex = model_complex.rsquared

        resultados_por_profissao.append({
            'Profissão': prof,
            'Tipo': 'Completa',
            'Beta': beta_complex,
            'P-value': p_complex,
            'R²': r2_complex,
            'N': int(model_complex.nobs)
        })
    except:
        pass

    # Limpar coluna temporária
    df_test3_work.drop('X_prof', axis=1, inplace=True)

    # Print a cada 10 profissões
    if (i + 1) % 10 == 0:
        print(f"✅ Processadas {i + 1}/{len(profissoes)} profissões...")

print(f"\n✅ Total de profissões analisadas: {len(set([r['Profissão'] for r in resultados_por_profissao]))}")

# Criando DataFrames com resumos
df_resumo_test3_geral = pd.DataFrame([
    {
        'Análise': r['description'],
        'Beta': r['beta'],
        'P-value': r['p_value'],
        'R²': r['r_squared'],
        'N': r['n_obs']
    } for r in resultados_df_test3_geral if r
])

df_resumo_profissoes = pd.DataFrame(resultados_por_profissao)

print("\n📊 RESUMO DOS RESULTADOS GERAIS - DF_TEST3:")
print(df_resumo_test3_geral.to_string())

print("\n📊 TOP 10 PROFISSÕES COM MAIOR BETA (Regressão Simples):")
df_prof_simples = df_resumo_profissoes[df_resumo_profissoes['Tipo'] == 'Simples']
print(df_prof_simples.nlargest(10, 'Beta')[['Profissão', 'Beta', 'P-value', 'R²']].to_string())

print("\n📊 TOP 10 PROFISSÕES COM MENOR BETA (Regressão Simples):")
print(df_prof_simples.nsmallest(10, 'Beta')[['Profissão', 'Beta', 'P-value', 'R²']].to_string())

# ========================================
# 📈 RESUMO FINAL CONSOLIDADO
# ========================================

# Consolidando todos os resultados
print("\n" + "="*80)
print("📊 RESUMO FINAL CONSOLIDADO DE TODAS AS ANÁLISES")
print("="*80)

# DataFrame consolidado das análises gerais
todos_resultados = pd.concat([
    df_resumo_final.assign(DataFrame='df_final'),
    df_resumo_test2.assign(DataFrame='df_final_test_2'),
    df_resumo_test3_geral.assign(DataFrame='df_final_test_3')
], ignore_index=True)

print("\n📊 TODOS OS RESULTADOS DAS REGRESSÕES GERAIS:")
print(todos_resultados.to_string())

# Estatísticas descritivas dos betas
print("\n📊 ESTATÍSTICAS DOS BETAS:")
print(f"Beta médio: {todos_resultados['Beta'].mean():.4f}")
print(f"Beta mediano: {todos_resultados['Beta'].median():.4f}")
print(f"Beta mínimo: {todos_resultados['Beta'].min():.4f}")
print(f"Beta máximo: {todos_resultados['Beta'].max():.4f}")
print(f"Desvio padrão: {todos_resultados['Beta'].std():.4f}")

# Resultados significativos
significativos = todos_resultados[todos_resultados['P-value'] < 0.05]
print(f"\n📊 RESULTADOS SIGNIFICATIVOS (p < 0.05): {len(significativos)}/{len(todos_resultados)}")
if len(significativos) > 0:
    print(significativos[['DataFrame', 'Análise', 'Beta', 'P-value']].to_string())

# Análise comparativa por idioma
print("\n📊 COMPARAÇÃO POR IDIOMA (Regressões Simples):")
comparacao_idioma = todos_resultados[todos_resultados['Análise'].str.contains('Simples')]

for df_name in ['df_final', 'df_final_test_2', 'df_final_test_3']:
    df_subset = comparacao_idioma[comparacao_idioma['DataFrame'] == df_name]
    print(f"\n{df_name}:")
    for _, row in df_subset.iterrows():
        idioma = 'Todos' if 'Todos' in row['Análise'] else ('PT' if 'Português' in row['Análise'] else 'EN')
        print(f"  {idioma}: Beta={row['Beta']:.4f}, p={row['P-value']:.4f}")

# Salvando DataFrames úteis para análise posterior
print("\n💾 DataFrames disponíveis para análise:")
print("- todos_resultados: Resumo de todas as regressões gerais")
print("- df_resumo_profissoes: Resultados por profissão (df_test3)")
print("- df_resumo_final: Resultados do df_final")
print("- df_resumo_test2: Resultados do df_final_test_2")
print("- df_resumo_test3_geral: Resultados gerais do df_final_test_3")

print("\n✅ ANÁLISE COMPLETA FINALIZADA!")

In [ ]:
# ========================================
# ANÁLISE DE REGRESSÃO OLS - GENDER VS VALENCIA
# VERSÃO CORRIGIDA
# ========================================

# Imports necessários
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

print("📦 Bibliotecas importadas com sucesso!")

# ========================================
# CARREGANDO OS DATAFRAMES
# ========================================



print(f"✅ df_final carregado: {df_final.shape}")
print(f"✅ df_final_test_2 carregado: {df_final_test_2.shape}")
print(f"✅ df_final_test_3 carregado: {df_final_test_3.shape}")

# VERIFICANDO COLUNAS DO DF_FINAL_TEST_3
print("\n📋 Colunas do df_final_test_3:")
print(list(df_final_test_3.columns))
print("\n📋 Valores únicos de 'valencia' em df_final_test_3:")
print(df_final_test_3['valencia'].value_counts())

# ========================================
# FUNÇÕES AUXILIARES MELHORADAS
# ========================================

def run_regression_detailed(df, formula, description=""):
    """Roda uma regressão OLS e retorna TODOS os coeficientes"""
    try:
        model = ols(formula, data=df).fit()

        print(f"\n{'='*60}")
        print(f"📊 {description}")
        print(f"{'='*60}")
        print(f"R²: {model.rsquared:.4f}")
        print(f"N obs: {int(model.nobs)}")
        print("\n📈 TODOS OS COEFICIENTES:")

        # Criar dataframe com todos os coeficientes
        coef_df = pd.DataFrame({
            'Coeficiente': model.params.values,
            'P-value': model.pvalues.values,
            'Std Error': model.bse.values
        }, index=model.params.index)

        print(coef_df.to_string())

        # Pegar o beta do X especificamente
        beta_x = model.params.get('X', None)
        p_value_x = model.pvalues.get('X', None)

        return {
            'description': description,
            'beta_x': beta_x,
            'p_value_x': p_value_x,
            'r_squared': model.rsquared,
            'n_obs': int(model.nobs),
            'model': model,
            'all_coefs': coef_df
        }
    except Exception as e:
        print(f"❌ Erro na regressão {description}: {str(e)}")
        return None

def prepare_dummies(df):
    """Prepara variáveis dummy para modelo e idioma"""
    df_work = df.copy()

    # Dummy para idioma
    df_work['idioma_pt'] = (df_work['idioma'] == 'pt').astype(int)

    # Dummies para modelos - corrigindo para valores NaN
    modelos_unicos = [m for m in df_work['modelo'].unique() if pd.notna(m)]
    print(f"\n📋 Modelos encontrados (válidos): {list(modelos_unicos)}")

    for modelo in modelos_unicos:
        col_name = f"modelo_{modelo.replace('-', '_')}"
        df_work[col_name] = (df_work['modelo'] == modelo).astype(int)

    return df_work, modelos_unicos

# ========================================
# 1️⃣ ANÁLISE DF_FINAL
# ========================================

print("\n" + "="*80)
print("🎯 ANÁLISE DF_FINAL")
print("="*80)

# Preparando variáveis binárias
df_final_work = df_final.copy()

# Y = 1 se Male, 0 caso contrário
df_final_work['Y'] = (df_final_work['gender'].str.lower() == 'male').astype(int)

# X = 1 se valencia positiva
valencia_positivas = ['Positive', 'Positiva', 'positive', 'positiva']
df_final_work['X'] = df_final_work['valencia'].isin(valencia_positivas).astype(int)

print(f"\n📊 Distribuição de Y (Male=1): {df_final_work['Y'].value_counts().to_dict()}")
print(f"📊 Distribuição de X (Positiva=1): {df_final_work['X'].value_counts().to_dict()}")

# Preparando dummies
df_final_work, modelos_final = prepare_dummies(df_final_work)

# Lista para armazenar resultados
resultados_df_final = []

# 1. REGRESSÃO SIMPLES - TODOS OS IDIOMAS
result = run_regression_detailed(df_final_work, 'Y ~ X',
                        "DF_FINAL - Regressão Simples (Todos os idiomas)")
if result: resultados_df_final.append(result)

# 2. REGRESSÃO COMPLETA - TODOS OS IDIOMAS
modelo_cols = [f"modelo_{m.replace('-', '_')}" for m in modelos_final[1:]]
formula_completa = f"Y ~ X + idioma_pt + {' + '.join(modelo_cols)}"
result = run_regression_detailed(df_final_work, formula_completa,
                        "DF_FINAL - Regressão Completa (Todos os idiomas)")
if result: resultados_df_final.append(result)

# 3. REGRESSÃO SIMPLES - APENAS PORTUGUÊS
df_pt = df_final_work[df_final_work['idioma'] == 'pt'].copy()
result = run_regression_detailed(df_pt, 'Y ~ X',
                        "DF_FINAL - Regressão Simples (Apenas Português)")
if result: resultados_df_final.append(result)

# 4. REGRESSÃO COMPLETA - APENAS PORTUGUÊS
if len(modelo_cols) > 0:
    result = run_regression_detailed(df_pt, f"Y ~ X + {' + '.join(modelo_cols)}",
                            "DF_FINAL - Regressão Completa (Apenas Português)")
    if result: resultados_df_final.append(result)

# 5. REGRESSÃO SIMPLES - APENAS INGLÊS
df_en = df_final_work[df_final_work['idioma'] == 'en'].copy()
result = run_regression_detailed(df_en, 'Y ~ X',
                        "DF_FINAL - Regressão Simples (Apenas Inglês)")
if result: resultados_df_final.append(result)

# 6. REGRESSÃO COMPLETA - APENAS INGLÊS
if len(modelo_cols) > 0:
    result = run_regression_detailed(df_en, f"Y ~ X + {' + '.join(modelo_cols)}",
                            "DF_FINAL - Regressão Completa (Apenas Inglês)")
    if result: resultados_df_final.append(result)

# ========================================
# 2️⃣ ANÁLISE DF_FINAL_TEST_2
# ========================================

print("\n" + "="*80)
print("🎯 ANÁLISE DF_FINAL_TEST_2")
print("="*80)

# Preparando o dataframe
df_test2_work = df_final_test_2.copy()

# Normalizando valencia
print("\n🔧 Normalizando coluna 'valencia'...")
print(f"Valores únicos antes: {df_test2_work['valencia'].unique()}")

# Mapeamento para normalizar
valencia_map = {
    'positive': 'positive',
    'Positive': 'positive',
    'positiva': 'positive',
    'Positiva': 'positive',
    'positivo': 'positive',
    'Positivo': 'positive',
    'negative': 'negative',
    'Negative': 'negative',
    'negativa': 'negative',
    'Negativa': 'negative',
    'negativo': 'negative',
    'Negativo': 'negative'
}

df_test2_work['valencia'] = df_test2_work['valencia'].map(valencia_map).fillna(df_test2_work['valencia'])
print(f"Valores únicos depois: {df_test2_work['valencia'].unique()}")

# Criando variáveis binárias
df_test2_work['Y'] = (df_test2_work['gender'].str.lower() == 'male').astype(int)
df_test2_work['X'] = (df_test2_work['valencia'] == 'positive').astype(int)

print(f"\n📊 Distribuição de Y (Male=1): {df_test2_work['Y'].value_counts().to_dict()}")
print(f"📊 Distribuição de X (Positiva=1): {df_test2_work['X'].value_counts().to_dict()}")

# Preparando dummies
df_test2_work, modelos_test2 = prepare_dummies(df_test2_work)

# Lista para armazenar resultados
resultados_df_test2 = []

# 1. REGRESSÃO SIMPLES - TODOS OS IDIOMAS
result = run_regression_detailed(df_test2_work, 'Y ~ X',
                        "DF_TEST2 - Regressão Simples (Todos os idiomas)")
if result: resultados_df_test2.append(result)

# 2. REGRESSÃO COMPLETA - TODOS OS IDIOMAS
modelo_cols = [f"modelo_{m.replace('-', '_')}" for m in modelos_test2[1:]]
formula_completa = f"Y ~ X + idioma_pt + {' + '.join(modelo_cols)}"
result = run_regression_detailed(df_test2_work, formula_completa,
                        "DF_TEST2 - Regressão Completa (Todos os idiomas)")
if result: resultados_df_test2.append(result)

# 3. REGRESSÃO SIMPLES - APENAS PORTUGUÊS
df_pt = df_test2_work[df_test2_work['idioma'] == 'pt'].copy()
result = run_regression_detailed(df_pt, 'Y ~ X',
                        "DF_TEST2 - Regressão Simples (Apenas Português)")
if result: resultados_df_test2.append(result)

# 4. REGRESSÃO COMPLETA - APENAS PORTUGUÊS
if len(modelo_cols) > 0:
    result = run_regression_detailed(df_pt, f"Y ~ X + {' + '.join(modelo_cols)}",
                            "DF_TEST2 - Regressão Completa (Apenas Português)")
    if result: resultados_df_test2.append(result)

# 5. REGRESSÃO SIMPLES - APENAS INGLÊS
df_en = df_test2_work[df_test2_work['idioma'] == 'en'].copy()
result = run_regression_detailed(df_en, 'Y ~ X',
                        "DF_TEST2 - Regressão Simples (Apenas Inglês)")
if result: resultados_df_test2.append(result)

# 6. REGRESSÃO COMPLETA - APENAS INGLÊS
if len(modelo_cols) > 0:
    result = run_regression_detailed(df_en, f"Y ~ X + {' + '.join(modelo_cols)}",
                            "DF_TEST2 - Regressão Completa (Apenas Inglês)")
    if result: resultados_df_test2.append(result)

# ========================================
# 3️⃣ ANÁLISE DF_FINAL_TEST_3
# ========================================

print("\n" + "="*80)
print("🎯 ANÁLISE DF_FINAL_TEST_3")
print("="*80)

# Preparando o dataframe
df_test3_work = df_final_test_3.copy()

# Verificando a estrutura
print("\n📋 Investigando df_final_test_3:")
print("Colunas:", list(df_test3_work.columns))
print("\nValores únicos em 'valencia':")
print(df_test3_work['valencia'].value_counts())

# SE a coluna 'caracteristica' existir, usá-la
if 'caracteristica' in df_test3_work.columns:
    print("\n📋 Valores únicos em 'caracteristica':")
    print(df_test3_work['caracteristica'].value_counts())

# Criando Y
df_test3_work['Y'] = (df_test3_work['gender'].str.lower() == 'male').astype(int)

# CORREÇÃO: Verificar melhor como identificar valência positiva
# Talvez a coluna tenha valores diferentes
print("\n🔍 Analisando melhor a coluna valencia:")
print(df_test3_work['valencia'].value_counts())

# Se todos os valores são negativos, pode ser que a análise seja diferente
# Vamos usar a coluna 'caracteristica' se ela existir
if 'caracteristica' in df_test3_work.columns:
    caracteristicas = df_test3_work['caracteristica'].unique()
    print(f"\n📋 Características encontradas: {caracteristicas[:10]}")  # Primeiras 10

    # Para cada característica, fazer análise
    resultados_por_caracteristica = []

    # Preparando dummies para modelos
    df_test3_work, modelos_test3 = prepare_dummies(df_test3_work)

    for caract in caracteristicas[:20]:  # Limitando a 20 para teste
        if pd.isna(caract):
            continue

        # Criando X binário para esta característica
        df_test3_work['X_caract'] = (df_test3_work['caracteristica'] == caract).astype(int)

        # Verificar se há variância
        if df_test3_work['X_caract'].sum() < 10:
            continue

        # Regressão simples
        try:
            model_simple = ols('Y ~ X_caract', data=df_test3_work).fit()
            beta_simple = model_simple.params['X_caract']
            p_simple = model_simple.pvalues['X_caract']
            r2_simple = model_simple.rsquared

            resultados_por_caracteristica.append({
                'Característica': caract,
                'Tipo': 'Simples',
                'Beta': beta_simple,
                'P-value': p_simple,
                'R²': r2_simple,
                'N': int(model_simple.nobs)
            })

            print(f"✅ {caract}: Beta={beta_simple:.4f}, p={p_simple:.4f}")
        except Exception as e:
            print(f"❌ Erro com {caract}: {str(e)}")

        # Limpar coluna temporária
        df_test3_work.drop('X_caract', axis=1, inplace=True)

    # Criar DataFrame com resultados
    if resultados_por_caracteristica:
        df_resumo_caracteristicas = pd.DataFrame(resultados_por_caracteristica)
        print("\n📊 TOP 10 CARACTERÍSTICAS COM MAIOR BETA:")
        print(df_resumo_caracteristicas.nlargest(10, 'Beta')[['Característica', 'Beta', 'P-value', 'R²']].to_string())

        print("\n📊 TOP 10 CARACTERÍSTICAS COM MENOR BETA:")
        print(df_resumo_caracteristicas.nsmallest(10, 'Beta')[['Característica', 'Beta', 'P-value', 'R²']].to_string())
else:
    print("\n⚠️ Coluna 'caracteristica' não encontrada no df_final_test_3")
    print("Tentando análise alternativa com valencia...")

    # Criar X baseado em alguma lógica alternativa
    # Por exemplo, se há algum padrão nos valores de valencia
    df_test3_work['X'] = 0  # Placeholder

# ========================================
# RESUMO CONSOLIDADO COM TODOS OS COEFICIENTES
# ========================================

print("\n" + "="*80)
print("📊 RESUMO FINAL - COEFICIENTES DAS DUMMIES")
print("="*80)

# Para cada modelo com regressão completa, mostrar todos os coeficientes
for result in resultados_df_final + resultados_df_test2:
    if result and 'Completa' in result['description']:
        print(f"\n📈 {result['description']}:")
        print(result['all_coefs'].to_string())

print("\n✅ ANÁLISE COMPLETA FINALIZADA!")

In [ ]:
import matplotlib.pyplot as plt

# ---------------------------------------------------
# Helper: criar figura com título, texto e tabela
# ---------------------------------------------------
def criar_slide_tabela(
    titulo,
    linhas_texto,
    col_labels,
    row_labels,
    cell_values,
    filename,
    figsize=(10, 6),
):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("white")
    ax.set_axis_off()

    # Título
    ax.text(
        0.02, 0.95, titulo,
        transform=ax.transAxes,
        fontsize=18,
        fontweight="bold",
        va="top",
        ha="left",
    )

    # Linhas de explicação (abaixo do título)
    y = 0.88
    for linha in linhas_texto:
        ax.text(
            0.02, y, linha,
            transform=ax.transAxes,
            fontsize=11,
            va="top",
            ha="left",
        )
        y -= 0.045

    # Tabela
    table_y = 0.05
    table = ax.table(
        cellText=cell_values,
        rowLabels=row_labels,
        colLabels=col_labels,
        cellLoc="center",
        loc="bottom",
        bbox=[0.02, table_y, 0.96, 0.55],  # [x, y, width, height]
    )

    # Estilo da tabela: cabeçalho mais escuro, corpo cinza claro
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("black")
        cell.set_linewidth(0.4)
        if row == 0:
            cell.set_facecolor("#e0e0e0")  # header
            cell.set_fontsize(11)
            cell.set_text_props(weight="bold")
        else:
            cell.set_facecolor("#f7f7f7")
            cell.set_fontsize(10)

    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close(fig)


# ===================================================
# 1) Teste 1 – FEEDBACK (df_final) – Regressão simples
# ===================================================
titulo_t1_simples = "Resultados Teste FEEDBACK – Regressão simples"
texto_t1_simples = [
    r"$Y_i = 1$ se o protagonista é homem; $0$ caso contrário.",
    r"$X_i = 1$ se o feedback recebido é positivo; $0$ se é negativo.",
    r"$\beta_1 < 0$  $\Rightarrow$  feedback positivo reduz a chance de protagonista masculino.",
]

col_labels_t1 = [r"$\beta_1$", "p-valor", r"$R^2$", "N"]
row_labels_t1 = ["Todos os idiomas", "Português", "Inglês"]
cell_values_t1_simples = [
    ["-0.6268", "< 0.001", "0.393", "2031"],
    ["-0.5778", "< 0.001", "0.363", "1028"],
    ["-0.6767", "< 0.001", "0.490", "1003"],
]

criar_slide_tabela(
    titulo_t1_simples,
    texto_t1_simples,
    col_labels_t1,
    row_labels_t1,
    cell_values_t1_simples,
    "slide_t1_feedback_simples.png",
)

# ===================================================
# 2) Teste 1 – FEEDBACK (df_final) – Modelo completo
# ===================================================
titulo_t1_comp = "Teste FEEDBACK – Modelo com idioma e modelo"
texto_t1_comp = [
    r"$Y_i = \alpha + \beta_1 X_i + \gamma_1 PT_i + \delta_1 Nano_i + \delta_2 GPT4oMini_i + \varepsilon_i$.",
    r"$Y_i$: protagonista homem;  $X_i$: feedback positivo.",
    r"$PT_i = 1$ se a história está em português (0 = inglês).",
    r"Dummies de modelo: $Nano_i$ e $GPT4oMini_i$ (base: gpt-5-mini).",
]

cell_values_t1_comp = [
    ["-0.6259", "< 0.001", "0.504", "2031"],
    ["-0.5741", "< 0.001", "0.156", "479"],
    ["-0.6768", "< 0.001", "0.510", "1003"],
]

criar_slide_tabela(
    titulo_t1_comp,
    texto_t1_comp,
    col_labels_t1,
    row_labels_t1,
    cell_values_t1_comp,
    "slide_t1_feedback_completo.png",
)

# ===================================================
# 3) Teste 2 – QUALIDADE (df_final_test_2) – simples
# ===================================================
titulo_t2_simples = "Resultados Teste QUALIDADE – Regressão simples"
texto_t2_simples = [
    r"$Y_i = 1$ se o protagonista é homem; $0$ caso contrário.",
    r"$X_i = 1$ se a qualidade atribuída é positiva; $0$ se é negativa.",
    r"Textos normalizados: 'positive/Positive' e 'negative/Negative'.",
    r"$\beta_1 < 0$  $\Rightarrow$  qualidades positivas reduzem a chance de protagonista masculino.",
]

row_labels_t2 = ["Todos os idiomas", "Português", "Inglês"]
cell_values_t2_simples = [
    ["-0.1977", "< 0.001", "0.039", "979"],
    ["-0.1561", "< 0.001", "0.055", "479"],
    ["-0.2194", "< 0.001", "0.071", "498"],
]

criar_slide_tabela(
    titulo_t2_simples,
    texto_t2_simples,
    col_labels_t1,
    row_labels_t2,
    cell_values_t2_simples,
    "slide_t2_qualidade_simples.png",
)

# ===================================================
# 4) Teste 2 – QUALIDADE – Modelo completo
# ===================================================
titulo_t2_comp = "Teste QUALIDADE – Modelo com idioma e modelo"
texto_t2_comp = [
    r"$Y_i = \alpha + \beta_1 X_i + \gamma_1 PT_i + \delta_1 Nano_i + \delta_2 GPT4oMini_i + \varepsilon_i$.",
    r"$X_i$: qualidade positiva (=1) vs negativa (=0).",
    r"Dummies de idioma e modelo definidos como no Teste FEEDBACK.",
]

cell_values_t2_comp = [
    ["-0.1884", "< 0.001", "0.470", "979"],
    ["-0.1559", "< 0.001", "0.156", "479"],
    ["-0.2212", "< 0.001", "0.135", "498"],
]

criar_slide_tabela(
    titulo_t2_comp,
    texto_t2_comp,
    col_labels_t1,
    row_labels_t2,
    cell_values_t2_comp,
    "slide_t2_qualidade_completo.png",
)

# ===================================================
# 5) Teste 3 – PROFISSÃO – Mais masculinizadas
# ===================================================
titulo_t3_masc = "Teste PROFISSÃO – Profissões mais masculinizadas"
texto_t3_masc = [
    r"Para cada profissão $p$: $Y_i = \alpha + \beta_p D_{ip} + \varepsilon_i$.",
    r"$Y_i = 1$ se o protagonista é homem.",
    r"$D_{ip} = 1$ se a história $i$ tem a profissão $p$; $0$ caso contrário.",
    r"$\beta_p > 0$  $\Rightarrow$  profissão mais associada a protagonistas homens.",
]

col_labels_t3 = [r"$\beta_p$", "p-valor", r"$R^2$", "N"]
row_labels_t3_masc = [
    "blue collar",
    "funcionário(a) braçal",
    "white collar",
    "plane pilot",
    "manager",
]
cell_values_t3_masc = [
    ["0.8478", "< 0.001", "0.248", "5947"],
    ["0.6749", "< 0.001", "0.158", "5947"],
    ["0.1471", "< 0.001", "0.007", "5947"],
    ["0.1000", "< 0.001", "0.003", "5947"],
    ["0.0559", "0.011",  "0.001", "5947"],
]

criar_slide_tabela(
    titulo_t3_masc,
    texto_t3_masc,
    col_labels_t3,
    row_labels_t3_masc,
    cell_values_t3_masc,
    "slide_t3_profissao_masculinizadas.png",
)

# ===================================================
# 6) Teste 3 – PROFISSÃO – Mais feminizadas
# ===================================================
titulo_t3_fem = "Teste PROFISSÃO – Profissões mais feminizadas"
texto_t3_fem = [
    r"Mesma especificação: $Y_i = \alpha + \beta_p D_{ip} + \varepsilon_i$.",
    r"$\beta_p < 0$  $\Rightarrow$  profissão mais associada a protagonistas mulheres.",
]

row_labels_t3_fem = [
    "secretary",
    "nurse",
    "professor(a) do ensino básico",
    "secretario(a)",
    "physician",
]
cell_values_t3_fem = [
    ["-0.1730", "< 0.001", "0.010", "5947"],
    ["-0.1729", "< 0.001", "0.010", "5947"],
    ["-0.1694", "< 0.001", "0.010", "5947"],
    ["-0.1694", "< 0.001", "0.010", "5947"],
    ["-0.1694", "< 0.001", "0.010", "5947"],
]

criar_slide_tabela(
    titulo_t3_fem,
    texto_t3_fem,
    col_labels_t3,
    row_labels_t3_fem,
    cell_values_t3_fem,
    "slide_t3_profissao_feminizadas.png",
)

print("Slides gerados!")

In [ ]:
# ===================================================
# 1) Teste 1 – FEEDBACK (df_final) – Regressão simples
# ===================================================
titulo_t1_simples = "Resultados Teste FEEDBACK – Regressão simples"
texto_t1_simples = [
    r"$Y_i = 1$ se o protagonista é homem; $0$ caso contrário.",
    r"$X_i = 1$ se o feedback recebido é positivo; $0$ se é negativo.",
    r"$\beta_1 < 0 \Rightarrow$ feedback positivo reduz a chance de protagonista masculino.",
]

col_labels_t1 = [r"$\beta_1$", "p-valor", r"$R^2$", "N"]
row_labels_t1 = ["Todos os idiomas", "Português", "Inglês"]
cell_values_t1_simples = [
    ["-0.6268", "< 0.001", "0.393", "2031"],
    ["-0.5778", "< 0.001", "0.363", "1028"],
    ["-0.6767", "< 0.001", "0.490", "1003"],
]

criar_slide_tabela(
    titulo_t1_simples,
    texto_t1_simples,
    col_labels_t1,
    row_labels_t1,
    cell_values_t1_simples,
    "slide_t1_feedback_simples.png",
)

# ===================================================
# 2) Teste 1 – FEEDBACK (df_final) – Modelo completo
# ===================================================
titulo_t1_comp = "Teste FEEDBACK – Modelo com idioma e modelo"
texto_t1_comp = [
    r"$Y_i = \alpha + \beta_1 X_i + \gamma_1 PT_i + \delta_1 Nano_i + \delta_2 GPT4oMini_i + \varepsilon_i.$",
    r"$Y_i$: protagonista homem;  $X_i$: feedback positivo.",
    r"$PT_i = 1$ se a história está em português (0 = inglês).",
    r"Dummies de modelo: $Nano_i$ e $GPT4oMini_i$ (base: gpt-5-mini).",
]

cell_values_t1_comp = [
    ["-0.6259", "< 0.001", "0.504", "2031"],
    ["-0.5741", "< 0.001", "0.446", "1028"],
    ["-0.6768", "< 0.001", "0.510", "1003"],
]

criar_slide_tabela(
    titulo_t1_comp,
    texto_t1_comp,
    col_labels_t1,
    row_labels_t1,
    cell_values_t1_comp,
    "slide_t1_feedback_completo.png",
)

# ===================================================
# 3) Teste 2 – QUALIDADE (df_final_test_2) – simples
# ===================================================
titulo_t2_simples = "Resultados Teste QUALIDADE – Regressão simples"
texto_t2_simples = [
    r"$Y_i = 1$ se o protagonista é homem; $0$ caso contrário.",
    r"$X_i = 1$ se a qualidade atribuída é positiva; $0$ se é negativa.",
    r"Textos normalizados: 'positive/Positive' e 'negative/Negative'.",
    r"$\beta_1 < 0 \Rightarrow$ qualidades positivas reduzem a chance de protagonista masculino.",
]

row_labels_t2 = ["Todos os idiomas", "Português", "Inglês"]
cell_values_t2_simples = [
    ["-0.1977", "< 0.001", "0.039", "979"],
    ["-0.1561", "< 0.001", "0.055", "479"],
    ["-0.2194", "< 0.001", "0.071", "498"],
]

criar_slide_tabela(
    titulo_t2_simples,
    texto_t2_simples,
    col_labels_t1,
    row_labels_t2,
    cell_values_t2_simples,
    "slide_t2_qualidade_simples.png",
)

# ===================================================
# 4) Teste 2 – QUALIDADE – Modelo completo
# ===================================================
titulo_t2_comp = "Teste QUALIDADE – Modelo com idioma e modelo"
texto_t2_comp = [
    r"$Y_i = \alpha + \beta_1 X_i + \gamma_1 PT_i + \delta_1 Nano_i + \delta_2 GPT4oMini_i + \varepsilon_i.$",
    r"$X_i$: qualidade positiva (=1) vs negativa (=0).",
    r"Dummies de idioma e modelo definidos como no Teste FEEDBACK.",
]

cell_values_t2_comp = [
    ["-0.1884", "< 0.001", "0.470", "979"],
    ["-0.1559", "< 0.001", "0.156", "479"],
    ["-0.2212", "< 0.001", "0.135", "498"],
]

criar_slide_tabela(
    titulo_t2_comp,
    texto_t2_comp,
    col_labels_t1,
    row_labels_t2,
    cell_values_t2_comp,
    "slide_t2_qualidade_completo.png",
)

# ===================================================
# 5) Teste 3 – PROFISSÃO – Mais masculinizadas
# ===================================================
titulo_t3_masc = "Teste PROFISSÃO – Profissões mais masculinizadas"
texto_t3_masc = [
    r"Para cada profissão $p$: $Y_i = \alpha + \beta_p D_{ip} + \varepsilon_i.$",
    r"$Y_i = 1$ se o protagonista é homem.",
    r"$D_{ip} = 1$ se a história $i$ tem a profissão $p$; $0$ caso contrário.",
    r"$\beta_p > 0 \Rightarrow$ profissão mais associada a protagonistas homens.",
]

col_labels_t3 = [r"$\beta_p$", "p-valor", r"$R^2$", "N"]
row_labels_t3_masc = [
    "blue collar",
    "funcionário(a) braçal",
    "white collar",
    "plane pilot",
    "manager",
]
cell_values_t3_masc = [
    ["0.8478", "< 0.001", "0.248", "5947"],
    ["0.6749", "< 0.001", "0.158", "5947"],
    ["0.1471", "< 0.001", "0.007", "5947"],
    ["0.1000", "< 0.001", "0.003", "5947"],
    ["0.0559", "0.011",  "0.001", "5947"],
]

criar_slide_tabela(
    titulo_t3_masc,
    texto_t3_masc,
    col_labels_t3,
    row_labels_t3_masc,
    cell_values_t3_masc,
    "slide_t3_profissao_masculinizadas.png",
)

# ===================================================
# 6) Teste 3 – PROFISSÃO – Mais feminizadas
# ===================================================
titulo_t3_fem = "Teste PROFISSÃO – Profissões mais feminizadas"
texto_t3_fem = [
    r"Mesma especificação: $Y_i = \alpha + \beta_p D_{ip} + \varepsilon_i.$",
    r"$\beta_p < 0 \Rightarrow$ profissão mais associada a protagonistas mulheres.",
]

row_labels_t3_fem = [
    "secretary",
    "nurse",
    "professor(a) do ensino básico",
    "secretario(a)",
    "physician",
]
cell_values_t3_fem = [
    ["-0.1730", "< 0.001", "0.010", "5947"],
    ["-0.1729", "< 0.001", "0.010", "5947"],
    ["-0.1694", "< 0.001", "0.010", "5947"],
    ["-0.1694", "< 0.001", "0.010", "5947"],
    ["-0.1694", "< 0.001", "0.010", "5947"],
]

criar_slide_tabela(
    titulo_t3_fem,
    texto_t3_fem,
    col_labels_t3,
    row_labels_t3_fem,
    cell_values_t3_fem,
    "slide_t3_profissao_feminizadas.png",
)

print("Slides gerados com sucesso!")


In [ ]:
resultados_df_final

# Medindo Correlação

In [ ]:
import pandas as pd
import numpy as np

def calculate_gender_valencia_correlations(df,
                                          gender_col='gender',
                                          valencia_col='valencia',
                                          model_col='modelo',
                                          language_col='idioma',
                                          neutral_char_col='caracteristica_neutra',
                                          debug=True):
    """
    Calculate correlations between gender and valence at multiple levels of granularity.

    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe containing the relevant columns
    gender_col : str
        Column name for gender (default: 'gender')
    valencia_col : str
        Column name for valence (default: 'valencia')
    model_col : str
        Column name for model (default: 'modelo')
    language_col : str
        Column name for language (default: 'idioma')
    neutral_char_col : str
        Column name for neutral characteristic (default: 'caracteristica_neutra')
    debug : bool
        If True, print debug information (default: True)

    Returns:
    --------
    dict
        Dictionary containing correlation results at different levels
    """

    # Criar cópia para não modificar o dataframe original
    df_work = df.copy()

    if debug:
        print("\n" + "="*80)
        print("DEBUG: ANÁLISE INICIAL DOS DADOS")
        print("="*80)
        print(f"\nTotal de linhas: {len(df_work)}")
        print(f"\nColunas disponíveis: {df_work.columns.tolist()}")
        print(f"\nValores únicos em '{gender_col}': {df_work[gender_col].unique()}")
        print(f"Contagem: {df_work[gender_col].value_counts().to_dict()}")
        print(f"\nValores únicos em '{valencia_col}': {df_work[valencia_col].unique()}")
        print(f"Contagem: {df_work[valencia_col].value_counts().to_dict()}")
        print(f"\nModelos: {df_work[model_col].unique()}")
        print(f"Idiomas: {df_work[language_col].unique()}")
        print(f"Características: {df_work[neutral_char_col].unique()}")

    # Converter para variáveis binárias
    df_work['is_male'] = df_work[gender_col].apply(lambda x: 1 if x == 'Male' else 0)
    df_work['is_positive'] = df_work[valencia_col].apply(lambda x: 1 if x == 'Positive' else 0)

    if debug:
        print(f"\n--- Após conversão binária ---")
        print(f"is_male - valores únicos: {df_work['is_male'].unique()}")
        print(f"is_male - contagem: {df_work['is_male'].value_counts().to_dict()}")
        print(f"is_positive - valores únicos: {df_work['is_positive'].unique()}")
        print(f"is_positive - contagem: {df_work['is_positive'].value_counts().to_dict()}")

        # Verificar se há variância
        print(f"\nis_male - variância: {df_work['is_male'].var():.4f}")
        print(f"is_positive - variância: {df_work['is_positive'].var():.4f}")

        if df_work['is_male'].var() == 0:
            print("\n⚠️  PROBLEMA: is_male não tem variância (todos os valores são iguais)!")
        if df_work['is_positive'].var() == 0:
            print("\n⚠️  PROBLEMA: is_positive não tem variância (todos os valores são iguais)!")

    results = {}

    # 1. Correlação por modelo e característica neutra
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por modelo e característica")
        print("="*80)

    by_model_char = []

    for (model, char), group in df_work.groupby([model_col, neutral_char_col]):
        n_obs = len(group)
        var_male = group['is_male'].var()
        var_positive = group['is_positive'].var()

        if debug:
            print(f"\n{model} | {char}")
            print(f"  N observações: {n_obs}")
            print(f"  is_male: {group['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
            print(f"  is_positive: {group['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

        if n_obs >= 2 and var_male > 0 and var_positive > 0:
            corr = group[['is_male', 'is_positive']].corr().iloc[0, 1]
            if debug:
                print(f"  ✓ Correlação: {corr:.3f}")
        else:
            corr = np.nan
            if debug:
                if var_male == 0:
                    print(f"  ✗ Sem variância em is_male (todos {group['is_male'].iloc[0]})")
                if var_positive == 0:
                    print(f"  ✗ Sem variância em is_positive (todos {group['is_positive'].iloc[0]})")

        by_model_char.append({
            'modelo': model,
            'caracteristica_neutra': char,
            'idioma': group[language_col].iloc[0],
            'correlation': corr,
            'n_observations': n_obs,
            'var_male': var_male,
            'var_positive': var_positive
        })

    results['by_model_and_characteristic'] = pd.DataFrame(by_model_char)

    # 2. Correlação por modelo e idioma
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por modelo e idioma")
        print("="*80)

    by_model_lang = []

    for (model, lang), group in df_work.groupby([model_col, language_col]):
        n_obs = len(group)
        var_male = group['is_male'].var()
        var_positive = group['is_positive'].var()

        if debug:
            print(f"\n{model} | {lang}")
            print(f"  N observações: {n_obs}")
            print(f"  is_male: {group['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
            print(f"  is_positive: {group['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

        if n_obs >= 2 and var_male > 0 and var_positive > 0:
            corr = group[['is_male', 'is_positive']].corr().iloc[0, 1]
            if debug:
                print(f"  ✓ Correlação: {corr:.3f}")
        else:
            corr = np.nan
            if debug:
                if var_male == 0:
                    print(f"  ✗ Sem variância em is_male")
                if var_positive == 0:
                    print(f"  ✗ Sem variância em is_positive")

        by_model_lang.append({
            'modelo': model,
            'idioma': lang,
            'correlation': corr,
            'n_observations': n_obs,
            'n_characteristics': group[neutral_char_col].nunique(),
            'var_male': var_male,
            'var_positive': var_positive
        })

    results['by_model_and_language'] = pd.DataFrame(by_model_lang)

    # 3. Correlação por idioma
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por idioma")
        print("="*80)

    by_lang = []

    for lang, group in df_work.groupby(language_col):
        n_obs = len(group)
        var_male = group['is_male'].var()
        var_positive = group['is_positive'].var()

        if debug:
            print(f"\n{lang}")
            print(f"  N observações: {n_obs}")
            print(f"  is_male: {group['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
            print(f"  is_positive: {group['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

        if n_obs >= 2 and var_male > 0 and var_positive > 0:
            corr = group[['is_male', 'is_positive']].corr().iloc[0, 1]
            if debug:
                print(f"  ✓ Correlação: {corr:.3f}")
        else:
            corr = np.nan
            if debug:
                if var_male == 0:
                    print(f"  ✗ Sem variância em is_male")
                if var_positive == 0:
                    print(f"  ✗ Sem variância em is_positive")

        by_lang.append({
            'idioma': lang,
            'correlation': corr,
            'n_observations': n_obs,
            'n_models': group[model_col].nunique(),
            'n_characteristics': group[neutral_char_col].nunique(),
            'var_male': var_male,
            'var_positive': var_positive
        })

    results['by_language'] = pd.DataFrame(by_lang)

    # 4. Correlação por modelo (agregando idiomas e características)
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por modelo")
        print("="*80)

    by_model = []
    for model, group in df_work.groupby(model_col):
        n_obs = len(group)
        var_male = group['is_male'].var()
        var_positive = group['is_positive'].var()

        if debug:
            print(f"\n{model}")
            print(f"  N observações: {n_obs}")
            print(f"  is_male: {group['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
            print(f"  is_positive: {group['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

        if n_obs >= 2 and var_male > 0 and var_positive > 0:
            corr = group[['is_male', 'is_positive']].corr().iloc[0, 1]
            if debug:
                print(f"  ✓ Correlação: {corr:.3f}")
        else:
            corr = np.nan

        by_model.append({
            'modelo': model,
            'correlation': corr,
            'n_observations': n_obs,
            'n_languages': group[language_col].nunique(),
            'n_characteristics': group[neutral_char_col].nunique()
        })

    results['by_model'] = pd.DataFrame(by_model)
    # 5. Correlação por característica (agregando modelos)
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlações por característica")
        print("="*80)

    by_char = []
    for char, group in df_work.groupby(neutral_char_col):
        n_obs = len(group)
        var_male = group['is_male'].var()
        var_positive = group['is_positive'].var()

        if debug:
            print(f"\n{char}")
            print(f"  N observações: {n_obs}")
            print(f"  is_male: {group['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
            print(f"  is_positive: {group['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

        if n_obs >= 2 and var_male > 0 and var_positive > 0:
            corr = group[['is_male', 'is_positive']].corr().iloc[0, 1]
            if debug:
                print(f"  ✓ Correlação: {corr:.3f}")
        else:
            corr = np.nan

        by_char.append({
            'caracteristica_neutra': char,
            'idioma': group[language_col].iloc[0],
            'correlation': corr,
            'n_observations': n_obs,
            'n_models': group[model_col].nunique()
        })

    results['by_characteristic'] = pd.DataFrame(by_char)
    # 6. Correlação geral
    if debug:
        print("\n" + "="*80)
        print("DEBUG: Correlação geral")
        print("="*80)

    n_obs = len(df_work)
    var_male = df_work['is_male'].var()
    var_positive = df_work['is_positive'].var()

    if debug:
        print(f"N observações: {n_obs}")
        print(f"is_male: {df_work['is_male'].value_counts().to_dict()} | var={var_male:.4f}")
        print(f"is_positive: {df_work['is_positive'].value_counts().to_dict()} | var={var_positive:.4f}")

    if n_obs >= 2 and var_male > 0 and var_positive > 0:
        overall_corr = df_work[['is_male', 'is_positive']].corr().iloc[0, 1]
        if debug:
            print(f"✓ Correlação: {overall_corr:.3f}")
    else:
        overall_corr = np.nan
        if debug:
            if var_male == 0:
                print(f"✗ Sem variância em is_male")
            if var_positive == 0:
                print(f"✗ Sem variância em is_positive")

    results['overall'] = {
        'correlation': overall_corr,
        'n_observations': n_obs,
        'n_models': df_work[model_col].nunique(),
        'n_languages': df_work[language_col].nunique(),
        'n_characteristics': df_work[neutral_char_col].nunique(),
        'var_male': var_male,
        'var_positive': var_positive
    }

    return results


# Uso com debug ativado:
results = calculate_gender_valencia_correlations(df_final, debug=True)

# Depois de identificar o problema, pode rodar sem debug:
# results = calculate_gender_valencia_correlations(df_final, debug=False)

In [ ]:
def print_correlation_results(results):
    """
    Print bonito dos resultados de correlação.
    """
    print("\n" + "="*80)
    print("ANÁLISE DE CORRELAÇÃO: GÊNERO × VALÊNCIA".center(80))
    print("="*80)

    # 1. Por modelo e característica
    print("\n┌─ CORRELAÇÕES POR MODELO E CARACTERÍSTICA")
    print("│")
    if not results['by_model_and_characteristic'].empty:
        df_display = results['by_model_and_characteristic'].sort_values(
            ['modelo', 'idioma', 'caracteristica_neutra']
        )

        current_model = None
        for _, row in df_display.iterrows():
            if current_model != row['modelo']:
                if current_model is not None:
                    print("│")
                current_model = row['modelo']
                print(f"├── {row['modelo']}")

            corr_str = f"{row['correlation']:+.3f}" if not pd.isna(row['correlation']) else "  N/A"
            print(f"│   └─ [{row['idioma']}] {row['caracteristica_neutra']:20s} │ r = {corr_str} │ n={row['n_observations']:3d}")
    else:
        print("│   (sem dados suficientes)")

    # 2. Por modelo e idioma
    print("\n├─ CORRELAÇÕES POR MODELO E IDIOMA")
    print("│")
    if not results['by_model_and_language'].empty:
        df_display = results['by_model_and_language'].sort_values(['modelo', 'idioma'])

        current_model = None
        for _, row in df_display.iterrows():
            if current_model != row['modelo']:
                if current_model is not None:
                    print("│")
                current_model = row['modelo']
                print(f"├── {row['modelo']}")

            corr_str = f"{row['correlation']:+.3f}" if not pd.isna(row['correlation']) else "  N/A"
            print(f"│   └─ [{row['idioma']}] r = {corr_str} │ n={row['n_observations']:3d} │ {row['n_characteristics']} características")
    else:
        print("│   (sem dados suficientes)")
# 3. Por modelo
    print("\n├─ CORRELAÇÕES POR MODELO (agregando idiomas e características)")
    print("│")
    if not results['by_model'].empty:
        df_display = results['by_model'].sort_values('modelo')
        for _, row in df_display.iterrows():
            corr_str = f"{row['correlation']:+.3f}" if not pd.isna(row['correlation']) else "  N/A"
            print(f"│   └─ {row['modelo']:15s} │ r = {corr_str} │ n={row['n_observations']:3d} │ {row['n_languages']} idiomas │ {row['n_characteristics']} características")

    # 4. Por característica
    print("\n├─ CORRELAÇÕES POR CARACTERÍSTICA (agregando modelos)")
    print("│")
    if not results['by_characteristic'].empty:
        df_display = results['by_characteristic'].sort_values(['idioma', 'caracteristica_neutra'])

        current_lang = None
        for _, row in df_display.iterrows():
            if current_lang != row['idioma']:
                if current_lang is not None:
                    print("│")
                current_lang = row['idioma']
                print(f"├── [{row['idioma']}]")

            corr_str = f"{row['correlation']:+.3f}" if not pd.isna(row['correlation']) else "  N/A"
            print(f"│   └─ {row['caracteristica_neutra']:20s} │ r = {corr_str} │ n={row['n_observations']:3d} │ {row['n_models']} modelos")
    # 3. Por idioma
    print("\n├─ CORRELAÇÕES POR IDIOMA")
    print("│")
    if not results['by_language'].empty:
        df_display = results['by_language'].sort_values('idioma')
        for _, row in df_display.iterrows():
            corr_str = f"{row['correlation']:+.3f}" if not pd.isna(row['correlation']) else "  N/A"
            print(f"│   └─ [{row['idioma']}] r = {corr_str} │ n={row['n_observations']:4d} │ {row['n_models']} modelos │ {row['n_characteristics']} características")
    else:
        print("│   (sem dados suficientes)")

    # 4. Geral
    print("\n└─ CORRELAÇÃO GERAL")
    if 'overall' in results:
        overall = results['overall']
        corr_str = f"{overall['correlation']:+.3f}" if not pd.isna(overall['correlation']) else "N/A"
        print(f"    r = {corr_str} │ n={overall['n_observations']} │ {overall['n_models']} modelos │ {overall['n_languages']} idiomas │ {overall['n_characteristics']} características")
    else:
        print("    (sem dados suficientes)")

    print("\n" + "="*80)
    print()


# Uso:
results = calculate_gender_valencia_correlations(df_final, debug=False)
print_correlation_results(results)


In [ ]:
results

In [ ]:
import pandas as pd

def analyze_gender_distribution(df, gender_col='gender'):
    """
    Analisa a distribuição de gêneros no dataframe.
    """
    print("="*80)
    print("DISTRIBUIÇÃO DE GÊNEROS".center(80))
    print("="*80)

    # Contagem absoluta
    counts = df[gender_col].value_counts()
    print("\nContagem absoluta:")
    for gender, count in counts.items():
        print(f"  {gender:10s}: {count:4d}")

    # Percentuais
    percentages = df[gender_col].value_counts(normalize=True) * 100
    print("\nPercentuais:")
    for gender, pct in percentages.items():
        print(f"  {gender:10s}: {pct:5.2f}%")

    print("\n" + "="*80 + "\n")

    return percentages


def analyze_gender_by_valence(df,
                               gender_col='gender',
                               valencia_col='valencia',
                               model_col='modelo',
                               language_col='idioma'):
    """
    Analisa a distribuição de gêneros separada por valência.
    Mostra análise geral, por modelo, e por idioma.
    """
    print("="*80)
    print("DISTRIBUIÇÃO DE GÊNEROS POR VALÊNCIA".center(80))
    print("="*80)

    # 1. ANÁLISE GERAL
    print("\n┌─ ANÁLISE GERAL")
    print("│")

    for valencia in df[valencia_col].unique():
        filtered = df[df[valencia_col] == valencia]
        percentages = filtered[gender_col].value_counts(normalize=True) * 100
        counts = filtered[gender_col].value_counts()

        print(f"├── Valência: {valencia}")
        print(f"│   Total: {len(filtered)} observações")
        print("│   Distribuição:")
        for gender in percentages.index:
            print(f"│     {gender:10s}: {percentages[gender]:5.2f}% (n={counts[gender]:3d})")
        print("│")

    # 2. ANÁLISE POR MODELO
    print("├─ ANÁLISE POR MODELO")
    print("│")

    for model in sorted(df[model_col].unique()):
        print(f"├── {model}")
        model_df = df[df[model_col] == model]

        for valencia in df[valencia_col].unique():
            filtered = model_df[model_df[valencia_col] == valencia]
            if len(filtered) > 0:
                percentages = filtered[gender_col].value_counts(normalize=True) * 100
                counts = filtered[gender_col].value_counts()

                print(f"│   └─ Valência: {valencia} (n={len(filtered)})")
                for gender in percentages.index:
                    print(f"│       {gender:10s}: {percentages[gender]:5.2f}% (n={counts[gender]:3d})")
        print("│")

    # 3. ANÁLISE POR IDIOMA
    print("├─ ANÁLISE POR IDIOMA")
    print("│")

    for lang in sorted(df[language_col].unique()):
        print(f"├── [{lang}]")
        lang_df = df[df[language_col] == lang]

        for valencia in df[valencia_col].unique():
            filtered = lang_df[lang_df[valencia_col] == valencia]
            if len(filtered) > 0:
                percentages = filtered[gender_col].value_counts(normalize=True) * 100
                counts = filtered[gender_col].value_counts()

                print(f"│   └─ Valência: {valencia} (n={len(filtered)})")
                for gender in percentages.index:
                    print(f"│       {gender:10s}: {percentages[gender]:5.2f}% (n={counts[gender]:3d})")
        print("│")

    # 4. COMPARAÇÃO DIRETA: Positive vs Negative
    print("└─ COMPARAÇÃO DIRETA: POSITIVE vs NEGATIVE")
    print()

    # Tabela comparativa
    print("    Gênero      │  Positive  │  Negative  │  Diferença")
    print("    " + "-"*60)

    positive = df[df[valencia_col] == 'Positive']
    negative = df[df[valencia_col] == 'Negative']

    pct_positive = positive[gender_col].value_counts(normalize=True) * 100
    pct_negative = negative[gender_col].value_counts(normalize=True) * 100

    all_genders = set(pct_positive.index) | set(pct_negative.index)

    for gender in sorted(all_genders):
        pos_val = pct_positive.get(gender, 0)
        neg_val = pct_negative.get(gender, 0)
        diff = pos_val - neg_val

        print(f"    {gender:10s}  │  {pos_val:5.2f}%    │  {neg_val:5.2f}%    │  {diff:+5.2f}%")

    print("\n" + "="*80 + "\n")


# USO:

# 1. Ver distribuição geral de gêneros
analyze_gender_distribution(df_fixed)

# 2. Ver distribuição de gêneros por valência (análise completa)
analyze_gender_by_valence(df_fixed)

In [ ]:
import re

# Função para limpar e padronizar os valores de gênero
def clean_gender(gender):
    if isinstance(gender, str):
        # Remover quebras de linha, espaços em excesso e aspas
        gender = gender.strip().replace("\n", "").replace("'", "")
        # Manter apenas os valores desejados: 'Female', 'Male', 'Other'
        if re.search(r'\bFemale\b', gender, re.IGNORECASE):
            return 'Female'
        elif re.search(r'\bMale\b', gender, re.IGNORECASE):
            return 'Male'
        elif re.search(r'\bOther\b', gender, re.IGNORECASE):
            return 'Other'
        else:
            return 'Other'  # ou você pode escolher descartar valores inválidos
    return gender

# Aplicar a função de limpeza na coluna 'Gênero'
df_enus4['Gênero'] = df_enus4['Gênero'].apply(clean_gender)

# Agora recalculando as porcentagens de gênero em df_enus4
percent_enus4 = df_enus4['Gênero'].value_counts(normalize=True) * 100





# Mapeando os valores de gênero em português para inglês
df_ptbr['Gênero'] = df_ptbr['Gênero'].map({'Mulher': 'Female', 'Homem': 'Male'})

# Calculando porcentagem de gêneros em df_ptbr
percent_ptbr = df_ptbr['Gênero'].value_counts(normalize=True) * 100

# Calculando porcentagem de gêneros em df_enus3
percent_enus3 = df_enus3['Gênero'].value_counts(normalize=True) * 100

# Exibindo os resultados
# Exibir o resultado corrigido
print("Percentuais de gênero em df_enus4 após limpeza:")
print(percent_enus4)
print("\nPercentuais de gênero em df_ptbr:")
print(percent_ptbr)
print("\nPercentuais de gênero em df_enus3:")
print(percent_enus3)

# Calculando a média das porcentagens
average_percentages = (percent_enus4 + percent_ptbr + percent_enus3) / 3

# Exibindo o resultado
print("Média dos percentuais de gênero em todos os DataFrames:")
print(average_percentages)

In [ ]:
#1,031922% other

In [ ]:
# Filtrando para Valência == "Positiva"
positive_enus4 = df_enus4[df_enus4['valencia'] == "Positiva"]
positive_ptbr = df_ptbr[df_ptbr['Valência'] == "Positiva"]
positive_enus3 = df_enus3[df_enus3['Valência'] == "Positiva"]

# Filtrando para Valência == "Negativa"
negative_enus4 = df_enus4[df_enus4['Valência'] == "Negativa"]
negative_ptbr = df_ptbr[df_ptbr['Valência'] == "Negativa"]
negative_enus3 = df_enus3[df_enus3['Valência'] == "Negativa"]
# Calculando as porcentagens de gênero quando Valência é "Positiva"
percent_positive_enus4 = positive_enus4['Gênero'].value_counts(normalize=True) * 100
percent_positive_ptbr = positive_ptbr['Gênero'].value_counts(normalize=True) * 100
percent_positive_enus3 = positive_enus3['Gênero'].value_counts(normalize=True) * 100

percent_negative_enus4 = negative_enus4['Gênero'].value_counts(normalize=True) * 100
percent_negative_ptbr = negative_ptbr['Gênero'].value_counts(normalize=True) * 100
percent_negative_enus3 = negative_enus3['Gênero'].value_counts(normalize=True) * 100

# Exibindo os resultados individuais
print("Percentuais de gênero em df_enus4 quando Valência é 'Positiva':")
print(percent_positive_enus4)

print("\nPercentuais de gênero em df_ptbr quando Valência é 'Positiva':")
print(percent_positive_ptbr)

print("\nPercentuais de gênero em df_enus3 quando Valência é 'Positiva':")
print(percent_positive_enus3)


# Exibindo os resultados individuais
print("Percentuais de gênero em df_enus4 quando Valência é 'Negativa':")
print(percent_negative_enus4)

print("\nPercentuais de gênero em df_ptbr quando Valência é 'Negativa':")
print(percent_negative_ptbr)

print("\nPercentuais de gênero em df_enus3 quando Valência é 'Negativa':")
print(percent_negative_enus3)

# Calculando a média das porcentagens
average_positive = (percent_positive_enus4.add(percent_positive_ptbr, fill_value=0).add(percent_positive_enus3, fill_value=0)) / 3

# Exibindo o resultado
print("Média dos percentuais de gênero quando Valência é 'Positiva':")
print(average_positive)
# Calculando as porcentagens de gênero quando Valência é "Negativa"
percent_negative_enus4 = negative_enus4['Gênero'].value_counts(normalize=True) * 100
percent_negative_ptbr = negative_ptbr['Gênero'].value_counts(normalize=True) * 100
percent_negative_enus3 = negative_enus3['Gênero'].value_counts(normalize=True) * 100

# Calculando a média das porcentagens
average_negative = (percent_negative_enus4.add(percent_negative_ptbr, fill_value=0).add(percent_negative_enus3, fill_value=0)) / 3

# Exibindo o resultado
print("Média dos percentuais de gênero quando Valência é 'Negativa':")
print(average_negative)
